# DRMD trên LAMDA — V15 hợp nhất giao thức huấn luyện và kiểm toán

Notebook này là hiện vật huấn luyện duy nhất dùng để nộp kèm mã nguồn. Toàn
bộ sáu phương pháp được đặt trong cùng một giao thức dữ liệu, cùng mười hạt
giống và cùng cơ chế checkpoint; không cần ghép thủ công notebook V13/V14.

* `phase_1_baselines` chạy Static-MLP, DRMD-IRAL và DRMD-IRAAL (30 lượt).
* `phase_2_proposed` chạy DRMD-FN, DRMD-FN-BHR và mốc phản hồi đầy đủ
  DRMD-FFCR (30 lượt).
* `aggregate_only` không fit mô hình; pha này chỉ công nhận báo cáo khi đủ
  60/60 khóa `(phương pháp, seed)` hợp lệ.

Hai chiến lược đề xuất cùng tuân thủ quan hệ nhân quả: dự đoán chu kỳ t được
khóa trước; điểm trôi dạt chỉ đọc X_t; nhãn kiểm toán của t và hệ số/quota mới
chỉ tác động từ lần fit ở t+1. DRMD-FN dùng cổng trôi dạt; DRMD-FN-BHR dùng bộ
nhớ phát lại với quota thích nghi và không ghép cổng trôi dạt. Notebook không
dùng họ mã độc, mức độ nguy hại hoặc trường VirusTotal để điều khiển phần thưởng.


## Ô 1 — Giao thức V15 và ma trận ba pha


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

# ── Bootstrap Kaggle/Google Colab theo cấu hình tường minh ───────────────
RUNTIME_PLATFORM = "kaggle"
VALID_RUNTIME_PLATFORMS = {"kaggle", "colab", "auto"}
# Tạo secret KAGGLE_API_TOKEN trong Colab (biểu tượng chìa khóa), không dán
# token vào mã nguồn. Checkpoint là tùy chọn và phải cùng giao thức V15.
COLAB_REQUIRED_KAGGLE_DATASETS = [
    {
        "handle": "thanhngo1007/drmd-lamda-dataset",
        "target": "drmd-lamda-dataset",
        "required_directory": "DRMD_LAMDA_DATASET",
    },
    {
        "handle": "thanhngo1007/lamda-full-processed",
        "target": "lamda-full-processed",
        "required_directory": "Baseline",
    },
]
COLAB_CHECKPOINT_DATASET = ""  # Tùy chọn: "owner/checkpoint-dataset-slug"
COLAB_BACKUP_TO_MOUNTED_DRIVE = True
COLAB_DRIVE_BACKUP_DIR = "/content/drive/MyDrive/DRMD_checkpoints"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORK_ROOT = Path("/kaggle/working")
COLAB_INPUT_ROOT = Path("/content/drmd_colab/input")
COLAB_WORK_ROOT = Path("/content/drmd_colab/working")
COLAB_CHECKPOINT_INPUT_ROOTS = []


def _detect_google_colab():
    try:
        from google.colab import userdata as colab_userdata
    except (ImportError, ModuleNotFoundError):
        return False, None
    return True, colab_userdata


def resolve_runtime_platform(
    requested: str,
    kaggle_available: bool,
    colab_available: bool,
) -> str:
    """Phân giải nền tảng chạy theo cấu hình tường minh.

    ``auto`` ưu tiên Kaggle để việc import được ``google.colab`` trong ảnh
    runtime Kaggle không còn làm notebook ghi nhầm vào ``/content``.
    """

    normalized = str(requested).strip().lower()
    valid = {"kaggle", "colab", "auto"}
    if normalized not in valid:
        raise ValueError(
            "RUNTIME_PLATFORM phải là một trong: kaggle, colab, auto; "
            f"nhận {requested!r}."
        )
    if normalized == "auto":
        if bool(kaggle_available):
            return "kaggle"
        if bool(colab_available):
            return "colab"
        raise RuntimeError(
            "Không phát hiện môi trường Kaggle hoặc Google Colab. "
            "Hãy đặt RUNTIME_PLATFORM tường minh trên một nền tảng được hỗ trợ."
        )

    available = (
        bool(kaggle_available) if normalized == "kaggle"
        else bool(colab_available)
    )
    if not available:
        raise RuntimeError(
            f"RUNTIME_PLATFORM={normalized!r} nhưng nền tảng này không khả dụng."
        )
    return normalized


KAGGLE_ENVIRONMENT_SIGNAL = bool(
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    or os.environ.get("KAGGLE_URL_BASE"))
KAGGLE_RUNTIME_AVAILABLE = KAGGLE_WORK_ROOT.is_dir()
COLAB_RUNTIME_AVAILABLE, _COLAB_USERDATA = _detect_google_colab()
RESOLVED_RUNTIME_PLATFORM = resolve_runtime_platform(
    RUNTIME_PLATFORM,
    kaggle_available=KAGGLE_RUNTIME_AVAILABLE,
    colab_available=COLAB_RUNTIME_AVAILABLE,
)
IS_KAGGLE = RESOLVED_RUNTIME_PLATFORM == "kaggle"
IS_GOOGLE_COLAB = RESOLVED_RUNTIME_PLATFORM == "colab"
RUNTIME_INPUT_ROOT = (KAGGLE_INPUT_ROOT if IS_KAGGLE
                      else COLAB_INPUT_ROOT)
RUNTIME_WORK_ROOT = (KAGGLE_WORK_ROOT if IS_KAGGLE
                     else COLAB_WORK_ROOT)


def _read_colab_secret(name):
    if not IS_GOOGLE_COLAB or _COLAB_USERDATA is None:
        return None
    try:
        value = _COLAB_USERDATA.get(name)
    except Exception:
        return None
    value = str(value).strip() if value is not None else ""
    return value or None


def _import_kagglehub_for_colab():
    try:
        import kagglehub
    except ModuleNotFoundError:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--quiet", "kagglehub",
        ])
        import kagglehub
    return kagglehub


def _find_required_directory(root, directory_name):
    if not root.is_dir():
        return None
    if root.name == directory_name:
        return root
    return next((path for path in root.rglob(directory_name) if path.is_dir()), None)


def _download_colab_dataset(kagglehub, handle, target_name,
                            required_directory=None):
    target = COLAB_INPUT_ROOT / target_name
    existing = (_find_required_directory(target, required_directory)
                if required_directory else None)
    if existing is not None:
        print({"colab_dataset": handle, "status": "already_present",
               "path": str(existing)})
        return target

    try:
        target.mkdir(parents=True, exist_ok=True)
        write_probe = target / ".colab_write_probe"
        write_probe.write_text("ok", encoding="utf-8")
        write_probe.unlink()
    except OSError as exc:
        raise RuntimeError(
            f"Không thể ghi Kaggle Dataset {handle!r} vào {target}: {exc}. "
            "Colab phải dùng thư mục dưới /content; /kaggle/input là input "
            "chỉ đọc dành cho Kaggle."
        ) from exc

    try:
        downloaded = kagglehub.dataset_download(handle, output_dir=str(target))
    except OSError as exc:
        raise RuntimeError(
            f"Lỗi hệ thống tệp khi tải Kaggle Dataset {handle!r} vào "
            f"{target}: {exc}."
        ) from exc
    except Exception as exc:
        raise RuntimeError(
            f"Không tải được Kaggle Dataset {handle!r}. Hãy tạo Colab secret "
            "KAGGLE_API_TOKEN, bật quyền truy cập notebook và kiểm tra quyền "
            "truy cập Dataset."
        ) from exc

    downloaded_path = Path(downloaded).expanduser()
    if required_directory:
        # KaggleHub trên Colab có thể bỏ qua output_dir và trả input cache
        # chỉ đọc dưới /kaggle/input. Tìm ở cả target lẫn đường dẫn trả về;
        # nếu dùng cache thì tạo liên kết tên chuẩn dưới /content để các ô sau
        # giữ nguyên giao thức định vị mà không sao chép hoặc sửa dữ liệu.
        marker = _find_required_directory(target, required_directory)
        if marker is None:
            marker = _find_required_directory(downloaded_path, required_directory)
        if marker is None:
            searched = [str(target), str(downloaded_path)]
            raise RuntimeError(
                f"Dataset {handle!r} đã tải nhưng thiếu thư mục "
                f"{required_directory!r}. Đã kiểm tra: {searched}."
            )
        try:
            marker.resolve().relative_to(target.resolve())
            marker_is_under_target = True
        except ValueError:
            marker_is_under_target = False
        if not marker_is_under_target:
            alias = target / required_directory
            if alias.is_symlink():
                if alias.resolve() != marker.resolve():
                    raise RuntimeError(
                        f"Liên kết cache hiện có {alias} không trỏ tới {marker}.")
            elif alias.exists():
                raise RuntimeError(
                    f"Không thể liên kết cache vì {alias} đã tồn tại.")
            else:
                alias.symlink_to(marker.resolve(), target_is_directory=True)
            marker = alias
    else:
        marker = downloaded_path
    print({"colab_dataset": handle, "status": "ready",
           "path": str(marker), "kagglehub_returned": str(downloaded_path)})
    return marker


def _prepare_colab_kaggle_inputs():
    if not IS_GOOGLE_COLAB:
        return
    COLAB_INPUT_ROOT.mkdir(parents=True, exist_ok=True)
    COLAB_WORK_ROOT.mkdir(parents=True, exist_ok=True)

    token = os.environ.get("KAGGLE_API_TOKEN") or _read_colab_secret(
        "KAGGLE_API_TOKEN")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token

    kagglehub = _import_kagglehub_for_colab()
    for dataset in COLAB_REQUIRED_KAGGLE_DATASETS:
        _download_colab_dataset(
            kagglehub,
            dataset["handle"],
            dataset["target"],
            dataset["required_directory"],
        )

    checkpoint_handle = (
        str(COLAB_CHECKPOINT_DATASET).strip()
        or (_read_colab_secret("KAGGLE_CHECKPOINT_DATASET") or "")
    )
    if checkpoint_handle:
        checkpoint_root = _download_colab_dataset(
            kagglehub,
            checkpoint_handle,
            "v15-checkpoint",
            required_directory=None,
        )
        COLAB_CHECKPOINT_INPUT_ROOTS.append(Path(checkpoint_root))
    print({"runtime": "google_colab", "kaggle_input": str(COLAB_INPUT_ROOT),
           "optional_checkpoint_dataset": bool(checkpoint_handle)})


_prepare_colab_kaggle_inputs()
print({
    "runtime_platform_requested": RUNTIME_PLATFORM,
    "runtime_platform_resolved": RESOLVED_RUNTIME_PLATFORM,
    "runtime_input_root": str(RUNTIME_INPUT_ROOT),
    "runtime_work_root": str(RUNTIME_WORK_ROOT),
    "kaggle_environment_signal": KAGGLE_ENVIRONMENT_SIGNAL,
})

# ── Nhận dạng giao thức V15 ──────────────────────────────────────────────────
BASE_EXPERIMENT_NAME = "lamda_2013_2025_drmd_fn_bhr_unified_v15"
PROTOCOL_VERSION = "drmd_lamda_fn_bhr_unified_v15_2026-08-15"
STRATEGY_RUN_TAG = "aligned_drmd_strategies_v15"
DATASET_YEARS = [2013, 2014, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
LAMDA_HF_REPO_ID = "IQSeC-Lab/LAMDA"
LAMDA_HF_REVISION = "ad9614bdd5556767f97ced2fce797c2f06408ebf"
EXPECTED_LAMDA_FILE_SHA256 = {
    "2013/2013_test.parquet": "208528e70c2c71c3765c4749d0293cd2816ca316dce60ef6eef42899d5aca323",
    "2013/2013_train.parquet": "0b83f9a317fa1388406f21837de422cb587cd5ab0ec401dec63f78888b847989",
    "2014/2014_test.parquet": "b5967aa77a1d25e9a3671fa38e266cdddd34e5436755cbaf48463ede5b4bb5bc",
    "2014/2014_train.parquet": "1f65b6053a174eb34a050c26162b294822d3b30e1cf83deae594f26d0b9d9a6f",
    "2016/2016_test.parquet": "daca37c4c706d60497b2cb25bf4cec0d35a5c73dabdfbe747d6f2bf36aa3fc3f",
    "2016/2016_train.parquet": "8f0871a41515e1cbb45cc7cc10d6d489629efe80fc6499c8187c0c148a6155cd",
    "2017/2017_test.parquet": "56a35dccadc8b51eacba89b279ada9be3e7620477d159da4068c0db3efeafa38",
    "2017/2017_train.parquet": "435e5bd1be3be3f36b332d3eb89ecfc5250c6c618f1fb1085f2f0842242b3c4c",
    "2018/2018_test.parquet": "70618444fa642388cdbb7a0747e5b871100c73ec3289a9e78329aa2041a9bcb2",
    "2018/2018_train.parquet": "ad01e1224ef3a459a29309c8ad5eceaabd6a7af8b262df0a438f9aeeae1671dd",
    "2019/2019_test.parquet": "400a46fe12a3d8110ab4a1073545f023ba148495bfe9c64220fe3dceee71653c",
    "2019/2019_train.parquet": "d7ceb62a6730f02652982549a2ed4eac8160351a19b628fba1302fe97ed7d5eb",
    "2020/2020_test.parquet": "6d9fe6fe5232c281b08aa34d25d23eb44fbf2a14597ba6eb7d28376db8aaae7b",
    "2020/2020_train.parquet": "19d5a25c2304d8af929167724f8785632fe8fe7249266a099603146ca11a7b80",
    "2021/2021_test.parquet": "10b2bf2a96be746a80a22d59a7a95c9e4112079d29c92c67ce3c62f556458a7a",
    "2021/2021_train.parquet": "b1642ef38f7669951403367bf56c745c535a9250aba4bb25a2502e123c1c30c1",
    "2022/2022_test.parquet": "4f74131f710c0dff2daf622183df0ef72b5be12ee01418897b3e4a4729313691",
    "2022/2022_train.parquet": "f41b5c4f051dd5dced9adc322b221ac4fab5fa16202b07c168a80ca3d3a1662b",
    "2023/2023_test.parquet": "58f4f6356d1f88fcaeb5fa04c67a1260b4cbe736c18961dc3e51b6cd121598c2",
    "2023/2023_train.parquet": "b1dfb0962484943537d1a02445f69454408c6e8a82910e9ef17f418f6dfc6af0",
    "2024/2024_test.parquet": "e360ff7a9a0b3f1f8c61b17c0aa5de25ce7cd3411e5bb3e2016fdf724c18f2b9",
    "2024/2024_train.parquet": "8cdae48a4765c9af4aae760e92f0d14dc4c2fa89a4164297a110ad0f450f78ff",
    "2025/2025_test.parquet": "702e8131a328dee30c7c1ef4373cefafc39ccc0ba8c5282257836876c783a3c6",
    "2025/2025_train.parquet": "cf65587318bd2820081fba89f4d1f648a33b44519dbbb88b392ba448ff498846",
}
EXPECTED_LAMDA_SAMPLES = 1_008_381
EXPECTED_LAMDA_FEATURES = 4_561
EXPECTED_INITIAL_TRAIN_SAMPLES = 122_434
EXPECTED_TEST_PERIODS = 110
EXPECTED_TEST_SAMPLES = 885_947
TRAINING_WINDOW_MODE = "calendar_12"
TRAINING_WINDOW = 12 if TRAINING_WINDOW_MODE == "calendar_12" else 14
TESTING_WINDOW, GRANULARITY = 1, "month"
EXPERIMENT_NAME = (BASE_EXPERIMENT_NAME if TRAINING_WINDOW_MODE == "calendar_12"
                   else BASE_EXPERIMENT_NAME + "_datawin12")

WORK = RUNTIME_WORK_ROOT
OUT = WORK / "results" / EXPERIMENT_NAME
DATA_OUT = WORK / "DRMD_datasets" / EXPERIMENT_NAME
RAW_OUT, TABLE_OUT, FIG_OUT = OUT / "raw", OUT / "tables", OUT / "figures"
STRATEGY_RAW_OUT = RAW_OUT / STRATEGY_RUN_TAG
STAGING_OUT, QUARANTINE_OUT = OUT / ".staging", OUT / "quarantine"
for path in (OUT, DATA_OUT, RAW_OUT, TABLE_OUT, FIG_OUT, STRATEGY_RAW_OUT,
             STAGING_OUT, QUARANTINE_OUT):
    path.mkdir(parents=True, exist_ok=True)

# ── Điểm vận hành và BHR khóa trước khi chạy ─────────────────────────────────
SEEDS = [0, 1, 7, 13, 26, 42, 73, 2026, 314159, 281083886]
MP_FIXED = 1.0
DRMD_TRAINING_EPOCHS = 5
AL_SELECTION_MODE = "Probs"
STRATEGY_FINETUNING_SIZE = 5000
STRATEGY_REJECT_COST = -0.1
RECENCY_POLICY = "recent_calendar_months_seeded_boundary_v2"
FEEDBACK_UPDATE_BUDGET = 15
BHR_FN_FRACTION = 0.35
BHR_FP_FRACTION = 0.25
BHR_BACKGROUND_FRACTION = 0.40
BHR_FN_MAX_REPEAT = 4
BHR_FP_MAX_REPEAT = 2
FN_CONTROLLER_CONFIDENCE_LEVEL = 0.90
FN_CONTROLLER_WINDOW_PERIODS = TRAINING_WINDOW
DRIFT_PROJECTION_DIM = 32
DRIFT_REFERENCE_WINDOW = TRAINING_WINDOW
DRIFT_THRESHOLD_QUANTILE = 0.95
DRIFT_MINIMUM_SCORE_HISTORY = 4
DRIFT_MAX_ROWS_PER_PERIOD = 2048
DRIFT_DECAY_PERIODS = TRAINING_WINDOW
ADAPTIVE_BHR_WINDOW_PERIODS = TRAINING_WINDOW
ADAPTIVE_BHR_MAX_HARD_FRACTION = 0.60

IRAL_POLICY = {"name": "IRAL", "scope": "rejected_feedback_endogenous"}
IRAAL_POLICY = {"name": "IRAAL", "scope": "budgeted_update",
                "budget": FEEDBACK_UPDATE_BUDGET}
FFCR_POLICY = {"name": "FFCR", "scope": "full_feedback_reference"}

def strategy_variant(name, audit_sampling=False, adaptive_fn_penalty=False,
                     balanced_hard_replay=False, full_feedback=False,
                     feedback_budget=FEEDBACK_UPDATE_BUDGET,
                     selector_mode="uncertainty", policy=None,
                     reject_cost=None, drift_aware_fn_penalty=False,
                     adaptive_bhr_quota=False, implementation="existing"):
    if policy is None:
        policy = {"name": "IRAAL", "scope": "budgeted_update",
                  "budget": int(feedback_budget)}
    return {
        "name": name, "model": "PPO_Bandit",
        "minority_priority": MP_FIXED, "majority_priority": 1.0,
        "temporal_rewards": True, "temporal_scaling": 6.0,
        "training_epochs": DRMD_TRAINING_EPOCHS, "fn_penalty": 1.0,
        "selector_mode": str(selector_mode),
        "feedback_budget": int(feedback_budget),
        "candidate_multiplier": 1,
        "full_feedback": bool(full_feedback),
        "audit_sampling": bool(audit_sampling),
        "adaptive_fn_penalty": bool(adaptive_fn_penalty),
        "balanced_hard_replay": bool(balanced_hard_replay),
        "drift_aware_fn_penalty": bool(drift_aware_fn_penalty),
        "adaptive_bhr_quota": bool(adaptive_bhr_quota),
        "implementation": str(implementation),
        "reject_cost": float(
            STRATEGY_REJECT_COST if reject_cost is None else reject_cost),
        "policy": dict(policy),
    }

PHASE_1_STRATEGY_VARIANTS = [
    strategy_variant("DRMD-IRAL", feedback_budget=0,
                     selector_mode="iral", policy=IRAL_POLICY),
    strategy_variant("DRMD-IRAAL"),
]
PHASE_2_STRATEGY_VARIANTS = [
    strategy_variant(
        "DRMD-FN", audit_sampling=True, adaptive_fn_penalty=True,
        drift_aware_fn_penalty=True, implementation="v15_drift_gate"),
    strategy_variant(
        "DRMD-FN-BHR", audit_sampling=True, adaptive_fn_penalty=True,
        balanced_hard_replay=True, adaptive_bhr_quota=True,
        implementation="v15_adaptive_bhr_quota"),
    strategy_variant("DRMD-FFCR", full_feedback=True, feedback_budget=0,
                     selector_mode="full_feedback", policy=FFCR_POLICY),
]
ALL_STRATEGY_VARIANTS = (
    list(PHASE_1_STRATEGY_VARIANTS) + list(PHASE_2_STRATEGY_VARIANTS))
PHASE_1_METHODS = ["Static-MLP"] + [
    variant["name"] for variant in PHASE_1_STRATEGY_VARIANTS]
PHASE_2_METHODS = [variant["name"] for variant in PHASE_2_STRATEGY_VARIANTS]
EXPECTED_METHOD_NAMES = PHASE_1_METHODS + PHASE_2_METHODS
ALL_STRATEGY_NAMES = [variant["name"] for variant in ALL_STRATEGY_VARIANTS]
assert len(PHASE_1_METHODS) == 3
assert len(PHASE_2_METHODS) == 3
assert len(set(EXPECTED_METHOD_NAMES)) == 6

VALID_RUN_PHASES = {"phase_1_baselines", "phase_2_proposed", "aggregate_only"}
KAGGLE_RUN_PHASE = "phase_1_baselines"
assert KAGGLE_RUN_PHASE in VALID_RUN_PHASES
if KAGGLE_RUN_PHASE == "phase_1_baselines":
    STRATEGY_VARIANTS = list(PHASE_1_STRATEGY_VARIANTS)
elif KAGGLE_RUN_PHASE == "phase_2_proposed":
    STRATEGY_VARIANTS = list(PHASE_2_STRATEGY_VARIANTS)
else:
    STRATEGY_VARIANTS = list(ALL_STRATEGY_VARIANTS)
RUN_STRATEGY_EXPERIMENT = KAGGLE_RUN_PHASE != "aggregate_only"

# ── Điều khiển Kaggle ────────────────────────────────────────────────────────
SESSION_LAUNCH_CUTOFF_HOURS = 10.5
RUNTIME_PRIOR = {"source": "v12_observed_runtime",
                 "median_minutes_per_run": 4.788,
                 "note": "chỉ dùng lập kế hoạch; V15 có checkpoint hợp nhất"}
STOP_AFTER_DRMD_RUNS = None
STOP_AFTER_STATIC_RUNS = None
MAX_CONSECUTIVE_FAILURES = 3
SESSION_STARTED_AT = time.monotonic()
FINALIZE_REPORT = KAGGLE_RUN_PHASE == "aggregate_only"
STRICT_INVARIANTS = FINALIZE_REPORT
REQUIRE_CUDA = True
BATCH_PREDICT_ROWS = 4096
LAMDA_ROOT_OVERRIDE = None

STATIC_RUN_COUNT = (len(SEEDS)
                    if KAGGLE_RUN_PHASE == "phase_1_baselines" else 0)
STRATEGY_RUN_COUNT = (len(STRATEGY_VARIANTS) * len(SEEDS)
                      if RUN_STRATEGY_EXPERIMENT else 0)
TOTAL_PLANNED_RUN_COUNT = STATIC_RUN_COUNT + STRATEGY_RUN_COUNT

PARAMETER_PROVENANCE = {
    "training_window_months": {
        "value": TRAINING_WINDOW, "category": "temporal_protocol",
        "basis": "12 calendar months; missing months are not replaced"},
    "dataset_revision": {
        "value": LAMDA_HF_REVISION, "category": "dataset_protocol",
        "validation": "all 24 Baseline parquet files match locked SHA-256"},
    "testing_window_months": {
        "value": TESTING_WINDOW, "category": "dataset_protocol"},
    "minority_priority": {
        "value": MP_FIXED, "category": "controlled_factor",
        "basis": "neutral malware reward multiplier shared by all DRMD arms"},
    "reject_cost": {
        "value": STRATEGY_REJECT_COST, "category": "locked_operating_point",
        "basis": "held fixed across the five DRMD arms"},
    "feedback_update_budget": {
        "value": FEEDBACK_UPDATE_BUDGET,
        "category": "locked_operating_point",
        "validation": "all IRAAL/FN arms use the same B=15"},
    "finetuning_size": {
        "value": STRATEGY_FINETUNING_SIZE,
        "category": "controlled_capacity"},
    "fn_controller_existing": {
        "value": {"confidence_level": FN_CONTROLLER_CONFIDENCE_LEVEL,
                  "window_periods": FN_CONTROLLER_WINDOW_PERIODS},
        "category": "existing_method"},
    "drift_gate": {
        "value": {"projection_dim": DRIFT_PROJECTION_DIM,
                  "reference_window": DRIFT_REFERENCE_WINDOW,
                  "threshold_quantile": DRIFT_THRESHOLD_QUANTILE,
                  "minimum_score_history": DRIFT_MINIMUM_SCORE_HISTORY,
                  "max_rows_per_period": DRIFT_MAX_ROWS_PER_PERIOD,
                  "decay_periods": DRIFT_DECAY_PERIODS},
        "category": "proposed_method",
        "basis": "bounded early warning from unlabeled P(X)",
        "validation": "threshold uses past scores; effect begins at t+1"},
    "bhr_existing": {
        "value": {"fn_fraction": BHR_FN_FRACTION,
                  "fp_fraction": BHR_FP_FRACTION,
                  "background_fraction": BHR_BACKGROUND_FRACTION,
                  "fn_max_repeat": BHR_FN_MAX_REPEAT,
                  "fp_max_repeat": BHR_FP_MAX_REPEAT},
        "category": "existing_method"},
    "adaptive_bhr": {
        "value": {"window_periods": ADAPTIVE_BHR_WINDOW_PERIODS,
                  "maximum_hard_fraction": ADAPTIVE_BHR_MAX_HARD_FRACTION,
                  "evidence_source": "random_audit_only"},
        "category": "proposed_method",
        "basis": "Jeffreys-smoothed FN/FP risk with availability limits",
        "validation": "unfillable hard quota transfers to recent background"},
    "experiment_matrix": {
        "value": {"phase_1_runs": 30, "phase_2_runs": 30,
                  "aggregate_expected_runs": 60},
        "category": "predeclared_analysis_plan"},
    "seeds": {
        "value": SEEDS, "category": "locked_replication",
        "validation": "all comparisons are paired over ten seeds"},
}

PROTOCOL_DEVIATIONS = {
    "drift_gate": {
        "scope": "unlabeled drift warning for next-period FN penalty",
        "label_budget_change": 0},
    "adaptive_bhr": {
        "scope": "training-memory composition from random-audit evidence",
        "label_budget_change": 0},
    "checkpoint_compatibility": "same_v15_protocol_and_source_hashes_only",
}

assert abs(BHR_FN_FRACTION + BHR_FP_FRACTION + BHR_BACKGROUND_FRACTION - 1.0) < 1e-12
print({"experiment": EXPERIMENT_NAME, "protocol_version": PROTOCOL_VERSION,
       "phase": KAGGLE_RUN_PHASE, "seeds": SEEDS,
       "static_runs": STATIC_RUN_COUNT,
       "strategy_runs": STRATEGY_RUN_COUNT,
       "phase_1_methods": PHASE_1_METHODS,
       "phase_2_methods": PHASE_2_METHODS,
       "phase_strategy_variants": [item["name"] for item in STRATEGY_VARIANTS],
       "all_strategy_variants": ALL_STRATEGY_NAMES,
       "total_planned_runs": TOTAL_PLANNED_RUN_COUNT,
       "expected_methods": EXPECTED_METHOD_NAMES,
       "aggregate_expected_runs": 60})


## Ô 2 — Khôi phục checkpoint hợp nhất V15


In [ ]:
# Chỉ khôi phục checkpoint mang đúng tên thí nghiệm V15.
RESTORE_EXISTING_RESULTS = True
if RESTORE_EXISTING_RESULTS:
    import shutil
    import zipfile

    input_root = RUNTIME_INPUT_ROOT
    checkpoint_search_roots = [input_root, RUNTIME_WORK_ROOT] + [
        Path(root) for root in COLAB_CHECKPOINT_INPUT_ROOTS
        if Path(root).exists() and Path(root) != input_root
    ]
    allowed_roots = {STRATEGY_RUN_TAG, "static_baselines"}
    checkpoint_glob = f"{EXPERIMENT_NAME}_checkpoint_raw*.zip"
    checkpoint_zips = sorted({
        path for root in checkpoint_search_roots
        for path in root.rglob(checkpoint_glob)
    })
    checkpoint_inputs = sorted({
        path for root in checkpoint_search_roots for path in root.rglob("raw")
        if path.is_dir() and (path / STRATEGY_RUN_TAG).exists()
    })
    restored = 0
    already_present = 0

    for checkpoint_root in checkpoint_inputs:
        for src in list(checkpoint_root.rglob("*.p")) + list(checkpoint_root.rglob("*.p.meta.json")):
            relative = src.relative_to(checkpoint_root)
            if not relative.parts or relative.parts[0] not in allowed_roots:
                continue
            dst = RAW_OUT / relative
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                already_present += 1
            else:
                shutil.copy2(src, dst); restored += 1

    for zip_path in checkpoint_zips:
        if not zipfile.is_zipfile(zip_path):
            continue
        with zipfile.ZipFile(zip_path) as archive:
            for member in archive.namelist():
                parts = Path(member).parts
                if (len(parts) < 2 or parts[0] != "raw"
                        or parts[1] not in allowed_roots
                        or Path(member).is_absolute()
                        or ".." in parts
                        or not (member.endswith(".p") or member.endswith(".p.meta.json"))):
                    continue
                dst = OUT / member
                dst.parent.mkdir(parents=True, exist_ok=True)
                if dst.exists():
                    already_present += 1
                else:
                    with archive.open(member) as src, open(dst, "wb") as out:
                        shutil.copyfileobj(src, out)
                    restored += 1

    print({"checkpoint_protocol": PROTOCOL_VERSION,
           "checkpoint_archives_found": len(checkpoint_zips),
           "checkpoint_raw_roots_found": len(checkpoint_inputs),
           "restored_artifact_files": restored,
           "already_present_artifact_files": already_present})
    if not checkpoint_zips and not checkpoint_inputs:
        print("ℹ️ Không có checkpoint V15; bắt đầu ma trận mới.")


## Ô 3 — Kiểm kê artifact V15 đã khôi phục


In [ ]:
# Kiểm kê sơ bộ sau khôi phục; chưa dùng để công nhận kết quả.
import collections
import json
import pickle
from pathlib import Path

def kiem_ke_artifact_da_khoi_phuc(raw_out: Path) -> dict:
    period_distribution = collections.Counter()
    files_by_variant = collections.Counter()
    unreadable = []
    missing_sidecar = []
    for artifact_path in sorted(Path(raw_out).rglob("*.p")):
        if "static_baselines" in artifact_path.parts:
            continue
        try:
            with open(artifact_path, "rb") as handle:
                result = pickle.load(handle)
            period_distribution[len(result.get("f1", []))] += 1
            files_by_variant[artifact_path.parent.name] += 1
            if not Path(str(artifact_path) + ".meta.json").exists():
                missing_sidecar.append(str(artifact_path))
        except Exception as exc:
            unreadable.append({
                "path": str(artifact_path),
                "error": f"{type(exc).__name__}: {exc}",
            })
    return {
        "period_distribution": dict(period_distribution),
        "files_by_variant": dict(files_by_variant),
        "unreadable": unreadable,
        "missing_sidecar": missing_sidecar,
        "status": "inventory_only_not_scientific_validation",
    }

RESTORED_ARTIFACT_INVENTORY = kiem_ke_artifact_da_khoi_phuc(RAW_OUT)
print(json.dumps(RESTORED_ARTIFACT_INVENTORY, indent=2, ensure_ascii=False))
(TABLE_OUT / "restored_artifact_inventory.json").write_text(
    json.dumps(RESTORED_ARTIFACT_INVENTORY, indent=2, ensure_ascii=False),
    encoding="utf-8")


## Ô 4 — Bắt `UndefinedMetricWarning` thay vì để nó ngập log

Các chu kỳ không có đủ hỗ trợ lớp được ghi thành artifact kiểm toán thay vì làm ngập
nhật ký. Thanh tiến trình dùng trực tiếp `len(X_tests)`, không ghi cứng số chu kỳ.

In [ ]:
import warnings, csv as _csv
from sklearn.exceptions import UndefinedMetricWarning

warnings.simplefilter("always", UndefinedMetricWarning)
UNDEFINED_LOG: list[dict] = []
CURRENT_RUN_TAG, CURRENT_PERIOD_IDX = "", -1

_orig_showwarning = warnings.showwarning
def _ghi_nhat_ky(message, category, filename, lineno, file=None, line=None):
    if category is UndefinedMetricWarning:
        UNDEFINED_LOG.append({"run_tag": CURRENT_RUN_TAG, "period": CURRENT_PERIOD_IDX,
                              "message": str(message)})
        return                                   # không in, tránh ngập log Kaggle
    _orig_showwarning(message, category, filename, lineno, file, line)
warnings.showwarning = _ghi_nhat_ky

def xuat_nhat_ky_suy_bien():
    if not UNDEFINED_LOG:
        print("✅ Không có độ đo suy biến nào bị ghi nhận."); return
    with open(TABLE_OUT / "undefined_metric_periods.csv", "w", newline="", encoding="utf-8") as fh:
        w = _csv.DictWriter(fh, fieldnames=["run_tag", "period", "message"])
        w.writeheader(); w.writerows(UNDEFINED_LOG)
    print(f"⚠️  {len(UNDEFINED_LOG)} lần độ đo suy biến — xem undefined_metric_periods.csv")

## Ô 5 — Cố định các nguồn ngẫu nhiên chính và ghi dấu môi trường

Python `random`, NumPy, PyTorch, CUDA và thuật toán cuDNN được khóa ngay trước mỗi lượt.
`PYTHONHASHSEED` chỉ có hiệu lực đầy đủ khi được đặt trước lúc kernel khởi động, nên
notebook không dựa vào thứ tự hash cho bất kỳ phép chia hay chọn mẫu nào.

In [ ]:
import os, random, platform, subprocess
# Phải đặt trước lần khởi tạo ngữ cảnh CUDA đầu tiên.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np, torch

def co_dinh_hat_giong(seed: int) -> None:
    """Gọi NGAY TRƯỚC mỗi lần khởi tạo mô hình."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    os.environ["PYTHONHASHSEED"] = str(seed)  # áp dụng cho tiến trình con tạo sau đây

def git_commit_for(path):
    try:
        return subprocess.check_output(
            ["git", "-C", str(path), "rev-parse", "HEAD"],
            stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "không có"

def dau_moi_truong() -> dict:
    import sklearn, scipy
    return {"python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "scipy": scipy.__version__,
            "scikit_learn": sklearn.__version__,
            "cuda": torch.version.cuda if torch.cuda.is_available() else None,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "cudnn_deterministic_at_stamp": bool(
                torch.backends.cudnn.deterministic),
            "deterministic_algorithms_at_stamp": bool(
                torch.are_deterministic_algorithms_enabled()),
            "determinism_policy": (
                "co_dinh_hat_giong(seed) được gọi ngay trước mỗi lần khởi tạo mô hình"),
            "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
            "python_hash_seed_note": "kernel hiện tại phải được khởi động với biến này; không dùng hash-order trong giao thức",
            "working_directory_git_commit": git_commit_for(Path.cwd()),
            "protocol_version": PROTOCOL_VERSION,
            "protocol_deviations": PROTOCOL_DEVIATIONS,
            "training_window_mode": TRAINING_WINDOW_MODE}

ENVIRONMENT_STAMP = dau_moi_truong()
print(json.dumps(ENVIRONMENT_STAMP, indent=2, ensure_ascii=False))

## Ô 6 — Nạp thư viện và vá DRMD

Hàm phần thưởng được đối chiếu với `environment.py` của mã gốc: trước khi nhân trọng
số thời gian, `TN = +1`, `FP = −1`, `TP = +m_p`, `FN = −m_p · λ_FN`.

In [ ]:
# Kaggle already provides NumPy/SciPy/scikit-learn/PyArrow.  Install only the
# DRMD dependencies missing from the base image; avoiding broad upgrades keeps
# CUDA packages consistent with the Kaggle runtime.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "StrEnum==0.4.15", "hydra-core==1.3.2", "backpack-for-pytorch==1.7.1"])
from importlib import metadata as _package_metadata

_required_runtime_distributions = {
    "StrEnum": "0.4.15",
    "hydra-core": "1.3.2",
    "backpack-for-pytorch": "1.7.1",
}
_installed_runtime_distributions = {
    name: _package_metadata.version(name)
    for name in _required_runtime_distributions
}
if _installed_runtime_distributions != _required_runtime_distributions:
    raise RuntimeError(
        "Phiên bản dependency DRMD lệch cấu hình khóa: "
        f"{_installed_runtime_distributions}")
_torch_distribution_version = _package_metadata.version("torch")
if not str(torch.__version__).startswith(_torch_distribution_version):
    raise RuntimeError(
        "pip đã thay torch trên đĩa nhưng kernel còn giữ module cũ; "
        "hãy khởi động lại phiên Kaggle trước khi chạy.")
ENVIRONMENT_STAMP["installed_runtime_distributions"] = (
    _installed_runtime_distributions)


In [ ]:
import os, sys, re, csv, json, pickle, zipfile
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import scipy.sparse as sp

if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

INPUT_ROOT = RUNTIME_INPUT_ROOT
INPUT_PROJECT_CANDIDATES = [
    INPUT_ROOT / "datasets/thanhngo1007/drmd-lamda-dataset/DRMD_LAMDA_DATASET",
    INPUT_ROOT / "drmd-lamda-dataset/DRMD_LAMDA_DATASET",
    INPUT_ROOT / "drmd-lamda-dataset",
]

def is_project(path):
    return path.is_dir() and (path / "references/DRMD").is_dir()

PROJECT = next((p for p in INPUT_PROJECT_CANDIDATES if is_project(p)), None)
if PROJECT is None and INPUT_ROOT.exists():
    # Kaggle mounts datasets under a user-controlled slug.  Search only for
    # directories that contain the DRMD source to avoid selecting a wrong input.
    PROJECT = next((p for p in INPUT_ROOT.rglob("DRMD_LAMDA_DATASET") if is_project(p)), None)
if PROJECT is None and INPUT_ROOT.exists():
    PROJECT = next((p for p in INPUT_ROOT.iterdir() if is_project(p)), None)
if PROJECT is None:
    mounted = [p.name for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        "Khong tim thay DRMD_LAMDA_DATASET. Trong Kaggle, chon Add Input va gan dataset "
        "chua thu muc references/DRMD va du lieu LAMDA. Mounted inputs hien co: " + repr(mounted))

DRMD_ROOT = PROJECT / "references/DRMD"
EXP_DIR = DRMD_ROOT / "Experiments"
TESSERACT_ROOT = PROJECT / "references/tesseract-ml-release"
RPAL_ROOT = PROJECT / "references/RPAL"

for p in [PROJECT, DRMD_ROOT, EXP_DIR, TESSERACT_ROOT, RPAL_ROOT]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("PROJECT:", PROJECT)
print("DRMD source:", DRMD_ROOT)


ENVIRONMENT_STAMP.update({
    "project_git_commit": git_commit_for(PROJECT),
    "drmd_git_commit": git_commit_for(DRMD_ROOT),
})
print("Dấu vết mã nguồn:", {
    "project_git_commit": ENVIRONMENT_STAMP["project_git_commit"],
    "drmd_git_commit": ENVIRONMENT_STAMP["drmd_git_commit"],
})

# V15 tự mang theo năm mô-đun chiến lược và kiểm SHA-256 trước khi import.
import hashlib as _strategy_hashlib
import importlib as _strategy_importlib

BALANCED_HARD_REPLAY_EMBEDDED_SOURCE = r'''"""Balanced Hard Replay cho thí nghiệm DRMD-FN trên dòng dữ liệu LAMDA.

Mô-đun này chỉ thay cách cấu tạo bộ nhớ huấn luyện từ các nhãn đã được mở.
Nhãn của chu kỳ ``t`` phải được đưa vào bằng :meth:`observe` sau dự đoán và
chỉ được dùng khi gọi :meth:`build` cho chu kỳ ``t + 1`` hoặc muộn hơn.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Any

import numpy as np

try:  # Kaggle có SciPy; nhánh NumPy giữ cho kiểm thử logic chạy tối giản.
    import scipy.sparse as sp
except ModuleNotFoundError:  # pragma: no cover - chỉ dùng ở môi trường kiểm thử nhẹ
    sp = None


@dataclass(frozen=True)
class ReplayTelemetry:
    """Số liệu kiểm toán của một bộ nhớ được dựng cho lần fit kế tiếp."""

    fit_period: int
    memory_cap: int
    memory_rows: int
    unique_rows_used: int
    fn_unique_buffer: int
    fp_unique_buffer: int
    background_unique_buffer: int
    fn_rows: int
    fp_rows: int
    background_rows: int
    fn_replay_rows: int
    fp_replay_rows: int
    replay_fraction: float
    mean_labeled_age_periods: float
    max_labeled_age_periods: float
    newest_source_period: int
    causal_lag_ok: bool

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class ReplayBatch:
    """Ma trận huấn luyện và dấu vết nguồn của BHR."""

    X: Any
    y: np.ndarray
    t: np.ndarray
    source_period: np.ndarray
    source_bucket: np.ndarray
    telemetry: ReplayTelemetry


class _Bucket:
    """Bộ đệm thưa, độc quyền theo loại lỗi tại thời điểm nhãn được mở."""

    def __init__(self, name: str, unique_cap: int, seed: int) -> None:
        self.name = str(name)
        self.unique_cap = int(unique_cap)
        self.seed = int(seed)
        self.X: Any | None = None
        self.y = np.asarray([], dtype=np.int8)
        self.t = np.asarray([], dtype="datetime64[ns]")
        self.source_period = np.asarray([], dtype=np.int64)

    def __len__(self) -> int:
        return int(self.y.size)

    @staticmethod
    def _as_matrix(X: Any) -> Any:
        if sp is None:
            matrix = np.asarray(X)
            if matrix.ndim != 2:
                raise ValueError("X phải là ma trận hai chiều.")
            return matrix
        matrix = X if sp.issparse(X) else sp.csr_matrix(np.asarray(X))
        return matrix.tocsr()

    @staticmethod
    def _vstack(matrices: list[Any]) -> Any:
        if sp is None:
            return np.vstack(matrices)
        return sp.vstack(matrices, format="csr")

    def append(
        self,
        X: Any,
        y: np.ndarray,
        t: np.ndarray,
        source_period: int,
    ) -> None:
        y_array = np.asarray(y, dtype=np.int8).reshape(-1)
        if y_array.size == 0:
            return
        t_array = np.asarray(t).reshape(-1).astype("datetime64[ns]")
        X_matrix = self._as_matrix(X)
        if not (X_matrix.shape[0] == y_array.size == t_array.size):
            raise ValueError("X, y và t phải có cùng số hàng.")
        periods = np.full(y_array.size, int(source_period), dtype=np.int64)
        self.X = (
            X_matrix
            if self.X is None
            else self._vstack([self.X, X_matrix])
        )
        self.y = np.concatenate([self.y, y_array])
        self.t = np.concatenate([self.t, t_array])
        self.source_period = np.concatenate([self.source_period, periods])
        self._cap_to_recent()

    def _cap_to_recent(self) -> None:
        if len(self) <= self.unique_cap:
            return
        month_ids = self.t.astype("datetime64[M]").astype(np.int64)
        chosen: list[np.ndarray] = []
        remaining = self.unique_cap
        for month_id in np.sort(np.unique(month_ids))[::-1]:
            candidates = np.flatnonzero(month_ids == month_id)
            if candidates.size <= remaining:
                chosen.append(candidates)
                remaining -= int(candidates.size)
            else:
                local_seed = (
                    self.seed + 2654435761 * int(month_id)
                ) % (2**32)
                rng = np.random.default_rng(local_seed)
                chosen.append(
                    np.sort(rng.choice(candidates, remaining, replace=False))
                )
                remaining = 0
            if remaining == 0:
                break
        indexes = np.sort(np.concatenate(chosen).astype(np.int64, copy=False))
        if indexes.size != self.unique_cap:
            raise RuntimeError("Không dựng được bộ đệm gần nhất đúng kích thước.")
        assert self.X is not None
        self.X = self.X[indexes]
        self.y = self.y[indexes]
        self.t = self.t[indexes]
        self.source_period = self.source_period[indexes]


class BalancedHardReplayBuffer:
    """Bộ nhớ tái phát cân bằng giữa FN, FP khó và mẫu nền gần đây.

    Các ngăn là độc quyền theo dự đoán đã khóa tại lúc nhãn được mở:

    * ``fn``: nhãn thật 1, dự đoán nhị phân 0;
    * ``fp``: nhãn thật 0, dự đoán nhị phân 1;
    * ``background``: dữ liệu huấn luyện khởi tạo cùng TP/TN mới.

    Quota là quota hàng huấn luyện, không phải quota nhãn mới. Nếu ngăn FN/FP
    chưa đủ, các mẫu nền gần nhất bù phần trống. Mỗi mẫu FN/FP không vượt quá
    giới hạn lặp đã khóa.
    """

    def __init__(
        self,
        memory_cap: int = 5000,
        fn_fraction: float = 0.35,
        fp_fraction: float = 0.25,
        fn_max_repeat: int = 4,
        fp_max_repeat: int = 2,
        seed: int = 0,
    ) -> None:
        if int(memory_cap) <= 0:
            raise ValueError("memory_cap phải dương.")
        if not (0.0 <= fn_fraction <= 1.0 and 0.0 <= fp_fraction <= 1.0):
            raise ValueError("Các tỷ lệ phải thuộc [0, 1].")
        if fn_fraction + fp_fraction > 1.0:
            raise ValueError("Tổng tỷ lệ FN và FP không được vượt 1.")
        if int(fn_max_repeat) < 1 or int(fp_max_repeat) < 1:
            raise ValueError("Giới hạn lặp phải ít nhất bằng 1.")

        self.memory_cap = int(memory_cap)
        self.fn_fraction = float(fn_fraction)
        self.fp_fraction = float(fp_fraction)
        self.fn_max_repeat = int(fn_max_repeat)
        self.fp_max_repeat = int(fp_max_repeat)
        self.seed = int(seed)
        self._buckets = {
            "fn": _Bucket("fn", self.memory_cap, self.seed + 11),
            "fp": _Bucket("fp", self.memory_cap, self.seed + 23),
            "background": _Bucket(
                "background", self.memory_cap, self.seed + 37
            ),
        }

    def initialize(self, X: Any, y: np.ndarray, t: np.ndarray) -> None:
        """Đưa tập huấn luyện ban đầu vào ngăn nền, với nguồn trước test."""
        if any(len(bucket) for bucket in self._buckets.values()):
            raise RuntimeError("BHR chỉ được khởi tạo một lần.")
        y_array = np.asarray(y, dtype=np.int8).reshape(-1)
        if np.setdiff1d(np.unique(y_array), [0, 1]).size:
            raise ValueError("BHR yêu cầu nhãn nhị phân 0/1.")
        self._buckets["background"].append(X, y_array, t, source_period=-1)

    def observe(
        self,
        X: Any,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        t: np.ndarray,
        source_period: int,
    ) -> None:
        """Ghi nhãn đã mở sau dự đoán; chưa dựng bộ nhớ cho cùng chu kỳ."""
        if int(source_period) < 0:
            raise ValueError("source_period của phản hồi phải không âm.")
        y_array = np.asarray(y_true, dtype=np.int8).reshape(-1)
        pred_array = np.asarray(y_pred, dtype=np.int8).reshape(-1)
        t_array = np.asarray(t).reshape(-1)
        X_matrix = _Bucket._as_matrix(X)
        if not (
            X_matrix.shape[0] == y_array.size == pred_array.size == t_array.size
        ):
            raise ValueError("X, y_true, y_pred và t phải có cùng số hàng.")
        if np.setdiff1d(np.unique(y_array), [0, 1]).size:
            raise ValueError("y_true phải là nhãn nhị phân 0/1.")
        if np.setdiff1d(np.unique(pred_array), [0, 1]).size:
            raise ValueError("y_pred phải là dự đoán nhị phân 0/1.")

        masks = {
            "fn": (y_array == 1) & (pred_array == 0),
            "fp": (y_array == 0) & (pred_array == 1),
        }
        masks["background"] = ~(masks["fn"] | masks["fp"])
        for name, mask in masks.items():
            indexes = np.flatnonzero(mask)
            self._buckets[name].append(
                X_matrix[indexes], y_array[indexes], t_array[indexes], source_period
            )

    @staticmethod
    def _allocate(
        bucket_size: int,
        target: int,
        max_repeat: int,
        existing_counts: np.ndarray | None = None,
    ) -> tuple[list[int], np.ndarray]:
        counts = (
            np.zeros(bucket_size, dtype=np.int64)
            if existing_counts is None
            else np.asarray(existing_counts, dtype=np.int64).copy()
        )
        selected: list[int] = []
        # Chỉ số lớn hơn là mẫu mới hơn vì mọi lần append đi theo thời gian.
        recent_first = np.arange(bucket_size - 1, -1, -1, dtype=np.int64)
        while len(selected) < int(target):
            eligible = recent_first[counts[recent_first] < int(max_repeat)]
            if eligible.size == 0:
                break
            remaining = int(target) - len(selected)
            take = eligible[:remaining]
            selected.extend(int(index) for index in take)
            counts[take] += 1
        return selected, counts

    def build(self, fit_period: int) -> ReplayBatch:
        """Dựng bộ nhớ cho ``fit_period`` và cưỡng chế trễ phản hồi một chu kỳ."""
        fit_period = int(fit_period)
        newest = max(
            (
                int(bucket.source_period.max())
                for bucket in self._buckets.values()
                if len(bucket)
            ),
            default=-1,
        )
        if newest >= fit_period:
            raise ValueError(
                "Vi phạm nhân quả: phản hồi của chu kỳ hiện tại không được fit ngay."
            )

        fn_target = int(np.floor(self.memory_cap * self.fn_fraction))
        fp_target = int(np.floor(self.memory_cap * self.fp_fraction))
        background_target = self.memory_cap - fn_target - fp_target

        allocations: dict[str, list[int]] = {}
        counts: dict[str, np.ndarray] = {}
        allocations["fn"], counts["fn"] = self._allocate(
            len(self._buckets["fn"]), fn_target, self.fn_max_repeat
        )
        allocations["fp"], counts["fp"] = self._allocate(
            len(self._buckets["fp"]), fp_target, self.fp_max_repeat
        )
        allocations["background"], counts["background"] = self._allocate(
            len(self._buckets["background"]), background_target, 1
        )

        remaining = self.memory_cap - sum(len(items) for items in allocations.values())
        # Quota hard còn trống được bù bởi nền gần đây; sau đó mới dùng phần
        # sức chứa lặp hard còn lại. Không mẫu nào vượt giới hạn lặp đã khóa.
        for name, repeat_limit in (
            ("background", 1),
            ("fn", self.fn_max_repeat),
            ("fp", self.fp_max_repeat),
        ):
            if remaining <= 0:
                break
            extra, updated = self._allocate(
                len(self._buckets[name]), remaining, repeat_limit, counts[name]
            )
            allocations[name].extend(extra)
            counts[name] = updated
            remaining -= len(extra)

        matrices: list[Any] = []
        labels: list[np.ndarray] = []
        times: list[np.ndarray] = []
        source_periods: list[np.ndarray] = []
        bucket_names: list[np.ndarray] = []
        for name in ("background", "fn", "fp"):
            indexes = np.asarray(allocations[name], dtype=np.int64)
            if indexes.size == 0:
                continue
            bucket = self._buckets[name]
            assert bucket.X is not None
            matrices.append(bucket.X[indexes])
            labels.append(bucket.y[indexes])
            times.append(bucket.t[indexes])
            source_periods.append(bucket.source_period[indexes])
            bucket_names.append(np.full(indexes.size, name, dtype="U10"))

        if not matrices:
            raise RuntimeError("BHR chưa có dữ liệu để dựng bộ nhớ.")
        X_out = _Bucket._vstack(matrices)
        y_out = np.concatenate(labels)
        t_out = np.concatenate(times)
        periods_out = np.concatenate(source_periods)
        names_out = np.concatenate(bucket_names)

        rng = np.random.default_rng(
            (self.seed + 1000003 * fit_period) % (2**32)
        )
        order = rng.permutation(y_out.size)
        X_out = X_out[order]
        y_out = y_out[order]
        t_out = t_out[order]
        periods_out = periods_out[order]
        names_out = names_out[order]

        fn_rows = int(len(allocations["fn"]))
        fp_rows = int(len(allocations["fp"]))
        background_rows = int(len(allocations["background"]))
        fn_unique_used = int(np.count_nonzero(counts["fn"]))
        fp_unique_used = int(np.count_nonzero(counts["fp"]))
        background_unique_used = int(np.count_nonzero(counts["background"]))
        unique_used = fn_unique_used + fp_unique_used + background_unique_used
        replay_rows = (fn_rows - fn_unique_used) + (fp_rows - fp_unique_used)
        labeled_ages = fit_period - periods_out[periods_out >= 0]
        telemetry = ReplayTelemetry(
            fit_period=fit_period,
            memory_cap=self.memory_cap,
            memory_rows=int(y_out.size),
            unique_rows_used=unique_used,
            fn_unique_buffer=len(self._buckets["fn"]),
            fp_unique_buffer=len(self._buckets["fp"]),
            background_unique_buffer=len(self._buckets["background"]),
            fn_rows=fn_rows,
            fp_rows=fp_rows,
            background_rows=background_rows,
            fn_replay_rows=max(0, fn_rows - fn_unique_used),
            fp_replay_rows=max(0, fp_rows - fp_unique_used),
            replay_fraction=(
                float(replay_rows / y_out.size) if y_out.size else 0.0
            ),
            mean_labeled_age_periods=(
                float(np.mean(labeled_ages)) if labeled_ages.size else float("nan")
            ),
            max_labeled_age_periods=(
                float(np.max(labeled_ages)) if labeled_ages.size else float("nan")
            ),
            newest_source_period=newest,
            causal_lag_ok=bool(newest < fit_period),
        )
        return ReplayBatch(
            X=X_out,
            y=y_out,
            t=t_out,
            source_period=periods_out,
            source_bucket=names_out,
            telemetry=telemetry,
        )

    def configuration(self) -> dict[str, Any]:
        return {
            "memory_cap": self.memory_cap,
            "fn_fraction": self.fn_fraction,
            "fp_fraction": self.fp_fraction,
            "background_fraction": 1.0 - self.fn_fraction - self.fp_fraction,
            "fn_max_repeat": self.fn_max_repeat,
            "fp_max_repeat": self.fp_max_repeat,
            "seed": self.seed,
            "feedback_timing": "observe_after_predict_t_then_fit_t_plus_1",
        }
'''
BALANCED_HARD_REPLAY_EMBEDDED_SHA256 = '4bf5ac205ee4d3a183005a27894893a28d41833232a478751518edfea354b64a'

COORDINATED_BALANCED_HARD_REPLAY_EMBEDDED_SOURCE = r'''"""Điều phối BHR để bổ trợ, không khuếch đại trùng lặp DRMD-FN."""

from __future__ import annotations

from math import sqrt
from typing import Any

from .balanced_hard_replay import BalancedHardReplayBuffer, ReplayBatch


def coordinated_fn_fraction(base_fraction: float, fn_penalty: float) -> float:
    """Giảm quota phát lại FN theo căn bậc hai của mức phạt hiện hành.

    Khi hệ số phạt bằng 1, BHR giữ nguyên quota khóa trước. Khi hệ số phạt tăng,
    phần quota được giải phóng chuyển sang mẫu nền gần đây. Nhờ đó ảnh hưởng
    kết hợp tăng xấp xỉ theo căn bậc hai thay vì tích trực tiếp giữa trọng số
    phần thưởng và số lần phát lại.
    """

    if not 0.0 <= float(base_fraction) <= 1.0:
        raise ValueError("base_fraction phải thuộc [0, 1]")
    if float(fn_penalty) < 1.0:
        raise ValueError("fn_penalty phải ít nhất bằng 1")
    return float(base_fraction) / sqrt(float(fn_penalty))


class CoordinatedBalancedHardReplayBuffer(BalancedHardReplayBuffer):
    """BHR có quota FN được điều phối bằng mức phạt dùng cho lần fit kế tiếp."""

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.base_fn_fraction = float(self.fn_fraction)
        self.last_coordination: dict[str, Any] = {}

    def build(self, fit_period: int, fn_penalty: float = 1.0) -> ReplayBatch:
        effective = coordinated_fn_fraction(
            self.base_fn_fraction, float(fn_penalty)
        )
        original = float(self.fn_fraction)
        self.fn_fraction = effective
        try:
            batch = super().build(fit_period=fit_period)
        finally:
            self.fn_fraction = original
        self.last_coordination = {
            "coordination_enabled": True,
            "coordination_rule": "base_fn_fraction_div_sqrt_fn_penalty",
            "base_fn_fraction": self.base_fn_fraction,
            "effective_fn_fraction": effective,
            "coordination_fn_penalty": float(fn_penalty),
        }
        return batch

    def configuration(self) -> dict[str, Any]:
        payload = super().configuration()
        payload.update({
            "base_fn_fraction": self.base_fn_fraction,
            "coordination_rule": "base_fn_fraction_div_sqrt_fn_penalty",
        })
        return payload
'''
COORDINATED_BALANCED_HARD_REPLAY_EMBEDDED_SHA256 = 'd3cd0bc24acb037956b53f0c7c0465b25d7fad4cedde19a2ed95503ac6c24e60'

RELIABLE_FN_PENALTY_EMBEDDED_SOURCE = r'''"""Bộ điều khiển phạt âm tính giả có cổng độ tin cậy cho DRMD-FN.

Mô-đun này tách biệt khỏi PyTorch để có thể kiểm thử độc lập. Bộ điều khiển
chỉ đọc nhãn của mẫu kiểm toán đã được chọn trước khi nhãn được tiết lộ. Mức
FNR mục tiêu và cận trên của hệ số phạt đều được suy ra từ bằng chứng kiểm
toán, không phải chọn bằng hiệu năng của các tháng kiểm thử tương lai.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from math import ceil, sqrt
from statistics import NormalDist
from typing import Any, Sequence

import numpy as np

try:
    from scipy.stats import beta as beta_distribution
except ModuleNotFoundError:  # pragma: no cover - Kaggle có SciPy
    beta_distribution = None


@dataclass(frozen=True)
class ReliableControllerSnapshot:
    """Dấu vết sau khi tiếp nhận phản hồi của một chu kỳ."""

    penalty_used: float
    penalty_next: float
    penalty_ceiling: float
    target_fnr: float
    posterior_mean_fnr: float
    posterior_ci_low: float
    posterior_ci_high: float
    audit_count: int
    audit_positive_count: int
    audit_false_negative_count: int
    recent_positive_count: int
    controller_step: int
    calibrated: bool
    update_applied: bool
    update_direction: str

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


def audit_sample_size(label_budget: int, period_size: int) -> int:
    """Số nhãn kiểm toán ``ceil(sqrt(B))`` trong tổng ngân sách ``B``."""

    budget = max(0, min(int(label_budget), int(period_size)))
    return 0 if budget == 0 else min(budget, max(1, ceil(sqrt(budget))))


def deterministic_audit_indices(
    period_size: int,
    label_budget: int,
    seed: int,
    period_index: int,
) -> np.ndarray:
    """Lấy mẫu kiểm toán đều, không hoàn lại và tái lập theo chu kỳ."""

    n_rows = int(period_size)
    if n_rows < 0:
        raise ValueError("period_size phải không âm")
    count = audit_sample_size(label_budget, n_rows)
    if count == 0:
        return np.asarray([], dtype=int)
    rng = np.random.default_rng(int(seed) + 1_000_003 * int(period_index))
    return np.sort(rng.choice(n_rows, size=count, replace=False).astype(int))


def merge_audit_with_active_selection(
    active_candidates: Sequence[int],
    audit_indices: Sequence[int],
    label_budget: int,
    period_size: int,
) -> np.ndarray:
    """Ghép mẫu kiểm toán và mẫu chủ động mà không vượt tổng ngân sách."""

    n_rows = int(period_size)
    total = max(0, min(int(label_budget), n_rows))
    audit = np.asarray(list(dict.fromkeys(map(int, audit_indices))), dtype=int)
    if np.any((audit < 0) | (audit >= n_rows)):
        raise ValueError("audit_indices chứa chỉ mục ngoài phạm vi")
    audit = audit[:total]
    selected = list(map(int, audit))
    selected_set = set(selected)
    for value in active_candidates:
        index = int(value)
        if index < 0 or index >= n_rows:
            raise ValueError("active_candidates chứa chỉ mục ngoài phạm vi")
        if index in selected_set:
            continue
        selected.append(index)
        selected_set.add(index)
        if len(selected) == total:
            break
    if len(selected) < total:
        for index in range(n_rows):
            if index not in selected_set:
                selected.append(index)
                selected_set.add(index)
            if len(selected) == total:
                break
    return np.asarray(selected, dtype=int)


class ReliableAdaptiveFNPenaltyController:
    """Điều khiển hệ số phạt FN bằng hậu nghiệm Beta và cập nhật nhân quả.

    Quy tắc gồm bốn lớp bảo vệ:

    1. FNR mục tiêu chỉ được khóa khi số mẫu mã độc kiểm toán tích lũy đạt
       ``2 * ceil(sqrt(B))`` và ít nhất tám mẫu.
    2. Mục tiêu là trung bình hậu nghiệm FNR của giai đoạn hiệu chuẩn. Mục tiêu
       vì vậy biểu diễn điểm vận hành đã quan sát trong quá khứ, không đặt ra
       một mức cải thiện tùy ý từ dữ liệu kiểm thử tương lai.
    3. Hệ số chỉ thay đổi khi khoảng tin cậy hậu nghiệm của cửa sổ gần nhất
       nằm hoàn toàn về một phía của mục tiêu.
    4. Cận trên ``1 + sqrt(ceil(sqrt(B)))`` tăng theo lượng bằng chứng kiểm
       toán của mỗi chu kỳ. Với B=15, cận trên bằng 3.
    """

    def __init__(
        self,
        *,
        label_budget: int,
        window_periods: int = 12,
        confidence_level: float = 0.90,
    ) -> None:
        if int(label_budget) <= 0:
            raise ValueError("label_budget phải dương")
        if int(window_periods) <= 0:
            raise ValueError("window_periods phải dương")
        if not 0.5 < float(confidence_level) < 1.0:
            raise ValueError("confidence_level phải thuộc (0.5, 1)")
        self.label_budget = int(label_budget)
        self.window_periods = int(window_periods)
        self.confidence_level = float(confidence_level)
        evidence_per_period = audit_sample_size(self.label_budget, self.label_budget)
        self.calibration_positive_support = max(8, 2 * evidence_per_period)
        self.update_positive_support = self.calibration_positive_support
        self.penalty_ceiling = 1.0 + sqrt(float(evidence_per_period))
        self.penalty = 1.0
        self._dual = 0.0
        self._target_fnr: float | None = None
        self._step = 0
        self._history: list[tuple[int, int]] = []

    @property
    def calibrated(self) -> bool:
        return self._target_fnr is not None

    @property
    def target_fnr(self) -> float:
        return (
            float(self._target_fnr)
            if self._target_fnr is not None
            else float("nan")
        )

    @staticmethod
    def _posterior(false_negative: int, positive: int) -> tuple[float, float]:
        alpha = float(false_negative) + 0.5
        beta = float(positive - false_negative) + 0.5
        mean = alpha / (alpha + beta)
        variance = alpha * beta / (
            (alpha + beta) ** 2 * (alpha + beta + 1.0)
        )
        return float(mean), float(sqrt(max(0.0, variance)))

    def observe(
        self,
        audit_y_true: Sequence[int],
        audit_y_pred: Sequence[int],
    ) -> ReliableControllerSnapshot:
        y_true = np.asarray(audit_y_true, dtype=int)
        y_pred = np.asarray(audit_y_pred, dtype=int)
        if y_true.shape != y_pred.shape:
            raise ValueError("Nhãn thật và dự đoán kiểm toán phải cùng kích thước")
        if np.any(~np.isin(y_true, [0, 1])) or np.any(~np.isin(y_pred, [0, 1])):
            raise ValueError("Bộ điều khiển chỉ nhận nhãn nhị phân 0/1")

        used = float(self.penalty)
        positive = int(np.sum(y_true == 1))
        false_negative = int(np.sum((y_true == 1) & (y_pred == 0)))
        self._history.append((false_negative, positive))

        posterior_mean = float("nan")
        ci_low = float("nan")
        ci_high = float("nan")
        update_applied = False
        update_direction = "not_calibrated"
        recent_positive = 0

        total_fn = int(sum(item[0] for item in self._history))
        total_positive = int(sum(item[1] for item in self._history))
        if (
            self._target_fnr is None
            and total_positive >= self.calibration_positive_support
        ):
            baseline_mean, _ = self._posterior(total_fn, total_positive)
            self._target_fnr = float(baseline_mean)
            update_direction = "calibrated_hold"
        elif self._target_fnr is not None:
            recent = self._history[-self.window_periods :]
            recent_fn = int(sum(item[0] for item in recent))
            recent_positive = int(sum(item[1] for item in recent))
            if recent_positive < self.update_positive_support:
                update_direction = "insufficient_support_hold"
            else:
                alpha = float(recent_fn) + 0.5
                beta = float(recent_positive - recent_fn) + 0.5
                posterior_mean = float(alpha / (alpha + beta))
                tail = (1.0 - self.confidence_level) / 2.0
                if beta_distribution is not None:
                    ci_low = float(beta_distribution.ppf(tail, alpha, beta))
                    ci_high = float(beta_distribution.ppf(1.0 - tail, alpha, beta))
                else:
                    # Đường kiểm thử tối giản không có SciPy: xấp xỉ chuẩn
                    # của hậu nghiệm Beta, chặn vào miền xác suất. Artifact
                    # ghi rõ backend; Kaggle dùng quantile Beta chính xác.
                    variance = alpha * beta / (
                        (alpha + beta) ** 2 * (alpha + beta + 1.0)
                    )
                    z_value = NormalDist().inv_cdf(1.0 - tail)
                    half_width = z_value * sqrt(max(0.0, variance))
                    ci_low = float(max(0.0, posterior_mean - half_width))
                    ci_high = float(min(1.0, posterior_mean + half_width))
                self._step += 1
                step_size = 1.0 / sqrt(float(self._step))
                if ci_low > self._target_fnr:
                    self._dual += step_size * (ci_low - self._target_fnr)
                    update_direction = "increase"
                    update_applied = True
                elif ci_high < self._target_fnr:
                    self._dual -= step_size * (self._target_fnr - ci_high)
                    update_direction = "decrease"
                    update_applied = True
                else:
                    update_direction = "credible_interval_hold"
                max_dual = self.penalty_ceiling - 1.0
                self._dual = float(np.clip(self._dual, 0.0, max_dual))
                self.penalty = 1.0 + self._dual

        return ReliableControllerSnapshot(
            penalty_used=used,
            penalty_next=float(self.penalty),
            penalty_ceiling=float(self.penalty_ceiling),
            target_fnr=self.target_fnr,
            posterior_mean_fnr=posterior_mean,
            posterior_ci_low=ci_low,
            posterior_ci_high=ci_high,
            audit_count=int(y_true.size),
            audit_positive_count=positive,
            audit_false_negative_count=false_negative,
            recent_positive_count=recent_positive,
            controller_step=int(self._step),
            calibrated=self.calibrated,
            update_applied=bool(update_applied),
            update_direction=update_direction,
        )

    def configuration(self) -> dict[str, Any]:
        return {
            "label_budget": self.label_budget,
            "window_periods": self.window_periods,
            "confidence_level": self.confidence_level,
            "calibration_positive_support": self.calibration_positive_support,
            "update_positive_support": self.update_positive_support,
            "penalty_ceiling": self.penalty_ceiling,
            "target_rule": "jeffreys_posterior_mean_at_calibration",
            "update_rule": "credible_interval_gated_robbins_monro",
            "credible_interval_backend": (
                "scipy_beta_ppf" if beta_distribution is not None
                else "normal_approximation_to_beta_posterior"
            ),
        }
'''
RELIABLE_FN_PENALTY_EMBEDDED_SHA256 = '15eac96ac030a7cc7f78c0d4110abdbc517e16e22004b0183cbec6141d9bfeda'

DRIFT_AWARE_FN_PENALTY_EMBEDDED_SOURCE = r'''"""Cổng trôi dạt không giám sát và bộ điều khiển phạt FN của V15.

Điểm trôi dạt chỉ được tính từ ma trận đặc trưng chưa gán nhãn ``X_t``. Ảnh
hưởng của tín hiệu này được trì hoãn: hệ số mới chỉ được dùng từ lần huấn
luyện ở chu kỳ ``t + 1``. Mô-đun không nhận nhãn lớp trong cổng trôi dạt và
không diễn giải dịch chuyển ``P(X)`` như bằng chứng trực tiếp của dịch chuyển
``P(Y|X)``.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from math import sqrt
from typing import Any, Iterable, Sequence

import numpy as np

try:  # Kaggle có SciPy; nhánh NumPy phục vụ kiểm thử tối giản.
    import scipy.sparse as sp
except ModuleNotFoundError:  # pragma: no cover
    sp = None

from .reliable_fn_penalty import (
    ReliableAdaptiveFNPenaltyController,
    ReliableControllerSnapshot,
    audit_sample_size,
)


@dataclass(frozen=True)
class DriftGateSnapshot:
    """Dấu vết một lần quan sát phân phối đặc trưng chưa gán nhãn."""

    period_index: int
    score: float
    threshold: float
    triggered: bool
    trigger_strength: float
    reference_count: int
    score_history_before: int
    period_size: int
    rows_used: int
    unlabeled_only: bool = True

    def to_dict(self) -> dict[str, Any]:
        payload = asdict(self)
        # JSON và phép so sánh tái lập không có ngữ nghĩa chuẩn cho NaN.
        # Ngưỡng chưa đủ lịch sử được xuất thành null, còn thuộc tính Python
        # vẫn giữ NaN để biểu thị đại lượng chưa xác định.
        if not np.isfinite(float(self.threshold)):
            payload["threshold"] = None
        return payload


@dataclass(frozen=True)
class DriftAwareControllerSnapshot:
    """Dấu vết cập nhật kết hợp nhãn kiểm toán và cảnh báo trôi dạt."""

    penalty_used: float
    penalty_next: float
    penalty_ceiling: float
    label_penalty_next: float
    drift_dual_used: float
    drift_dual_next: float
    drift_score: float
    drift_threshold: float
    drift_triggered: bool
    drift_trigger_strength: float
    drift_step: int
    audit_count: int
    audit_positive_count: int
    audit_false_negative_count: int
    target_fnr: float
    posterior_mean_fnr: float
    posterior_ci_low: float
    posterior_ci_high: float
    recent_positive_count: int
    calibrated: bool
    update_applied: bool
    update_direction: str
    label_update_direction: str
    causal_lag_ok: bool

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


class ProjectionMomentMMDGate:
    """Xấp xỉ MMD tuyến tính trên moment của phép chiếu ngẫu nhiên cố định.

    Mỗi hàng được chuẩn hóa L2, sau đó chiếu bằng ma trận Rademacher cố định.
    Tóm tắt của một chu kỳ gồm trung bình và moment bậc hai của các tọa độ
    chiếu. Bình phương khoảng cách giữa tóm tắt hiện tại và tâm cửa sổ tham
    chiếu tương đương MMD với kernel tuyến tính trên vector moment mở rộng.

    Ngưỡng ở chu kỳ ``t`` là phân vị của *chỉ các điểm quá khứ*. Điểm hiện tại
    được thêm vào lịch sử sau khi quyết định kích hoạt đã được khóa.
    """

    def __init__(
        self,
        *,
        input_dim: int,
        projection_dim: int = 32,
        reference_window: int = 12,
        threshold_quantile: float = 0.95,
        minimum_score_history: int = 4,
        max_rows_per_period: int = 2048,
        seed: int = 0,
    ) -> None:
        if int(input_dim) <= 0:
            raise ValueError("input_dim phải dương")
        if int(projection_dim) <= 0:
            raise ValueError("projection_dim phải dương")
        if int(reference_window) <= 0:
            raise ValueError("reference_window phải dương")
        if not 0.5 < float(threshold_quantile) < 1.0:
            raise ValueError("threshold_quantile phải thuộc (0.5, 1)")
        if int(minimum_score_history) <= 0:
            raise ValueError("minimum_score_history phải dương")
        if int(max_rows_per_period) <= 0:
            raise ValueError("max_rows_per_period phải dương")

        self.input_dim = int(input_dim)
        self.projection_dim = int(projection_dim)
        self.reference_window = int(reference_window)
        self.threshold_quantile = float(threshold_quantile)
        self.minimum_score_history = int(minimum_score_history)
        self.max_rows_per_period = int(max_rows_per_period)
        self.seed = int(seed)
        rng = np.random.default_rng(self.seed)
        self._projection = rng.choice(
            np.asarray([-1.0, 1.0], dtype=np.float32),
            size=(self.input_dim, self.projection_dim),
            replace=True,
        ) / sqrt(float(self.projection_dim))
        self._reference_summaries: list[np.ndarray] = []
        self.score_history: list[float] = []
        self._period_index = 0

    @staticmethod
    def _as_matrix(X: Any) -> Any:
        if sp is not None and sp.issparse(X):
            matrix = X.tocsr().astype(np.float32)
        else:
            matrix = np.asarray(X, dtype=np.float32)
        if matrix.ndim != 2:
            raise ValueError("X phải là ma trận hai chiều")
        return matrix

    def _sample_rows(self, X: Any, period_index: int) -> Any:
        n_rows = int(X.shape[0])
        if n_rows <= self.max_rows_per_period:
            return X
        rng = np.random.default_rng(
            (self.seed + 1_000_003 * int(period_index)) % (2**32)
        )
        indexes = np.sort(
            rng.choice(n_rows, self.max_rows_per_period, replace=False)
        )
        return X[indexes]

    def _summary(self, X: Any, period_index: int) -> tuple[np.ndarray, int]:
        matrix = self._as_matrix(X)
        if int(matrix.shape[1]) != self.input_dim:
            raise ValueError(
                f"X có {matrix.shape[1]} chiều, kỳ vọng {self.input_dim}"
            )
        sampled = self._sample_rows(matrix, period_index)
        rows_used = int(sampled.shape[0])
        if rows_used == 0:
            raise ValueError("Không thể ước lượng trôi dạt từ chu kỳ rỗng")

        if sp is not None and sp.issparse(sampled):
            norms = np.sqrt(sampled.multiply(sampled).sum(axis=1)).A1
            inverse = np.divide(
                1.0,
                norms,
                out=np.zeros_like(norms, dtype=np.float32),
                where=norms > 0,
            )
            normalized = sampled.multiply(inverse[:, None])
            projected = np.asarray(normalized @ self._projection)
        else:
            dense = np.asarray(sampled, dtype=np.float32)
            norms = np.linalg.norm(dense, axis=1, keepdims=True)
            normalized = np.divide(
                dense,
                norms,
                out=np.zeros_like(dense),
                where=norms > 0,
            )
            projected = normalized @ self._projection
        mean = np.mean(projected, axis=0, dtype=np.float64)
        second_moment = np.mean(
            np.square(projected), axis=0, dtype=np.float64
        )
        return np.concatenate([mean, second_moment]), rows_used

    def observe(self, X: Any) -> DriftGateSnapshot:
        period_index = int(self._period_index)
        matrix = self._as_matrix(X)
        period_size = int(matrix.shape[0])
        summary, rows_used = self._summary(matrix, period_index)
        reference_count = len(self._reference_summaries)
        if reference_count:
            reference_center = np.mean(
                np.vstack(self._reference_summaries), axis=0
            )
            score = float(np.mean(np.square(summary - reference_center)))
        else:
            score = 0.0

        history_before = len(self.score_history)
        threshold = float("nan")
        triggered = False
        strength = 0.0
        if history_before >= self.minimum_score_history:
            threshold = float(
                np.quantile(self.score_history, self.threshold_quantile)
            )
            triggered = bool(score > threshold)
            if triggered:
                denominator = abs(score) + abs(threshold) + np.finfo(float).eps
                strength = float(np.clip((score - threshold) / denominator, 0.0, 1.0))

        snapshot = DriftGateSnapshot(
            period_index=period_index,
            score=score,
            threshold=threshold,
            triggered=triggered,
            trigger_strength=strength,
            reference_count=reference_count,
            score_history_before=history_before,
            period_size=period_size,
            rows_used=rows_used,
            unlabeled_only=True,
        )

        self.score_history.append(score)
        self._reference_summaries.append(summary)
        self._reference_summaries = self._reference_summaries[
            -self.reference_window :
        ]
        self._period_index += 1
        return snapshot

    def calibrate(self, blocks: Iterable[Any]) -> list[DriftGateSnapshot]:
        if self._reference_summaries or self.score_history or self._period_index:
            raise RuntimeError("Cổng trôi dạt chỉ được hiệu chuẩn một lần")
        snapshots = [self.observe(block) for block in blocks]
        if not snapshots:
            raise ValueError("Cần ít nhất một khối quá khứ để hiệu chuẩn")
        return snapshots

    def configuration(self) -> dict[str, Any]:
        return {
            "input_dim": self.input_dim,
            "projection_dim": self.projection_dim,
            "reference_window": self.reference_window,
            "threshold_quantile": self.threshold_quantile,
            "minimum_score_history": self.minimum_score_history,
            "max_rows_per_period": self.max_rows_per_period,
            "seed": self.seed,
            "score": "linear_mmd_on_projected_first_and_second_moments",
            "threshold_history": "past_scores_only",
            "input": "unlabeled_X_only",
        }


class DriftAwareFNPenaltyController:
    """Ghép bộ điều khiển FN có cổng tin cậy với biến trôi dạt bị chặn."""

    def __init__(
        self,
        *,
        label_budget: int,
        label_window_periods: int = 12,
        label_confidence_level: float = 0.90,
        drift_projection_dim: int = 32,
        drift_reference_window: int = 12,
        drift_threshold_quantile: float = 0.95,
        drift_minimum_score_history: int = 4,
        drift_max_rows_per_period: int = 2048,
        drift_decay_periods: int = 12,
    ) -> None:
        if int(drift_decay_periods) <= 0:
            raise ValueError("drift_decay_periods phải dương")
        self.label_budget = int(label_budget)
        self.label_controller = ReliableAdaptiveFNPenaltyController(
            label_budget=self.label_budget,
            window_periods=label_window_periods,
            confidence_level=label_confidence_level,
        )
        self.drift_projection_dim = int(drift_projection_dim)
        self.drift_reference_window = int(drift_reference_window)
        self.drift_threshold_quantile = float(drift_threshold_quantile)
        self.drift_minimum_score_history = int(drift_minimum_score_history)
        self.drift_max_rows_per_period = int(drift_max_rows_per_period)
        self.drift_decay_periods = int(drift_decay_periods)
        self.penalty_ceiling = float(self.label_controller.penalty_ceiling)
        evidence = audit_sample_size(self.label_budget, self.label_budget)
        self._drift_increment_scale = 1.0 / sqrt(float(max(1, evidence)))
        self._drift_dual = 0.0
        self._drift_step = 0
        self._penalty = 1.0
        self.gate: ProjectionMomentMMDGate | None = None
        self._last_drift_period = -1

    @property
    def penalty(self) -> float:
        return float(self._penalty)

    def calibrate_unlabeled(
        self,
        blocks: Iterable[Any],
        *,
        input_dim: int,
        seed: int,
    ) -> list[DriftGateSnapshot]:
        if self.gate is not None:
            raise RuntimeError("Bộ điều khiển trôi dạt chỉ được hiệu chuẩn một lần")
        self.gate = ProjectionMomentMMDGate(
            input_dim=input_dim,
            projection_dim=self.drift_projection_dim,
            reference_window=self.drift_reference_window,
            threshold_quantile=self.drift_threshold_quantile,
            minimum_score_history=self.drift_minimum_score_history,
            max_rows_per_period=self.drift_max_rows_per_period,
            seed=seed,
        )
        return self.gate.calibrate(blocks)

    def observe_unlabeled(self, X_current: Any) -> DriftGateSnapshot:
        if self.gate is None:
            raise RuntimeError("Phải hiệu chuẩn cổng trôi dạt trước khi quan sát")
        snapshot = self.gate.observe(X_current)
        self._last_drift_period = int(snapshot.period_index)
        return snapshot

    def observe_labeled(
        self,
        audit_y_true: Sequence[int],
        audit_y_pred: Sequence[int],
        drift_snapshot: DriftGateSnapshot,
    ) -> DriftAwareControllerSnapshot:
        if not drift_snapshot.unlabeled_only:
            raise ValueError("Tín hiệu trôi dạt phải chỉ dùng đặc trưng chưa gán nhãn")
        if int(drift_snapshot.period_index) != self._last_drift_period:
            raise ValueError("Dấu vết trôi dạt không thuộc chu kỳ vừa quan sát")

        used = float(self._penalty)
        drift_used = float(self._drift_dual)
        label_snapshot: ReliableControllerSnapshot = self.label_controller.observe(
            audit_y_true, audit_y_pred
        )

        decay = 1.0 - 1.0 / float(self.drift_decay_periods)
        self._drift_dual *= decay
        if drift_snapshot.triggered:
            self._drift_step += 1
            self._drift_dual += (
                self._drift_increment_scale
                * float(drift_snapshot.trigger_strength)
                / sqrt(float(self._drift_step))
            )

        max_extra = max(
            0.0,
            self.penalty_ceiling - float(label_snapshot.penalty_next),
        )
        self._drift_dual = float(np.clip(self._drift_dual, 0.0, max_extra))
        self._penalty = float(
            np.clip(
                float(label_snapshot.penalty_next) + self._drift_dual,
                1.0,
                self.penalty_ceiling,
            )
        )

        return DriftAwareControllerSnapshot(
            penalty_used=used,
            penalty_next=self._penalty,
            penalty_ceiling=self.penalty_ceiling,
            label_penalty_next=float(label_snapshot.penalty_next),
            drift_dual_used=drift_used,
            drift_dual_next=float(self._drift_dual),
            drift_score=float(drift_snapshot.score),
            drift_threshold=float(drift_snapshot.threshold),
            drift_triggered=bool(drift_snapshot.triggered),
            drift_trigger_strength=float(drift_snapshot.trigger_strength),
            drift_step=int(self._drift_step),
            audit_count=int(label_snapshot.audit_count),
            audit_positive_count=int(label_snapshot.audit_positive_count),
            audit_false_negative_count=int(
                label_snapshot.audit_false_negative_count
            ),
            target_fnr=float(label_snapshot.target_fnr),
            posterior_mean_fnr=float(label_snapshot.posterior_mean_fnr),
            posterior_ci_low=float(label_snapshot.posterior_ci_low),
            posterior_ci_high=float(label_snapshot.posterior_ci_high),
            recent_positive_count=int(label_snapshot.recent_positive_count),
            calibrated=bool(label_snapshot.calibrated),
            update_applied=bool(
                label_snapshot.update_applied or drift_snapshot.triggered
            ),
            update_direction=(
                "drift_increase"
                if drift_snapshot.triggered
                else str(label_snapshot.update_direction)
            ),
            label_update_direction=str(label_snapshot.update_direction),
            causal_lag_ok=True,
        )

    def configuration(self) -> dict[str, Any]:
        return {
            "label_controller": self.label_controller.configuration(),
            "drift_projection_dim": self.drift_projection_dim,
            "drift_reference_window": self.drift_reference_window,
            "drift_threshold_quantile": self.drift_threshold_quantile,
            "drift_minimum_score_history": self.drift_minimum_score_history,
            "drift_max_rows_per_period": self.drift_max_rows_per_period,
            "drift_decay_periods": self.drift_decay_periods,
            "drift_increment_scale": self._drift_increment_scale,
            "combined_penalty_rule": "clip(label_dual_plus_decaying_drift_dual)",
            "causal_timing": "observe_X_t_then_update_lambda_for_fit_t_plus_1",
        }

    def calibration_state(self) -> dict[str, Any]:
        return {
            "gate_calibrated": self.gate is not None,
            "gate_configuration": (
                self.gate.configuration() if self.gate is not None else None
            ),
            "score_history_count": (
                len(self.gate.score_history) if self.gate is not None else 0
            ),
            "label_calibrated": self.label_controller.calibrated,
            "label_target_fnr": self.label_controller.target_fnr,
        }
'''
DRIFT_AWARE_FN_PENALTY_EMBEDDED_SHA256 = 'bb8dd3db5005cd3aea9e23743826c2920021bb27d4245929d5e8c2b6b0b957cd'

ADAPTIVE_BALANCED_HARD_REPLAY_EMBEDDED_SOURCE = r'''"""Bộ nhớ BHR V15 có quota thích nghi từ mẫu kiểm toán ngẫu nhiên.

Quota được ước lượng từ lịch sử nhãn kiểm toán, tách khỏi tập mẫu bất định mà
DRMD lựa chọn để học chủ động. Việc tách nguồn bằng chứng này tránh dùng một
tập đã bị thiên lệch bởi chính sách lựa chọn để ước lượng FNR/FPR vận hành.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Any, Sequence

import numpy as np

from .balanced_hard_replay import (
    BalancedHardReplayBuffer,
    ReplayBatch,
    ReplayTelemetry,
)
from .reliable_fn_penalty import audit_sample_size


@dataclass(frozen=True)
class AdaptiveQuotaSnapshot:
    """Phân bổ quota được yêu cầu sau phản hồi kiểm toán của một chu kỳ."""

    audit_count: int
    audit_false_negative_count: int
    audit_false_positive_count: int
    recent_positive_support: int
    recent_negative_support: int
    posterior_fnr: float
    posterior_fpr: float
    positive_reliability: float
    negative_reliability: float
    fn_penalty: float
    fn_risk: float
    fp_risk: float
    requested_hard_fraction: float
    requested_fn_fraction: float
    requested_fp_fraction: float
    requested_background_fraction: float
    evidence_source: str = "random_audit_only"

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class AdaptiveReplayTelemetry(ReplayTelemetry):
    """Dấu vết quota yêu cầu, khả dụng và thực tế của bộ nhớ V15."""

    requested_fn_fraction: float = 0.0
    requested_fp_fraction: float = 0.0
    requested_hard_fraction: float = 0.0
    requested_background_fraction: float = 1.0
    capacity_limited_fn_fraction: float = 0.0
    capacity_limited_fp_fraction: float = 0.0
    effective_fn_fraction: float = 0.0
    effective_fp_fraction: float = 0.0
    effective_hard_fraction: float = 0.0
    effective_background_fraction: float = 1.0
    fn_capacity_limited: bool = False
    fp_capacity_limited: bool = False
    posterior_fnr: float = float("nan")
    posterior_fpr: float = float("nan")
    positive_support: int = 0
    negative_support: int = 0
    quota_evidence_source: str = "random_audit_only"


class AdaptiveReplayQuotaController:
    """Ước lượng rủi ro FN/FP bằng cửa sổ Jeffreys và cấp quota động."""

    def __init__(
        self,
        *,
        label_budget: int,
        window_periods: int = 12,
        maximum_hard_fraction: float = 0.60,
    ) -> None:
        if int(label_budget) <= 0:
            raise ValueError("label_budget phải dương")
        if int(window_periods) <= 0:
            raise ValueError("window_periods phải dương")
        if not 0.0 < float(maximum_hard_fraction) < 1.0:
            raise ValueError("maximum_hard_fraction phải thuộc (0, 1)")
        self.label_budget = int(label_budget)
        self.window_periods = int(window_periods)
        self.maximum_hard_fraction = float(maximum_hard_fraction)
        per_period = audit_sample_size(self.label_budget, self.label_budget)
        self.support_target = max(8, 2 * int(per_period))
        self._history: list[tuple[int, int, int, int]] = []
        self.last_snapshot: AdaptiveQuotaSnapshot | None = None

    @staticmethod
    def _validate_binary(values: Sequence[int], name: str) -> np.ndarray:
        array = np.asarray(values, dtype=int).reshape(-1)
        if np.any(~np.isin(array, [0, 1])):
            raise ValueError(f"{name} chỉ nhận nhãn nhị phân 0/1")
        return array

    def observe(
        self,
        audit_y_true: Sequence[int],
        audit_y_pred: Sequence[int],
        *,
        fn_penalty: float,
    ) -> AdaptiveQuotaSnapshot:
        if float(fn_penalty) < 1.0:
            raise ValueError("fn_penalty phải ít nhất bằng 1")
        y_true = self._validate_binary(audit_y_true, "audit_y_true")
        y_pred = self._validate_binary(audit_y_pred, "audit_y_pred")
        if y_true.shape != y_pred.shape:
            raise ValueError("Nhãn thật và dự đoán kiểm toán phải cùng kích thước")

        positive = int(np.sum(y_true == 1))
        negative = int(np.sum(y_true == 0))
        false_negative = int(np.sum((y_true == 1) & (y_pred == 0)))
        false_positive = int(np.sum((y_true == 0) & (y_pred == 1)))
        self._history.append((false_negative, positive, false_positive, negative))
        recent = self._history[-self.window_periods :]
        recent_fn = int(sum(row[0] for row in recent))
        recent_positive = int(sum(row[1] for row in recent))
        recent_fp = int(sum(row[2] for row in recent))
        recent_negative = int(sum(row[3] for row in recent))

        posterior_fnr = float((recent_fn + 0.5) / (recent_positive + 1.0))
        posterior_fpr = float((recent_fp + 0.5) / (recent_negative + 1.0))
        positive_reliability = float(
            recent_positive / (recent_positive + self.support_target)
        )
        negative_reliability = float(
            recent_negative / (recent_negative + self.support_target)
        )
        fn_risk = float(fn_penalty) * posterior_fnr * positive_reliability
        fp_risk = posterior_fpr * negative_reliability
        total_risk = fn_risk + fp_risk
        requested_hard = min(
            self.maximum_hard_fraction,
            total_risk / (1.0 + total_risk),
        )
        if total_risk > 0.0:
            requested_fn = requested_hard * fn_risk / total_risk
            requested_fp = requested_hard * fp_risk / total_risk
        else:  # chỉ có thể xảy ra khi không có bằng chứng và rủi ro bằng 0
            requested_fn = 0.0
            requested_fp = 0.0
        requested_background = 1.0 - requested_fn - requested_fp

        snapshot = AdaptiveQuotaSnapshot(
            audit_count=int(y_true.size),
            audit_false_negative_count=false_negative,
            audit_false_positive_count=false_positive,
            recent_positive_support=recent_positive,
            recent_negative_support=recent_negative,
            posterior_fnr=posterior_fnr,
            posterior_fpr=posterior_fpr,
            positive_reliability=positive_reliability,
            negative_reliability=negative_reliability,
            fn_penalty=float(fn_penalty),
            fn_risk=fn_risk,
            fp_risk=fp_risk,
            requested_hard_fraction=float(requested_hard),
            requested_fn_fraction=float(requested_fn),
            requested_fp_fraction=float(requested_fp),
            requested_background_fraction=float(requested_background),
            evidence_source="random_audit_only",
        )
        self.last_snapshot = snapshot
        return snapshot

    def configuration(self) -> dict[str, Any]:
        return {
            "label_budget": self.label_budget,
            "window_periods": self.window_periods,
            "support_target": self.support_target,
            "maximum_hard_fraction": self.maximum_hard_fraction,
            "posterior": "beta_jeffreys_smoothed_error_rate",
            "reliability": "support_div_support_plus_target",
            "hard_fraction": "min(cap,risk_div_one_plus_risk)",
            "evidence_source": "random_audit_only",
        }


class AdaptiveBalancedHardReplayBuffer(BalancedHardReplayBuffer):
    """BHR với quota FN/FP co giãn theo rủi ro kiểm toán gần đây."""

    def __init__(
        self,
        *,
        memory_cap: int = 5000,
        fn_max_repeat: int = 4,
        fp_max_repeat: int = 2,
        label_budget: int = 15,
        quota_window_periods: int = 12,
        maximum_hard_fraction: float = 0.60,
        seed: int = 0,
    ) -> None:
        # Quota thực tế chỉ được thiết lập tạm thời trong build().
        super().__init__(
            memory_cap=memory_cap,
            fn_fraction=0.0,
            fp_fraction=0.0,
            fn_max_repeat=fn_max_repeat,
            fp_max_repeat=fp_max_repeat,
            seed=seed,
        )
        self.quota_controller = AdaptiveReplayQuotaController(
            label_budget=label_budget,
            window_periods=quota_window_periods,
            maximum_hard_fraction=maximum_hard_fraction,
        )
        self.last_allocation: dict[str, Any] = {}

    def observe_audit(
        self,
        audit_y_true: Sequence[int],
        audit_y_pred: Sequence[int],
        *,
        fn_penalty: float,
    ) -> AdaptiveQuotaSnapshot:
        """Cập nhật quota từ mẫu kiểm toán ngẫu nhiên, không lưu đặc trưng."""

        return self.quota_controller.observe(
            audit_y_true, audit_y_pred, fn_penalty=fn_penalty
        )

    def build(self, fit_period: int, fn_penalty: float = 1.0) -> ReplayBatch:
        snapshot = self.quota_controller.last_snapshot
        if snapshot is None:
            snapshot = self.quota_controller.observe(
                [], [], fn_penalty=fn_penalty
            )
        if not np.isclose(float(fn_penalty), float(snapshot.fn_penalty)):
            raise ValueError(
                "fn_penalty của lần fit phải khớp mức dùng để tính quota"
            )

        requested_fn_rows = int(
            np.floor(self.memory_cap * snapshot.requested_fn_fraction)
        )
        requested_fp_rows = int(
            np.floor(self.memory_cap * snapshot.requested_fp_fraction)
        )
        fn_capacity = len(self._buckets["fn"]) * self.fn_max_repeat
        fp_capacity = len(self._buckets["fp"]) * self.fp_max_repeat
        fn_target = min(requested_fn_rows, fn_capacity)
        fp_target = min(requested_fp_rows, fp_capacity)

        original_fn = float(self.fn_fraction)
        original_fp = float(self.fp_fraction)
        self.fn_fraction = float(fn_target / self.memory_cap)
        self.fp_fraction = float(fp_target / self.memory_cap)
        try:
            batch = super().build(fit_period=fit_period)
        finally:
            self.fn_fraction = original_fn
            self.fp_fraction = original_fp

        base = batch.telemetry
        denominator = float(max(1, base.memory_rows))
        effective_fn = float(base.fn_rows / denominator)
        effective_fp = float(base.fp_rows / denominator)
        effective_background = float(base.background_rows / denominator)
        effective_hard = effective_fn + effective_fp
        limited_fn = bool(fn_target < requested_fn_rows)
        limited_fp = bool(fp_target < requested_fp_rows)
        adaptive = AdaptiveReplayTelemetry(
            **asdict(base),
            requested_fn_fraction=float(snapshot.requested_fn_fraction),
            requested_fp_fraction=float(snapshot.requested_fp_fraction),
            requested_hard_fraction=float(snapshot.requested_hard_fraction),
            requested_background_fraction=float(
                snapshot.requested_background_fraction
            ),
            capacity_limited_fn_fraction=float(fn_target / self.memory_cap),
            capacity_limited_fp_fraction=float(fp_target / self.memory_cap),
            effective_fn_fraction=effective_fn,
            effective_fp_fraction=effective_fp,
            effective_hard_fraction=effective_hard,
            effective_background_fraction=effective_background,
            fn_capacity_limited=limited_fn,
            fp_capacity_limited=limited_fp,
            posterior_fnr=float(snapshot.posterior_fnr),
            posterior_fpr=float(snapshot.posterior_fpr),
            positive_support=int(snapshot.recent_positive_support),
            negative_support=int(snapshot.recent_negative_support),
            quota_evidence_source=str(snapshot.evidence_source),
        )
        self.last_allocation = {
            "requested_fn_fraction": float(snapshot.requested_fn_fraction),
            "requested_fp_fraction": float(snapshot.requested_fp_fraction),
            "requested_hard_fraction": float(snapshot.requested_hard_fraction),
            "requested_background_fraction": float(
                snapshot.requested_background_fraction
            ),
            "capacity_limited_fn_fraction": float(fn_target / self.memory_cap),
            "capacity_limited_fp_fraction": float(fp_target / self.memory_cap),
            "effective_fn_fraction": effective_fn,
            "effective_fp_fraction": effective_fp,
            "effective_hard_fraction": effective_hard,
            "effective_background_fraction": effective_background,
            "fn_capacity_limited": limited_fn,
            "fp_capacity_limited": limited_fp,
            "fn_unique_available": len(self._buckets["fn"]),
            "fp_unique_available": len(self._buckets["fp"]),
            "causal_lag_ok": bool(base.causal_lag_ok),
        }
        return ReplayBatch(
            X=batch.X,
            y=batch.y,
            t=batch.t,
            source_period=batch.source_period,
            source_bucket=batch.source_bucket,
            telemetry=adaptive,
        )

    def configuration(self) -> dict[str, Any]:
        payload = super().configuration()
        payload.update(
            {
                "quota_mode": "adaptive_from_random_audit",
                "quota_controller": self.quota_controller.configuration(),
                "unfillable_hard_quota": "transfer_to_recent_background",
            }
        )
        return payload
'''
ADAPTIVE_BALANCED_HARD_REPLAY_EMBEDDED_SHA256 = 'c56beae12feaa979130a9674ef7eb6e7ec11dd46f916187afad7466f0ae2c342'

STRATEGY_RUNTIME_ROOT = WORK / "_v15_embedded_source"
STRATEGY_MODULE_ROOT = STRATEGY_RUNTIME_ROOT / "experiments/strategies"
for _package_dir in (STRATEGY_RUNTIME_ROOT / "experiments", STRATEGY_MODULE_ROOT):
    _package_dir.mkdir(parents=True, exist_ok=True)
    (_package_dir / "__init__.py").write_text("", encoding="utf-8")
for _filename, _module_source, _expected_digest in (
    ('balanced_hard_replay.py', BALANCED_HARD_REPLAY_EMBEDDED_SOURCE, BALANCED_HARD_REPLAY_EMBEDDED_SHA256),
    ('coordinated_balanced_hard_replay.py', COORDINATED_BALANCED_HARD_REPLAY_EMBEDDED_SOURCE, COORDINATED_BALANCED_HARD_REPLAY_EMBEDDED_SHA256),
    ('reliable_fn_penalty.py', RELIABLE_FN_PENALTY_EMBEDDED_SOURCE, RELIABLE_FN_PENALTY_EMBEDDED_SHA256),
    ('drift_aware_fn_penalty.py', DRIFT_AWARE_FN_PENALTY_EMBEDDED_SOURCE, DRIFT_AWARE_FN_PENALTY_EMBEDDED_SHA256),
    ('adaptive_balanced_hard_replay.py', ADAPTIVE_BALANCED_HARD_REPLAY_EMBEDDED_SOURCE, ADAPTIVE_BALANCED_HARD_REPLAY_EMBEDDED_SHA256),
):
    _module_bytes = _module_source.encode("utf-8")
    if _strategy_hashlib.sha256(_module_bytes).hexdigest() != _expected_digest:
        raise RuntimeError(f"Mã chiến lược {_filename} không khớp SHA-256.")
    (STRATEGY_MODULE_ROOT / _filename).write_bytes(_module_bytes)
if str(STRATEGY_RUNTIME_ROOT) in sys.path:
    sys.path.remove(str(STRATEGY_RUNTIME_ROOT))
sys.path.insert(0, str(STRATEGY_RUNTIME_ROOT))
_strategy_importlib.invalidate_caches()
print({"embedded_strategy_root": str(STRATEGY_MODULE_ROOT),
       "modules": sorted(path.name for path in STRATEGY_MODULE_ROOT.glob("*.py"))})


In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHẦN 1: MONKEY-PATCH __init__ & __reward_function
# ═══════════════════════════════════════════════════════════════
import sys
import importlib
import torch

if REQUIRE_CUDA and KAGGLE_RUN_PHASE != "aggregate_only" and not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA is required. In Kaggle Settings, enable a GPU accelerator before Run All.")
import DRMD.environment as drmd_env

importlib.reload(drmd_env)
print("🔄 Reloaded DRMD.environment")

# ─── 1A: Patch __init__ để bổ sung hệ số phạt FN ───
_original_init = drmd_env.DRMD.__init__

def _patched_init(self, settings, *args, **kwargs):
    _original_init(self, settings, *args, **kwargs)
    self.fn_penalty = float(getattr(settings, '_fn_penalty', 1.0))
    if int(self.minority_label) != 1 or int(self.majority_label) != 0:
        raise AssertionError(
            "DRMD-FN yêu cầu nhãn mã độc=1 và lành tính=0; không dùng tần suất lớp để đổi nhãn.")
    self.adaptive_fn_penalty = bool(getattr(settings, '_adaptive_fn_penalty', False))
    self.afnp_audit_sampling = bool(getattr(settings, '_afnp_audit_sampling', False))
    self.afnp_budget = int(getattr(settings, '_afnp_budget', 0))
    self.afnp_seed = int(getattr(settings, '_afnp_seed', getattr(settings, 'seed', 0)))
    self.afnp_window = int(getattr(settings, '_afnp_window', 12))
    self.afnp_confidence = float(getattr(
        settings, '_afnp_confidence', FN_CONTROLLER_CONFIDENCE_LEVEL))
    self._drift_aware_fn = bool(getattr(
        settings, '_drift_aware_fn_penalty', False))
    self._adaptive_bhr_quota = bool(getattr(
        settings, '_adaptive_bhr_quota', False))
    self.afnp_controller = (
        ReliableAdaptiveFNPenaltyController(
            label_budget=self.afnp_budget,
            window_periods=self.afnp_window,
            confidence_level=self.afnp_confidence)
        if self.adaptive_fn_penalty and not self._drift_aware_fn else None)
    # Thuộc tính chiến lược gắn động, không sửa API kho DRMD tham chiếu.
    self._feedback_selector_mode = str(getattr(settings, '_feedback_selector_mode', 'iral'))
    self._feedback_budget = int(getattr(settings, '_feedback_budget', 0))
    self._candidate_multiplier = int(getattr(settings, '_candidate_multiplier', 10))
    self._full_feedback = bool(getattr(settings, '_full_feedback', False))
    self._training_history_cap = getattr(settings, '_training_history_cap', None)
    self._balanced_hard_replay = bool(getattr(settings, '_balanced_hard_replay', False))
    self._bhr_fn_fraction = float(getattr(settings, '_bhr_fn_fraction', 0.35))
    self._bhr_fp_fraction = float(getattr(settings, '_bhr_fp_fraction', 0.25))
    self._bhr_fn_max_repeat = int(getattr(settings, '_bhr_fn_max_repeat', 4))
    self._bhr_fp_max_repeat = int(getattr(settings, '_bhr_fp_max_repeat', 2))
    self._strategy_seed = int(settings.seed)
    self._fit_calls = 0
    self._selection_history = []
    self._temporal_reference_policy = FIXED_TEMPORAL_REFERENCE_POLICY
    self._temporal_reference_start = None
    self._temporal_reference_end = None
    self._temporal_reference_period = None
    self._current_memory_start = None
    self._current_memory_end = None
    self._current_memory_period = None
    self._temporal_reference_history = []
    if self.fn_penalty != 1.0:
        print(f"  FN Penalty = {self.fn_penalty} (Bất đối xứng)")

drmd_env.DRMD.__init__ = _patched_init
print(" Patched: DRMD.__init__ (FN penalty)")

# ─── 1B: Patch __reward_function (FN Penalty bất đối xứng) ───
_original_reward = drmd_env.DRMD._DRMD__reward_function

def _classification_reward(
        self, action, label, time=None, apply_fn_penalty=True):
    """Phần thưởng phân loại; không xử lý hành động từ chối."""
    reward = torch.where(
        action == label, self.correct_reward, self.incorrect_cost
    ).to(dtype=torch.float32, device=self.device)
    if apply_fn_penalty:
        fn_mask = (action == 0) & (label == self.minority_label)
        reward[fn_mask] *= float(getattr(self, 'fn_penalty', 1.0))
    reward *= torch.where(
        label == self.minority_label,
        self.minority_priority,
        self.majority_priority)
    if time is not None and self.temporal_rewards:
        temporal_position = (
            time + self.first_date - self.start_date + 1
        ) / self.training_period
        reward *= temporal_position * self.temporal_scaling
    return reward


def _patched_reward_function(self, action, label, time=None, obs=None):
    """FN chỉ đổi thưởng phân loại trực tiếp, không đổi thưởng từ chối.

    Nhánh phản thực của hành động từ chối dùng phần thưởng phân loại DRMD gốc
    (không phạt FN). Nhờ đó lambda không thể tự làm hành động từ chối hấp dẫn
    hơn chỉ vì hành động thay thế là một âm tính giả.
    """
    reward = _classification_reward(
        self, action, label, time=time, apply_fn_penalty=True)
    if self.is_reject_action:
        rejection_mask = action == self.reject_action
        reward[rejection_mask] = self.reject_cost
        if (obs is not None and self.reward_rejected_outcome
                and bool(torch.any(rejection_mask).item())):
            next_action = self.model.agent.get_next_likely_action(
                obs[rejection_mask], action[rejection_mask])
            rejected_outcome_reward = -_classification_reward(
                self, next_action, label[rejection_mask],
                time=None, apply_fn_penalty=False)
            scaling_factor = torch.where(
                rejected_outcome_reward > 0,
                self.reject_positive_scale,
                self.reject_negative_scale)
            reward[rejection_mask] += rejected_outcome_reward * scaling_factor
    return reward


drmd_env.DRMD._DRMD__reward_function = _patched_reward_function
print(" phạt FN tách khỏi phần thưởng phản thực của từ chối")

if 'DRMD.base' in sys.modules:
    importlib.reload(sys.modules['DRMD.base'])
    print("🔄 Reloaded DRMD.base")

# ═══════════════════════════════════════════════════════════════
# PHẦN 2: MONKEY-PATCH fit_predict_reject_sample_update
# ═══════════════════════════════════════════════════════════════
import DRMD.utils.classifier_utils as clf_utils
from copy import deepcopy
from tqdm import tqdm
import numpy as np
import scipy
from sklearn import metrics as skmetrics
from tesseract import utils as tess_utils, metrics as tess_metrics
from DRMD.utils.classifier_utils import UncertaintyRejector, UncertaintySelector
from experiments.strategies.coordinated_balanced_hard_replay import CoordinatedBalancedHardReplayBuffer
from experiments.strategies.adaptive_balanced_hard_replay import (
    AdaptiveBalancedHardReplayBuffer,
    AdaptiveReplayQuotaController,
)
from experiments.strategies.drift_aware_fn_penalty import DriftAwareFNPenaltyController, ProjectionMomentMMDGate
from experiments.strategies.reliable_fn_penalty import ReliableAdaptiveFNPenaltyController, deterministic_audit_indices

def _deterministic_recency_cap(X, y, t, cap, seed):
    """Giữ tháng gần nhất; lấy mẫu đều có seed tại tháng biên.

    Hàm không đọc nhãn để chọn hàng. Quy tắc tránh thiên lệch hệ thống
    do luôn giữ các hàng cuối khi một tháng vượt giới hạn bộ đệm; dấu
    vân tay nguồn vẫn là điều kiện để tái lập đúng tập hàng.
    """
    cap = int(cap)
    if cap <= 0:
        raise ValueError('Giới hạn lịch sử phải dương.')
    n_rows = int(len(y))
    if n_rows <= cap:
        return X, np.asarray(y), np.asarray(t)
    times = np.asarray(t)
    month_ids = times.astype('datetime64[M]').astype(np.int64)
    chosen, remaining = [], cap
    for month_id in np.sort(np.unique(month_ids))[::-1]:
        candidates = np.flatnonzero(month_ids == month_id)
        if candidates.size <= remaining:
            chosen.append(candidates)
            remaining -= int(candidates.size)
        else:
            local_seed = (
                int(seed) + 2654435761 * int(month_id)
            ) % (2 ** 32)
            rng = np.random.default_rng(local_seed)
            chosen.append(np.sort(rng.choice(
                candidates, size=remaining, replace=False)))
            remaining = 0
        if remaining == 0:
            break
    indexes = np.sort(np.concatenate(chosen).astype(int, copy=False))
    if indexes.size != cap:
        raise RuntimeError(
            f'Bộ đệm lịch sử có {indexes.size} hàng, kỳ vọng {cap}.')
    return X[indexes], np.asarray(y)[indexes], times[indexes]

def _apply_training_memory(clf, X, y, t):
    """Giữ đúng giới hạn 5.000 mẫu gần nhất bằng quy tắc đã khóa trong giao thức V15."""
    cap = getattr(clf, '_training_history_cap', None)
    if cap is None or X.shape[0] <= int(cap):
        return X, np.asarray(y), np.asarray(t)
    return _deterministic_recency_cap(
        X, y, t, int(cap), int(getattr(clf, '_strategy_seed', 0)))

def _actor_uncertainty(clf, X_batch, t_batch):
    """Tính bất định từ policy; không đọc nhãn của chu kỳ đang chọn."""
    actor = clf.model.agent.actor
    was_training = actor.training
    actor.eval()
    chunks = []
    try:
        with torch.no_grad():
            for start in range(0, X_batch.shape[0], max(1, int(BATCH_PREDICT_ROWS))):
                stop = min(start + max(1, int(BATCH_PREDICT_ROWS)), X_batch.shape[0])
                obs = _observation_chunk(clf, X_batch[start:stop], t_batch[start:stop])
                probabilities = torch.softmax(actor(obs), dim=-1)
                chunks.append((1.0 - probabilities.max(dim=1).values).cpu().numpy())
                del obs, probabilities
    finally:
        actor.train(was_training)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.concatenate(chunks).astype(np.float64, copy=False)

def _select_feedback_indexes(clf, X_test, t_test, rejected_indexes,
                             budget_override=None, excluded_indexes=()):
    """Dựng tập phản hồi sau khi khóa dự đoán, không đọc nhãn hiện tại."""
    selection_started = time.perf_counter()
    rejected = np.unique(np.asarray(rejected_indexes, dtype=int))
    excluded = np.unique(np.asarray(excluded_indexes, dtype=int))
    rejected = np.setdiff1d(rejected, excluded, assume_unique=False)
    mode = str(getattr(clf, '_feedback_selector_mode', 'iral'))
    budget = (int(getattr(clf, '_feedback_budget', 0))
              if budget_override is None else int(budget_override))
    available = np.setdiff1d(
        np.arange(int(X_test.shape[0]), dtype=int), excluded, assume_unique=False)
    base = {
        "selector_mode": mode, "budget": budget,
        "n_rejected": int(rejected.size), "n_rows": int(X_test.shape[0]),
        "selection_seconds": time.perf_counter() - selection_started,
    }
    if mode == 'iral':
        # IRAL nguyên gốc không có B cố định: mọi hành động từ chối tạo phản
        # hồi cho chu kỳ kế tiếp, tức Q_t = R_t.
        selected = rejected.copy()
        return selected, {
            **base, "budget_semantics": "endogenous_rejected_set",
            "candidate_pool_size": int(rejected.size),
            "n_selected": int(selected.size), "mean_uncertainty": float('nan')}
    if mode == 'full_feedback':
        # FFCR được ghi đè thành toàn bộ chỉ mục ở cuối pha quyết định. Trả về
        # ngay tại đây để không suy luận bất định thừa trên 885.947 mẫu.
        selected = available.copy()
        return selected, {
            **base, "budget_semantics": "full_feedback_reference",
            "candidate_pool_size": int(available.size),
            "n_selected": int(selected.size), "mean_uncertainty": float('nan')}
    if mode != 'uncertainty':
        raise ValueError(f'Bộ chọn phản hồi V15 không hợp lệ: {mode}')
    target = min(max(0, budget), int(available.size))
    uncertainty = _actor_uncertainty(clf, X_test, t_test)
    rejected_order = rejected[np.argsort(-uncertainty[rejected], kind='stable')]
    selected = rejected_order[:target].tolist()
    if len(selected) < target:
        remainder = np.setdiff1d(available, np.asarray(selected, dtype=int), assume_unique=False)
        remainder = remainder[np.argsort(-uncertainty[remainder], kind='stable')]
        selected.extend(remainder[:target - len(selected)].tolist())
    selected = np.asarray(selected, dtype=int)
    if selected.size != target or np.unique(selected).size != selected.size:
        raise RuntimeError(f'IRAAL chọn {selected.size} mẫu, kỳ vọng {target}.')
    return selected, {
        **base, "budget_semantics": "fixed_label_budget",
        "candidate_pool_size": int(available.size),
        "n_selected": int(selected.size),
        "selection_seconds": time.perf_counter() - selection_started,
        "mean_uncertainty": (float(np.mean(uncertainty[selected]))
                             if selected.size else float('nan')),
    }

def _patched_fprsu(clf, X_train, X_tests,
                   y_train, y_tests, t_train, t_tests,
                   fit_function=None, predict_function=None,
                   rebalancers=(), rejectors=(), selectors=(),
                   reject_label=None, select_reject_labeled=False, clf_uses_t=True,
                   posthoc_augmented_AL=0):

    fit_function = clf.fit if fit_function is None else fit_function
    predict_function = (tess_utils.select_prediction_function(clf, labels_only=True)
                        if predict_function is None else predict_function)

    for stage in tuple(rebalancers) + tuple(rejectors) + tuple(selectors):
        stage.resolve_schedule(len(X_tests))

    results = {}
    selected_indexes = None
    bhr_buffer = None
    if getattr(clf, '_balanced_hard_replay', False):
        if getattr(clf, '_adaptive_bhr_quota', False):
            bhr_buffer = AdaptiveBalancedHardReplayBuffer(
                memory_cap=int(clf._training_history_cap),
                fn_max_repeat=clf._bhr_fn_max_repeat,
                fp_max_repeat=clf._bhr_fp_max_repeat,
                label_budget=clf.afnp_budget,
                quota_window_periods=ADAPTIVE_BHR_WINDOW_PERIODS,
                maximum_hard_fraction=ADAPTIVE_BHR_MAX_HARD_FRACTION,
                seed=clf._strategy_seed)
        else:
            bhr_buffer = CoordinatedBalancedHardReplayBuffer(
                memory_cap=int(clf._training_history_cap),
                fn_fraction=clf._bhr_fn_fraction,
                fp_fraction=clf._bhr_fp_fraction,
                fn_max_repeat=clf._bhr_fn_max_repeat,
                fp_max_repeat=clf._bhr_fp_max_repeat,
                seed=clf._strategy_seed)
        bhr_buffer.initialize(X_train, y_train, t_train)

    drift_calibration = []
    if getattr(clf, '_drift_aware_fn', False):
        clf.afnp_controller = DriftAwareFNPenaltyController(
            label_budget=clf.afnp_budget,
            label_window_periods=clf.afnp_window,
            label_confidence_level=clf.afnp_confidence,
            drift_projection_dim=DRIFT_PROJECTION_DIM,
            drift_reference_window=DRIFT_REFERENCE_WINDOW,
            drift_threshold_quantile=DRIFT_THRESHOLD_QUANTILE,
            drift_minimum_score_history=DRIFT_MINIMUM_SCORE_HISTORY,
            drift_max_rows_per_period=DRIFT_MAX_ROWS_PER_PERIOD,
            drift_decay_periods=DRIFT_DECAY_PERIODS)
        _train_months = np.asarray(t_train).astype('datetime64[M]')
        _unlabeled_training_blocks = [
            X_train[np.flatnonzero(_train_months == month)]
            for month in np.sort(np.unique(_train_months))]
        drift_calibration = [snapshot.to_dict() for snapshot in
            clf.afnp_controller.calibrate_unlabeled(
                _unlabeled_training_blocks, input_dim=int(X_train.shape[1]),
                seed=clf._strategy_seed)]

    results['rejected_indexes_all'] = []
    results['selected_indexes_all'] = []
    results['audit_indexes_all'] = []
    results['fn_penalty_used'] = []
    results['fn_penalty_next'] = []
    results['afnp_target_fnr'] = []
    results['afnp_estimated_fnr'] = []
    results['fn_controller_telemetry'] = []
    results['drift_gate_telemetry'] = []
    results['adaptive_bhr_quota_telemetry'] = []
    results['drift_calibration_telemetry'] = drift_calibration
    results['audit_positive_count'] = []
    results['audit_fn_count'] = []
    results['audit_count'] = []
    results['bhr_telemetry'] = []

    results['rejected_y_true'] = []
    results['rejected_malware_count'] = []
    results['rejected_benign_count'] = []
    results['selected_y_true'] = []
    results['selected_malware_count'] = []
    results['selected_benign_count'] = []
    results['policy_action_all'] = []
    results['binary_y_true_all'] = []
    results['binary_y_preds_all'] = []
    results['binary_t_all'] = []
    results['selection_diagnostics'] = []

    global CURRENT_PERIOD_IDX
    for i, (X_test, y_test, t_test) in tqdm(
            enumerate(zip(X_tests, y_tests, t_tests)), total=len(X_tests),
            desc="Observed test periods"):
        CURRENT_PERIOD_IDX = i

        for rebalancer in rebalancers:
            if not rebalancer.schedule[i]:
                continue
            X_train, y_train, t_train = rebalancer.alter(
                clf, X_train, y_train, t_train, X_test, y_test, t_test)

        results = tess_metrics.get_train_info(
            X_train, y_train, t_train, existing=results)

        if ((selected_indexes is not None and len(selected_indexes) > 0)
                or i == 0):
            fit_function(X_train, y_train, t_train) if clf_uses_t else fit_function(X_train, y_train)

        kept_indexes, rejected_indexes, selected_indexes = None, None, None
        y_pred = predict_function(X_test, t_test) if clf_uses_t else predict_function(X_test)
        policy_action = np.asarray(y_pred, dtype=int).copy()
        results['policy_action_all'].append(policy_action.tolist())
        binary_pred = clf.predict_binary_all(X_test, t_test)
        results['binary_y_true_all'].append(np.asarray(y_test, dtype=int).tolist())
        results['binary_y_preds_all'].append(np.asarray(binary_pred, dtype=int).tolist())
        results['binary_t_all'].append(np.asarray(t_test).astype(str).tolist())

        # Cổng V15 chỉ đọc X_t. Dự đoán hiện tại đã khóa và tín hiệu này chỉ
        # được dùng để tạo hệ số cho lần fit t+1.
        drift_snapshot = None
        if getattr(clf, '_drift_aware_fn', False):
            drift_snapshot = clf.afnp_controller.observe_unlabeled(X_test)
            drift_telemetry = drift_snapshot.to_dict()
        else:
            drift_telemetry = {
                'period_index': int(i), 'score': float('nan'),
                'threshold': None, 'triggered': False,
                'trigger_strength': 0.0, 'reference_count': 0,
                'score_history_before': 0, 'period_size': int(len(y_test)),
                'rows_used': 0, 'unlabeled_only': True, 'enabled': False}
        results['drift_gate_telemetry'].append(drift_telemetry)

        # Mẫu kiểm toán được lấy đều mà không đọc nhãn. Phần còn lại của B mới
        # được giao cho bộ chọn bất định , do đó tổng số nhãn không vượt B.
        audit_indexes = np.array([], dtype=int)
        total_feedback_budget = min(
            int(getattr(clf, '_feedback_budget', 0)), int(len(y_test)))
        active_feedback_budget = total_feedback_budget
        if (getattr(clf, 'afnp_audit_sampling', False)
                and total_feedback_budget > 0
                and not getattr(clf, '_full_feedback', False)):
            audit_indexes = deterministic_audit_indices(
                len(y_test), total_feedback_budget, int(clf.afnp_seed), int(i))
            audit_count = int(audit_indexes.size)
            active_feedback_budget = total_feedback_budget - audit_count

        selection_diagnostic = {
            'selector_mode': str(getattr(clf, '_feedback_selector_mode', 'none')),
            'budget': int(getattr(clf, '_feedback_budget', 0)),
            'n_rows': int(len(y_test)), 'n_rejected': 0,
            'candidate_pool_size': 0, 'n_selected': 0}
        if reject_label is not None:
            y_pred_array = np.asarray(y_pred)
            rejected_indexes = np.flatnonzero(y_pred_array == reject_label).astype(int)
            kept_indexes = np.flatnonzero(y_pred_array != reject_label).astype(int)
            if rejected_indexes.size:
                y_pred_array[rejected_indexes] = np.asarray(
                    clf.predict_reject_alt(
                        X_test[rejected_indexes], t_test[rejected_indexes]),
                    dtype=int)
                y_pred = y_pred_array.tolist()
            if select_reject_labeled:
                active_indexes, selection_diagnostic = _select_feedback_indexes(
                    clf, X_test, t_test, rejected_indexes,
                    budget_override=active_feedback_budget,
                    excluded_indexes=audit_indexes)
                selected_indexes = np.concatenate(
                    (audit_indexes, np.asarray(active_indexes, dtype=int)))
                if np.unique(selected_indexes).size != selected_indexes.size:
                    raise RuntimeError('Mẫu kiểm toán và mẫu chủ động bị trùng.')
                selection_diagnostic.update({
                    'audit_count': int(audit_indexes.size),
                    'active_budget': int(active_feedback_budget),
                    'total_feedback_budget': int(total_feedback_budget)})

        for rejector in rejectors:
            if not rejector.schedule[i]:
                continue
            kept_indexes, rejected_indexes = rejector.reject_wrapper(
                clf, X_train, y_train, t_train,
                X_test, y_test, t_test,
                kept_indexes, rejected_indexes)

        for selector in selectors:
            if not selector.schedule[i]:
                continue
            selected_indexes = selector.query_wrapper(
                clf, X_train, y_train, t_train,
                X_test, y_test, t_test, selected_indexes)

        # Cập nhật sau dự đoán; penalty_next chỉ dùng từ lần fit t+1.
        lambda_used = float(getattr(clf, 'fn_penalty', 1.0))
        audit_y_true = np.asarray(y_test, dtype=int)[audit_indexes]
        audit_y_pred = np.asarray(binary_pred, dtype=int)[audit_indexes]
        if clf.afnp_controller is not None:
            if getattr(clf, '_drift_aware_fn', False):
                snapshot = clf.afnp_controller.observe_labeled(
                    audit_y_true, audit_y_pred, drift_snapshot)
            else:
                snapshot = clf.afnp_controller.observe(
                    audit_y_true, audit_y_pred)
            clf.fn_penalty = float(snapshot.penalty_next)
            controller_telemetry = snapshot.to_dict()
        else:
            controller_telemetry = {
                'penalty_used': lambda_used, 'penalty_next': lambda_used,
                'penalty_ceiling': 1.0, 'target_fnr': float('nan'),
                'posterior_mean_fnr': float('nan'),
                'posterior_ci_low': float('nan'),
                'posterior_ci_high': float('nan'),
                'audit_count': int(audit_indexes.size),
                'audit_positive_count': int(np.sum(audit_y_true == 1)),
                'audit_false_negative_count': int(np.sum(
                    (audit_y_true == 1) & (audit_y_pred == 0))),
                'recent_positive_count': 0, 'controller_step': 0,
                'calibrated': False, 'update_applied': False,
                'update_direction': 'disabled'}
        results['audit_indexes_all'].append(audit_indexes.tolist())
        results['fn_penalty_used'].append(lambda_used)
        results['fn_penalty_next'].append(float(clf.fn_penalty))
        results['afnp_target_fnr'].append(
            float(controller_telemetry.get('target_fnr', float('nan'))))
        results['afnp_estimated_fnr'].append(float(
            controller_telemetry.get('posterior_mean_fnr', float('nan'))))
        results['audit_positive_count'].append(
            int(controller_telemetry['audit_positive_count']))
        results['audit_fn_count'].append(
            int(controller_telemetry['audit_false_negative_count']))
        results['audit_count'].append(int(audit_indexes.size))
        results['fn_controller_telemetry'].append(controller_telemetry)

        if (bhr_buffer is not None
                and getattr(clf, '_adaptive_bhr_quota', False)):
            quota_snapshot = bhr_buffer.observe_audit(
                audit_y_true, audit_y_pred,
                fn_penalty=float(clf.fn_penalty))
            quota_telemetry = quota_snapshot.to_dict()
        else:
            quota_telemetry = {
                'enabled': False, 'audit_count': int(audit_indexes.size),
                'evidence_source': 'random_audit_only'}
        results['adaptive_bhr_quota_telemetry'].append(quota_telemetry)

        # Dự đoán tháng t hoàn tất trước khi nhãn tháng t được bổ sung.
        # Nhãn này chỉ tác động từ lần fit ở tháng t+1.
        if getattr(clf, '_full_feedback', False):
            selected_indexes = np.arange(len(y_test), dtype=int)
            selection_diagnostic.update({
                'selector_mode': 'full_feedback',
                'budget': int(len(y_test)),
                'candidate_pool_size': int(len(y_test)),
                'n_selected': int(len(y_test))})

        if rejected_indexes is not None and len(rejected_indexes) > 0:
            rej_y_true = y_test[rejected_indexes]
            results['rejected_y_true'].append(rej_y_true.tolist())
            results['rejected_malware_count'].append(int(np.sum(rej_y_true == 1)))
            results['rejected_benign_count'].append(int(np.sum(rej_y_true == 0)))
        else:
            results['rejected_y_true'].append([])
            results['rejected_malware_count'].append(0)
            results['rejected_benign_count'].append(0)

        if selected_indexes is not None and selected_indexes.shape[0] > 0:
            sel_y_true = y_test[selected_indexes]
            results['selected_y_true'].append(sel_y_true.tolist())
            results['selected_malware_count'].append(int(np.sum(sel_y_true == 1)))
            results['selected_benign_count'].append(int(np.sum(sel_y_true == 0)))

            X_selected = X_test[selected_indexes]
            y_selected = y_test[selected_indexes]
            t_selected = t_test[selected_indexes]
            if bhr_buffer is not None:
                bhr_buffer.observe(
                    X_selected, y_selected,
                    np.asarray(binary_pred)[selected_indexes], t_selected,
                    source_period=i)
            else:
                X_train = scipy.sparse.vstack((X_train, X_selected))
                y_train = np.hstack((y_train, y_selected))
                t_train = np.hstack((t_train, t_selected))
                X_train, y_train, t_train = _apply_training_memory(
                    clf, X_train, y_train, t_train)
            results['selected'].append(selected_indexes.size)
        else:
            results['selected_y_true'].append([])
            results['selected_malware_count'].append(0)
            results['selected_benign_count'].append(0)
            results['selected'].append(0)

        if bhr_buffer is not None:
            bhr_batch = bhr_buffer.build(
                fit_period=i + 1, fn_penalty=float(clf.fn_penalty))
            X_train, y_train, t_train = bhr_batch.X, bhr_batch.y, bhr_batch.t
            bhr_telemetry = bhr_batch.telemetry.to_dict()
            bhr_telemetry['enabled'] = True
            if getattr(clf, '_adaptive_bhr_quota', False):
                bhr_telemetry.update(bhr_buffer.last_allocation)
            else:
                bhr_telemetry.update(bhr_buffer.last_coordination)
        else:
            # Sau dự đoán đầu tiên, chuẩn hóa bộ nhớ cho mọi nhánh không BHR,
            # kể cả IRAL ở tháng không phát sinh hành động từ chối. Lần fit
            # khởi tạo phía trên vẫn dùng toàn bộ cửa sổ huấn luyện ban đầu.
            X_train, y_train, t_train = _apply_training_memory(
                clf, X_train, y_train, t_train)
            bhr_telemetry = {
                'enabled': False, 'fit_period': int(i + 1),
                'memory_rows': int(X_train.shape[0]),
                'causal_lag_ok': True,
            }
        results['bhr_telemetry'].append(bhr_telemetry)

        selection_diagnostic.update({
            'n_selected': int(0 if selected_indexes is None else len(selected_indexes)),
            'n_rejected_total': int(
                0 if rejected_indexes is None else len(rejected_indexes)),
            'selected_malware': int(results['selected_malware_count'][-1]),
            'selected_benign': int(results['selected_benign_count'][-1]),
            'rejected_malware': int(results['rejected_malware_count'][-1]),
            'rejected_benign': int(results['rejected_benign_count'][-1]),
            'selection_used_labels': False,
        })
        results['selection_diagnostics'].append(selection_diagnostic)
        clf._selection_history.append({
            'period_index': int(i), **selection_diagnostic})

        results['rejected_indexes_all'].append(
            rejected_indexes.tolist() if rejected_indexes is not None else [])
        results['selected_indexes_all'].append(
            selected_indexes.tolist() if selected_indexes is not None else [])

        if rejected_indexes is not None and len(rejected_indexes) > 0:
            y_pred_array = np.asarray(y_pred)
            rejected_true = y_test[rejected_indexes]
            rejected_pred = y_pred_array[rejected_indexes]
            _, rejected_fp, rejected_fn, rejected_tp = skmetrics.confusion_matrix(
                rejected_true, rejected_pred, labels=[0, 1]).ravel()
            rejected_f1_denominator = 2 * rejected_tp + rejected_fp + rejected_fn
            results['f1_r'].append(
                2 * rejected_tp / rejected_f1_denominator
                if rejected_f1_denominator else float('nan'))
            results['rejected'].append(len(rejected_indexes))
            if kept_indexes is not None and kept_indexes.shape[0] > 0:
                y_test = y_test[kept_indexes]
                y_pred = y_pred_array[kept_indexes]
                t_test = t_test[kept_indexes]
                results = tess_metrics.calculate_metrics(
                    y_test, y_pred, existing=results)
            else:
                y_test = y_test[:0]
                y_pred = y_pred_array[:0]
                t_test = t_test[:0]
                for key in ('tp', 'fp', 'tn', 'fn', 'p', 'n', 'tot'):
                    results.setdefault(key, []).append(0)
                for cumulative, base in (
                    ('tp_cumu', 'tp'), ('fp_cumu', 'fp'), ('tn_cumu', 'tn'),
                    ('fn_cumu', 'fn'), ('p_cumu', 'p'), ('n_cumu', 'n'),
                    ('tot_cumu', 'tot')):
                    results.setdefault(cumulative, []).append(sum(results[base]))
                for key in (
                    'tpr', 'fnr', 'fpr', 'tnr', 'precision', 'recall',
                    'f1', 'precision_n', 'recall_n', 'f1_n'):
                    results.setdefault(key, []).append(float('nan'))
        else:
            results['rejected'].append(0)
            results['f1_r'].append(float('nan'))
            results = tess_metrics.calculate_metrics(
                y_test, y_pred, existing=results)

        if 'y_preds' not in results:
            results['y_tests'] = [y_test]
            results['y_preds'] = [y_pred]
            results['t_tests'] = [t_test]
        else:
            results['y_tests'].append(y_test)
            results['y_preds'].append(y_pred)
            results['t_tests'].append(t_test)

    return results

clf_utils.fit_predict_reject_sample_update = _patched_fprsu
print("✅ Runtime V15: IRAL/IRAAL, DRMD-FN và Balanced Hard Replay nhân quả")


# ══════════════════════════════════════════════════════════════════════════════
# V15 — suy luận theo lô cho actor ba hành động
# ══════════════════════════════════════════════════════════════════════════════
# Gán trực tiếp vào namespace của DRMD.base để lần chạy lại cell trong cùng
# kernel không giữ tham chiếu tới hàm cũ đã import bằng `from ... import ...`.
import DRMD.base as _drmd_base_runtime
_drmd_base_runtime.fit_predict_reject_sample_update = _patched_fprsu

# Nhánh hai hành động giữ nguyên đường suy luận gốc; ma trận V15 chỉ dùng
# actor ba hành động và khóa kích thước lô trong chữ ký triển khai.
_original_drmd_test = drmd_env.DRMD._DRMD__test

def _observation_chunk(model_env, X_chunk, t_chunk):
    dense = X_chunk.toarray() if sp.issparse(X_chunk) else np.asarray(X_chunk)
    obs = torch.from_numpy(np.ascontiguousarray(dense, dtype=np.float32)).to(model_env.device)
    if model_env.temporal_feature:
        time_column = drmd_env.convert_time(t_chunk, model_env.device) - model_env.first_date
        obs = torch.cat((time_column.unsqueeze(1), obs), dim=1)
    return obs

def _batched_drmd_test(self, X, t):
    if not self.is_reject_action:
        return _original_drmd_test(self, X, t)
    if self.validation_episodes > 0:
        self._DRMD__load(opt=self.val_extension)
    self.model.agent.eval()
    actions, logprobs, entropies, values = [], [], [], []
    chunk_rows = max(1, int(BATCH_PREDICT_ROWS))
    for start in range(0, X.shape[0], chunk_rows):
        stop = min(start + chunk_rows, X.shape[0])
        obs = _observation_chunk(self, X[start:stop], t[start:stop])
        action, logprob, entropy, value = self.model.step(obs=obs)
        actions.append(action.detach().cpu())
        logprobs.append(logprob.detach().cpu())
        entropies.append(entropy.detach().cpu())
        values.append(value.detach().cpu().reshape(-1))
        del obs, action, logprob, entropy, value
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    action = torch.cat(actions)
    logprob = torch.cat(logprobs)
    entropy = torch.cat(entropies)
    value = torch.cat(values)
    # Giữ đúng ngữ nghĩa mã gốc: max entropy được lấy trên TOÀN chu kỳ.
    return (action.tolist(), torch.exp(logprob).tolist(),
            (entropy.max() - entropy).tolist(), value.tolist())

def _batched_predict_reject_alt(self, X_test, t_test):
    self.model.agent.eval()
    predictions = []
    chunk_rows = max(1, int(BATCH_PREDICT_ROWS))
    with torch.no_grad():
        for start in range(0, X_test.shape[0], chunk_rows):
            stop = min(start + chunk_rows, X_test.shape[0])
            obs = _observation_chunk(self, X_test[start:stop], t_test[start:stop])
            time_column = drmd_env.convert_time(t_test[start:stop], self.device)
            reject_action = torch.ones_like(time_column) * self.reject_action
            action = self.model.agent.get_next_likely_action(obs=obs, action=reject_action)
            predictions.extend(action.detach().cpu().reshape(-1).tolist())
            del obs, time_column, reject_action, action
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return predictions

drmd_env.DRMD._DRMD__test = _batched_drmd_test
drmd_env.DRMD.predict_reject_alt = _batched_predict_reject_alt


def _batched_predict_binary_all(self, X_test, t_test):
    self.model.agent.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, X_test.shape[0], max(1, int(BATCH_PREDICT_ROWS))):
            stop = min(start + max(1, int(BATCH_PREDICT_ROWS)), X_test.shape[0])
            obs = _observation_chunk(self, X_test[start:stop], t_test[start:stop])
            logits = self.model.agent.actor(obs)
            predictions.extend(torch.argmax(logits[:, :2], dim=1).cpu().tolist())
            del obs, logits
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return predictions

drmd_env.DRMD.predict_binary_all = _batched_predict_binary_all

# ══════════════════════════════════════════════════════════════════════════════
# V15 — ngân sách phản hồi và bộ nhớ Balanced Hard Replay
# ══════════════════════════════════════════════════════════════════════════════
import hashlib as _artifact_hashlib
import torch.nn.functional as _torch_F

FIXED_TEMPORAL_REFERENCE_POLICY = "fixed_initial_training_window_v1"

# DRMD gốc ghép toàn bộ siêu tham số vào ``clf.fpath``. Khi thư mục đích đã có
# chữ ký, tên sidecar ``.p.meta.json`` có thể vượt giới hạn 255 byte của ext4.
# Tên ngắn dưới đây chỉ là khóa lưu trữ; cấu hình đầy đủ vẫn nằm trong sidecar.
ARTIFACT_STEM_PREFIX = "drmd-r"
ARTIFACT_FILENAME_MAX_BYTES = 120

def compact_artifact_stem(legacy_stem):
    digest = _artifact_hashlib.sha256(
        str(legacy_stem).encode("utf-8")).hexdigest()[:20]
    stem = f"{ARTIFACT_STEM_PREFIX}-{digest}"
    if len((stem + ".p.meta.json").encode("utf-8")) > ARTIFACT_FILENAME_MAX_BYTES:
        raise OSError("Tên artifact rút gọn vẫn vượt ngưỡng an toàn.")
    return stem

_init_before_compact_artifact_name = drmd_env.DRMD.__init__

def _init_with_compact_artifact_name(self, settings, *args, **kwargs):
    _init_before_compact_artifact_name(self, settings, *args, **kwargs)
    self.legacy_fpath = self.fpath
    self.fpath = compact_artifact_stem(self.legacy_fpath)

drmd_env.DRMD.__init__ = _init_with_compact_artifact_name
_original_training_data_setup = drmd_env.DRMD._DRMD__training_data_setup
_original_drmd_fit = drmd_env.DRMD.fit

def _training_data_setup_with_common_temporal_axis(self, X_train, y_train, t_train):
    """Khóa chuẩn phần thưởng thời gian theo cửa sổ huấn luyện ban đầu.

    Mã DRMD gốc tính lại ``start_date`` từ mẫu nhỏ tuổi nhất còn trong bộ nhớ.
    Khi thành phần bộ nhớ thay đổi theo phản hồi, cách đó làm thay đổi đồng thời
    dữ liệu và chuẩn hóa phần thưởng. Giao thức V15 cố định gốc và mẫu số theo cửa sổ
    huấn luyện ban đầu, đúng ý đồ trọng số thời gian tuyệt đối tăng tuyến tính
    của DRMD. Quy tắc không đọc tháng kiểm thử tương lai và dùng chung cho mọi
    biến thể.
    """
    # DRMD gốc đọc trực tiếp thuộc tính .year/.month của từng mốc.
    # BHR lưu thời gian chuẩn numpy.datetime64 để kiểm toán và vì vậy
    # phải đổi giao diện sang datetime.datetime trước khi gọi mã gốc.
    # Phép đổi chỉ thay kiểu biểu diễn, không đổi tháng hay thứ tự mẫu.
    _time_array = np.asarray(t_train).reshape(-1)
    if np.issubdtype(_time_array.dtype, np.datetime64):
        t_train = _time_array.astype('datetime64[us]').astype(object)
    elif not all(hasattr(value, 'year') and hasattr(value, 'month')
                 for value in _time_array):
        raise TypeError('t_train không cung cấp thuộc tính year/month.')
    X_fit, y_fit, t_fit = _original_training_data_setup(
        self, X_train, y_train, t_train)
    current_start = int(self.start_date)
    current_end = int(self.end_date)
    current_period = int(self.training_period)
    if current_period <= 0:
        raise ValueError("Khoảng thời gian của bộ nhớ hiện hành không dương.")
    self._current_memory_start = current_start
    self._current_memory_end = current_end
    self._current_memory_period = current_period

    if self._temporal_reference_start is None:
        self._temporal_reference_start = current_start
        self._temporal_reference_end = current_end
        self._temporal_reference_period = current_period
        if int(self.first_date) != current_start:
            raise ValueError("first_date không khớp cửa sổ huấn luyện ban đầu.")
        if current_period != TRAINING_WINDOW:
            raise ValueError(
                f"Chuẩn thời gian ban đầu {current_period}, kỳ vọng {TRAINING_WINDOW} tháng.")

    self.start_date = int(self._temporal_reference_start)
    self.end_date = int(self._temporal_reference_end)
    self.training_period = int(self._temporal_reference_period)
    times = np.asarray(t_fit)
    month_ids = times.astype('datetime64[M]').astype(np.int64)
    years = times.astype('datetime64[Y]').astype(np.int64) + 1970
    months = month_ids - times.astype('datetime64[Y]').astype('datetime64[M]').astype(np.int64) + 1
    absolute_month = years * 12 + months
    factors = (absolute_month - self.start_date + 1) / self.training_period * self.temporal_scaling
    if not np.isfinite(factors).all():
        raise FloatingPointError("Hệ số phần thưởng thời gian không hữu hạn.")
    self._temporal_reference_history.append({
        "fit_index": int(self._fit_calls),
        "reference_start": int(self.start_date),
        "reference_end": int(self.end_date),
        "reference_period": int(self.training_period),
        "memory_start": current_start, "memory_end": current_end,
        "memory_period": current_period,
        "factor_min": float(np.min(factors)),
        "factor_max": float(np.max(factors)),
        "factor_mean": float(np.mean(factors)),
    })
    return X_fit, y_fit, t_fit

drmd_env.DRMD._DRMD__training_data_setup = (
    _training_data_setup_with_common_temporal_axis)


def _patched_drmd_fit(self, X_train, y_train, t_train):
    _original_drmd_fit(self, X_train, y_train, t_train)
    for parameter_name, parameter in self.model.agent.named_parameters():
        if not torch.isfinite(parameter).all():
            raise FloatingPointError(
                f"Tham số không hữu hạn sau fit {self._fit_calls}: {parameter_name}")
    self._fit_calls += 1

drmd_env.DRMD.fit = _patched_drmd_fit

# `references/DRMD/DRMD/base.py` trong workspace có một cache-check cục bộ làm
# khởi tạo DRMD hai lần. Staging của notebook đã xử lý cache; hàm chuẩn dưới đây
# khớp luồng gốc nhưng chỉ tạo classifier đúng một lần.
def _canonical_run(settings, feature_indexes=None):
    X_all, y_all, t_all = _drmd_base_runtime.load_data(settings)
    if feature_indexes is not None:
        X_all = X_all[:, feature_indexes]
        settings = deepcopy(settings)
        settings.fs_size = int(feature_indexes.sum().item())

    splits = _drmd_base_runtime.temporal.time_aware_train_test_split(
        X=X_all, y=y_all, t=t_all,
        train_size=settings.training_window,
        test_size=settings.testing_window,
        granularity=settings.granularity)
    if settings.virtual_months:
        splits = _drmd_base_runtime.convert_virtual_months(
            splits, X_all, y_all, t_all)

    clf = drmd_env.DRMD(settings)
    print(f"\n{clf.fpath}\n")
    if settings.is_reject_action:
        if (settings.is_active and settings.al_rate > 0
                and not settings.select_reject_labeled):
            selector = UncertaintySelector(settings.al_rate, clf_uses_t=True)
            results = _patched_fprsu(
                clf, *splits, reject_label=settings.reject_action_id,
                select_reject_labeled=False, selectors=[selector])
        else:
            results = _patched_fprsu(
                clf, *splits, reject_label=settings.reject_action_id,
                select_reject_labeled=settings.select_reject_labeled,
                posthoc_augmented_AL=settings.posthoc_augmented_AL)
    elif (settings.is_reject and settings.reject_rate > 0
          and settings.is_active and settings.al_rate > 0):
        rejector = UncertaintyRejector(settings.reject_rate, clf_uses_t=True)
        selector = UncertaintySelector(settings.al_rate, clf_uses_t=True)
        results = _patched_fprsu(
            clf, *splits, selectors=[selector], rejectors=[rejector])
    elif settings.is_reject and settings.reject_rate > 0:
        rejector = UncertaintyRejector(settings.reject_rate, clf_uses_t=True)
        results = _patched_fprsu(clf, *splits, rejectors=[rejector])
    elif settings.is_active and settings.al_rate > 0:
        selector = UncertaintySelector(settings.al_rate, clf_uses_t=True)
        results = _patched_fprsu(clf, *splits, selectors=[selector])
    else:
        results = _patched_fprsu(clf, *splits)

    results["selection_history"] = list(clf._selection_history)
    results["temporal_reference_history"] = list(clf._temporal_reference_history)
    results["strategy_runtime_config"] = {
        "feedback_selector_mode": str(clf._feedback_selector_mode),
        "feedback_budget": int(clf._feedback_budget),
        "candidate_multiplier": int(clf._candidate_multiplier),
        "adaptive_fn_penalty": bool(clf.adaptive_fn_penalty),
        "drift_aware_fn_penalty": bool(clf._drift_aware_fn),
        "adaptive_bhr_quota": bool(clf._adaptive_bhr_quota),
        "strategy_seed": int(clf._strategy_seed),
        "audit_sampling": bool(clf.afnp_audit_sampling),
        "balanced_hard_replay": bool(clf._balanced_hard_replay),
        "bhr_fn_fraction": float(clf._bhr_fn_fraction),
        "bhr_fp_fraction": float(clf._bhr_fp_fraction),
        "bhr_fn_max_repeat": int(clf._bhr_fn_max_repeat),
        "bhr_fp_max_repeat": int(clf._bhr_fp_max_repeat),
        "full_feedback": bool(clf._full_feedback),
        "bhr_error_partition_prediction": "binary_argmax_head",
        "fn_reward_scope": "direct_classification_only",
        "reject_counterfactual_reward": "unpenalized_drmd_classification",
        "fn_controller": ("drift_aware_beta_controller"
            if clf._drift_aware_fn else
            ("beta_credible_interval_gated"
             if clf.adaptive_fn_penalty else "disabled")),
        "bhr_fn_coordination": ("adaptive_audit_risk_quota"
            if clf._adaptive_bhr_quota else
            ("base_fraction_div_sqrt_penalty"
             if clf._balanced_hard_replay else "disabled")),
        "fn_controller_config": (
            clf.afnp_controller.configuration()
            if clf.afnp_controller is not None else None),
        "policy_action_space": [0, 1, 2],
        "reject_action_index": int(clf.reject_action),
        "training_history_cap": clf._training_history_cap,
        "temporal_reference_policy": clf._temporal_reference_policy,
        "temporal_reference_start": int(clf._temporal_reference_start),
        "temporal_reference_end": int(clf._temporal_reference_end),
        "temporal_reference_period": int(clf._temporal_reference_period),
        "feedback_timing": "predict_t_then_fit_t_plus_1",
        "single_classifier_initialization": True,
    }
    save_path = Path(settings.results_save_location) / f"{clf.fpath}.p"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "wb") as handle:
        pickle.dump(results, handle)
    try:
        _drmd_base_runtime.metrics.print_metrics(results)
        print("AUT theo F1 (%):", 100.0 * _drmd_base_runtime.metrics.aut(results, "f1"))
    except Exception as exc:
        print("Artifact đã lưu; bỏ qua lỗi hiển thị độ đo:", repr(exc))
    return results

_drmd_base_runtime.run = _canonical_run

# Chữ ký sidecar phải thay đổi khi chính mã monkey-patch trong notebook thay
# đổi, ngay cả khi checkout DRMD/Tesseract và tên giao thức chưa đổi. Không băm
# trực tiếp code object bằng marshal vì co_filename của IPython đổi theo phiên.
import types as _types
import hashlib as _hashlib

def _stable_code_descriptor(code):
    def normalize(value):
        if isinstance(value, _types.CodeType):
            return _stable_code_descriptor(value)
        if isinstance(value, bytes):
            return {"bytes_hex": value.hex()}
        if isinstance(value, tuple):
            return [normalize(item) for item in value]
        if isinstance(value, list):
            return {"list": [normalize(item) for item in value]}
        if isinstance(value, (set, frozenset)):
            # ``repr(frozenset)`` phụ thuộc PYTHONHASHSEED và đã làm chữ ký
            # Schema cũ thay đổi giữa hai phiên Kaggle dù mã không đổi.
            items = [normalize(item) for item in value]
            items.sort(key=lambda item: json.dumps(
                item, ensure_ascii=False, sort_keys=True,
                separators=(",", ":"), default=str))
            return {"set_type": type(value).__name__, "items": items}
        if isinstance(value, dict):
            items = [(normalize(key), normalize(item))
                     for key, item in value.items()]
            items.sort(key=lambda pair: json.dumps(
                pair[0], ensure_ascii=False, sort_keys=True,
                separators=(",", ":"), default=str))
            return {"dict_items": items}
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return {"type": type(value).__name__, "repr": repr(value)}

    return {
        "argcount": code.co_argcount,
        "posonlyargcount": code.co_posonlyargcount,
        "kwonlyargcount": code.co_kwonlyargcount,
        "code_hex": code.co_code.hex(),
        "consts": normalize(code.co_consts),
        "names": list(code.co_names),
        "varnames": list(code.co_varnames),
        "freevars": list(code.co_freevars),
        "cellvars": list(code.co_cellvars),
    }

_runtime_patch_components = (
    _patched_init, _classification_reward, _patched_reward_function, _patched_fprsu,
    _deterministic_recency_cap, _apply_training_memory,
    _observation_chunk, _batched_drmd_test, _batched_predict_reject_alt,
    _actor_uncertainty, _select_feedback_indexes, _patched_drmd_fit,
    _training_data_setup_with_common_temporal_axis,
    _batched_predict_binary_all, _canonical_run,
)
PATCH_IMPLEMENTATION_FINGERPRINT = _hashlib.sha256(
    json.dumps(
        [_stable_code_descriptor(function.__code__)
         for function in _runtime_patch_components],
        ensure_ascii=False, sort_keys=True, separators=(",", ":"),
    ).encode()
).hexdigest()
print("✅ Runtime patch V15: 5 nhánh cũ và 2 nhánh cải tiến")
print("Runtime patch fingerprint:", PATCH_IMPLEMENTATION_FINGERPRINT)

## Ô 7 — Nạp đúng bộ dữ liệu LAMDA đã khóa

Notebook chỉ chấp nhận bộ dữ liệu LAMDA khớp định danh kho, revision, lược đồ và dấu vân tay nguồn. Không quét rồi ghép tùy ý mọi tệp Parquet trong thư mục đầu vào Kaggle.


In [ ]:
import hashlib
import pyarrow.parquet as pq

EXPECTED_LAMDA_RELATIVE_FILES = {
    f"{year}/{year}_{split}.parquet"
    for year in DATASET_YEARS for split in ("train", "test")
}

def baseline_candidates(search_roots):
    candidates = set()
    for root in search_roots:
        if not root.exists():
            continue
        if root.is_dir() and root.name.lower() == "baseline":
            candidates.add(root.resolve())
        for pattern in ("Baseline", "baseline"):
            candidates.update(path.resolve() for path in root.rglob(pattern) if path.is_dir())
    return sorted(candidates, key=str)

def matching_lamda_files(root):
    files = []
    for path in root.rglob("*.parquet"):
        try:
            relative = path.relative_to(root).as_posix()
        except ValueError:
            continue
        if relative in EXPECTED_LAMDA_RELATIVE_FILES:
            files.append(path)
    return sorted(files, key=lambda path: path.relative_to(root).as_posix())

search_roots = [INPUT_ROOT, WORK / "LAMDA_FULL"]
if LAMDA_ROOT_OVERRIDE is not None:
    valid_roots = [Path(LAMDA_ROOT_OVERRIDE)]
else:
    valid_roots = [
        root for root in baseline_candidates(search_roots)
        if {path.relative_to(root).as_posix() for path in matching_lamda_files(root)}
           == EXPECTED_LAMDA_RELATIVE_FILES
    ]

if len(valid_roots) != 1:
    details = {
        str(root): len(matching_lamda_files(root))
        for root in baseline_candidates(search_roots)
    }
    raise RuntimeError(
        "Phải xác định đúng một nguồn LAMDA/Baseline có đủ "
        f"{len(EXPECTED_LAMDA_RELATIVE_FILES)} tệp. "
        f"Tìm thấy {len(valid_roots)} nguồn hợp lệ: {details}. "
        "Đặt LAMDA_ROOT_OVERRIDE tới thư mục Baseline cần dùng."
    )

LAMDA_BASELINE_ROOT = valid_roots[0]
lamda_parquets = matching_lamda_files(LAMDA_BASELINE_ROOT)
assert len(lamda_parquets) == 2 * len(DATASET_YEARS)

def source_file_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

source_files = []
for path in lamda_parquets:
    parquet_file = pq.ParquetFile(path)
    source_files.append({
        "relative_path": path.relative_to(LAMDA_BASELINE_ROOT).as_posix(),
        "bytes": int(path.stat().st_size),
        "sha256": source_file_sha256(path),
        "rows": int(parquet_file.metadata.num_rows),
        "columns": list(parquet_file.schema_arrow.names),
        "schema": [
            {"name": field.name, "type": str(field.type)}
            for field in parquet_file.schema_arrow
        ],
    })
observed_lamda_file_sha256 = {
    item["relative_path"]: item["sha256"] for item in source_files
}
if observed_lamda_file_sha256 != EXPECTED_LAMDA_FILE_SHA256:
    all_paths = sorted(
        set(observed_lamda_file_sha256) | set(EXPECTED_LAMDA_FILE_SHA256))
    mismatches = {
        relative_path: {
            "observed": observed_lamda_file_sha256.get(relative_path),
            "expected": EXPECTED_LAMDA_FILE_SHA256.get(relative_path),
        }
        for relative_path in all_paths
        if observed_lamda_file_sha256.get(relative_path)
        != EXPECTED_LAMDA_FILE_SHA256.get(relative_path)
    }
    raise ValueError(
        "LAMDA Baseline không khớp revision đã khóa "
        f"{LAMDA_HF_REVISION}: {json.dumps(mismatches, ensure_ascii=False)}")

source_descriptor = {
    "hf_repo_id": LAMDA_HF_REPO_ID,
    "hf_revision": LAMDA_HF_REVISION,
    "files": source_files,
}
fingerprint_payload = json.dumps(
    source_descriptor, ensure_ascii=False, sort_keys=True).encode()
DATASET_SOURCE_FINGERPRINT = hashlib.sha256(fingerprint_payload).hexdigest()
(TABLE_OUT / "lamda_source_manifest.json").write_text(
    json.dumps({"root": str(LAMDA_BASELINE_ROOT),
                "hf_repo_id": LAMDA_HF_REPO_ID,
                "hf_revision": LAMDA_HF_REVISION,
                "revision_verified_by_sha256": True,
                "source_fingerprint_sha256": DATASET_SOURCE_FINGERPRINT,
                "files": source_files}, indent=2, ensure_ascii=False),
    encoding="utf-8")

print("LAMDA Baseline root:", LAMDA_BASELINE_ROOT)
print("Selected parquet files:", len(lamda_parquets))
print("Source fingerprint:", DATASET_SOURCE_FINGERPRINT)
for path in lamda_parquets:
    print(" ", path.relative_to(LAMDA_BASELINE_ROOT))

In [ ]:
META_COLS = {
    "hash", "sha256", "label", "family", "vt_count",
    "year_month", "year", "month"
}

def parse_year_month(value):
    s = str(value)
    if len(s) >= 7:
        return datetime.strptime(s[:7] + "-01", "%Y-%m-%d")
    if len(s) == 4:
        return datetime.strptime(s + "-01-01", "%Y-%m-%d")
    return datetime.fromisoformat(s)

def label_to_int(v):
    if isinstance(v, (int, np.integer, float, np.floating)):
        value = int(v)
        if float(v) == value and value in (0, 1):
            return value
        raise ValueError(f"Unsupported numeric label: {v!r}")
    s = str(v).strip().lower()
    if s in {"1", "true", "malware", "malicious"}:
        return 1
    if s in {"0", "false", "benign", "goodware"}:
        return 0
    raise ValueError(f"Unsupported string label: {v!r}")

def feature_columns_from_schema(names):
    feat_cols = [c for c in names if c.lower().startswith("feat_")]
    if feat_cols:
        return feat_cols
    return [c for c in names if c.lower() not in META_COLS]

def load_all_rows_metadata(parquet_files):
    meta_rows = []
    feature_cols_ref = None

    for path in sorted(parquet_files):
        names = list(pq.ParquetFile(path).schema_arrow.names)
        lower_map = {c.lower(): c for c in names}

        label_col = lower_map.get("label")
        ym_col = lower_map.get("year_month")
        assert label_col is not None, f"Khong co cot label trong {path}"
        assert ym_col is not None, f"Khong co cot year_month trong {path}"

        feature_cols = feature_columns_from_schema(names)
        if feature_cols_ref is None:
            feature_cols_ref = feature_cols
        else:
            assert set(feature_cols) == set(feature_cols_ref), f"Feature columns khong dong nhat: {path}"

        id_col = lower_map.get("sha256") or lower_map.get("hash")
        metadata_columns = [label_col, ym_col] + ([id_col] if id_col else [])
        table = pq.read_table(path, columns=metadata_columns)
        labels = [label_to_int(x.as_py()) for x in table[label_col]]
        times = [parse_year_month(x.as_py()) for x in table[ym_col]]
        sample_ids = ([str(x.as_py()) for x in table[id_col]] if id_col
                      else [f"{path.name}:{idx}" for idx in range(len(labels))])

        for idx, (yv, tv, sample_id) in enumerate(zip(labels, times, sample_ids)):
            meta_rows.append({
                "path": path,
                "row_index": idx,
                "label": yv,
                "time": tv,
                "month": f"{tv.year:04d}-{tv.month:02d}",
                "sample_id": sample_id,
            })

    return meta_rows, feature_cols_ref

meta_rows, feature_cols = load_all_rows_metadata(lamda_parquets)
sample_ids = [row["sample_id"] for row in meta_rows]
duplicate_sample_ids = len(sample_ids) - len(set(sample_ids))
if duplicate_sample_ids:
    raise ValueError(f"LAMDA có {duplicate_sample_ids} mã mẫu trùng; dừng trước khi chia dữ liệu.")

print("Metadata rows:", len(meta_rows))
print("Features:", len(feature_cols))
print("Months:", sorted(set(r["month"] for r in meta_rows)))

In [ ]:
def build_Xyt_from_rows(rows, feature_cols):
    rows_by_path = defaultdict(list)
    for r in rows:
        rows_by_path[r["path"]].append(r)

    X_parts, y_parts, t_parts = [], [], []

    for path, rows in sorted(rows_by_path.items(), key=lambda kv: str(kv[0])):
        row_indices = [r["row_index"] for r in rows]
        table = pq.read_table(path, columns=feature_cols)
        X_table = (table.select(feature_cols)
                   if row_indices == list(range(table.num_rows))
                   else table.select(feature_cols).take(row_indices))
        # Không tạo một ma trận dense n_rows × 4.561. Chuyển từng khối
        # cột sang CSR rồi ghép ngang, giữ đỉnh bộ nhớ ổn định trên Kaggle.
        feature_block_columns = 256
        sparse_blocks = []
        for start in range(0, len(feature_cols), feature_block_columns):
            block_columns = feature_cols[start:start + feature_block_columns]
            dense_block = np.column_stack([
                X_table[column].combine_chunks().to_numpy(zero_copy_only=False)
                for column in block_columns
            ]).astype(np.float32, copy=False)
            sparse_blocks.append(sp.csr_matrix(dense_block))
            del dense_block
        X_parts.append(sp.hstack(sparse_blocks, format="csr"))
        del sparse_blocks, X_table, table
        y_parts.append(np.asarray([r["label"] for r in rows], dtype=np.int64))
        t_parts.append(np.asarray([r["time"] for r in rows], dtype=object))

    X = sp.vstack(X_parts).tocsr()
    y = np.concatenate(y_parts)
    t = np.concatenate(t_parts)

    # Giữ thứ tự nguồn/row_index trong cùng tháng. Quicksort mặc định
    # không ổn định và có thể làm thay đổi mẫu được chọn khi xác suất hòa.
    order = np.argsort(t, kind="stable")
    return X[order], y[order], t[order]

X, y, t = build_Xyt_from_rows(meta_rows, feature_cols)

DATASET_ARTIFACT_TAG = "LAMDA" + "_".join(str(year) for year in DATASET_YEARS)
pickle.dump(X, open(DATA_OUT / f"{DATASET_ARTIFACT_TAG}-X.p", "wb"))
pickle.dump(y, open(DATA_OUT / f"{DATASET_ARTIFACT_TAG}-y.p", "wb"))
pickle.dump(t, open(DATA_OUT / f"{DATASET_ARTIFACT_TAG}-t.p", "wb"))
pickle.dump(feature_cols, open(DATA_OUT / f"{DATASET_ARTIFACT_TAG}-feature-cols.p", "wb"))

print("X:", X.shape)
print("y:", y.shape, "malware_rate:", round(float(y.mean()), 4))
print("t:", min(t), "->", max(t))
print("Saved dataset tag:", DATASET_ARTIFACT_TAG)

In [ ]:
def month_key(dt):
    return f"{dt.year:04d}-{dt.month:02d}"

def validate_dataset(X, y, t):
    months = sorted(set(month_key(x) for x in t))
    rows = []
    for m in months:
        idx = np.array([i for i, tv in enumerate(t) if month_key(tv) == m])
        yy = y[idx]
        rows.append({
            "month": m,
            "total": int(len(idx)),
            "malware": int(yy.sum()),
            "benign": int((yy == 0).sum()),
            "malware_rate": float(yy.mean()) if len(yy) else 0,
        })

    zero_rows = int((X.getnnz(axis=1) == 0).sum()) if sp.issparse(X) else int((X.sum(axis=1) == 0).sum())
    sparsity = 1.0 - (X.nnz / (X.shape[0] * X.shape[1])) if sp.issparse(X) else float(np.mean(X == 0))
    stored_values = X.data if sp.issparse(X) else np.asarray(X).reshape(-1)
    nonfinite_values = int((~np.isfinite(stored_values)).sum())
    if nonfinite_values:
        raise ValueError(f"Ma trận đặc trưng có {nonfinite_values} giá trị NaN/Inf.")
    if set(np.unique(y)) - {0, 1}:
        raise ValueError(f"Nhãn ngoài miền nhị phân: {np.unique(y)}")
    if X.shape[0] != len(y) or len(y) != len(t):
        raise ValueError("X, y và t không có cùng số mẫu.")
    if X.shape[1] != len(feature_cols):
        raise ValueError("Số cột X không khớp danh sách đặc trưng.")
    if any(t[i] > t[i + 1] for i in range(len(t) - 1)):
        raise ValueError("Dữ liệu chưa được sắp tăng dần theo thời gian.")

    report = {
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "n_months": int(len(months)),
        "start_month": months[0],
        "end_month": months[-1],
        "malware_rate": float(y.mean()),
        "zero_rows": zero_rows,
        "sparsity": float(sparsity),
        "nonfinite_stored_values": nonfinite_values,
        "labels": sorted(int(value) for value in np.unique(y)),
    }

    return report, rows

validation_report, month_rows = validate_dataset(X, y, t)

expected_dataset_values = {
    "n_samples": EXPECTED_LAMDA_SAMPLES,
    "n_features": EXPECTED_LAMDA_FEATURES,
}
dataset_mismatches = {
    key: {"observed": validation_report.get(key), "expected": expected}
    for key, expected in expected_dataset_values.items()
    if int(validation_report.get(key, -1)) != int(expected)
}
if dataset_mismatches:
    raise ValueError(
        "LAMDA không khớp revision đã khóa: "
        + json.dumps(dataset_mismatches, ensure_ascii=False))

with open(TABLE_OUT / "dataset_validation_report.json", "w") as f:
    json.dump(validation_report, f, indent=2)

with open(TABLE_OUT / "month_distribution.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=month_rows[0].keys())
    writer.writeheader()
    writer.writerows(month_rows)

print(validation_report)
for r in month_rows:
    print(r)

In [ ]:
from tesseract import temporal

# ================= MONKEY PATCH: SỬA HÀM CHIA DỮ LIỆU TRỰC TIẾP TRONG RAM =================
if not hasattr(temporal, "original_time_aware_train_test_split"):
    temporal.original_time_aware_train_test_split = temporal.time_aware_train_test_split

def custom_time_aware_train_test_split(*args, **kwargs):
    # Gọi hàm chia dữ liệu gốc
    splits = temporal.original_time_aware_train_test_split(*args, **kwargs)
    X_train, X_tests, y_train, y_tests, t_train, t_tests = splits
    
    # Lọc bỏ hoàn toàn các period test bị rỗng (ví dụ: các tháng trống của năm 2015)
    non_empty_idx = [i for i, xt in enumerate(X_tests) if xt.shape[0] > 0]
    X_tests = [X_tests[i] for i in non_empty_idx]
    y_tests = [y_tests[i] for i in non_empty_idx]
    t_tests = [t_tests[i] for i in non_empty_idx]
    
    return X_train, X_tests, y_train, y_tests, t_train, t_tests

# Ghi đè hàm gốc trong bộ nhớ
temporal.time_aware_train_test_split = custom_time_aware_train_test_split
print("Đã vá (monkey-patched) hàm chia dữ liệu thành công!")
# =========================================================================================

# Thực hiện chia dữ liệu (hàm mới sẽ tự động lọc các period trống của năm 2015)
splits = temporal.time_aware_train_test_split(
    X=X,
    y=y,
    t=t,
    train_size=TRAINING_WINDOW,
    test_size=TESTING_WINDOW,
    granularity=GRANULARITY,
)

X_train, X_tests, y_train, y_tests, t_train, t_tests = splits

split_rows = []
split_rows.append({
    "period": "train",
    "n": int(X_train.shape[0]),
    "start": str(min(t_train)),
    "end": str(max(t_train)),
    "malware_rate": float(y_train.mean()),
})

for i, (xt, yt, tt) in enumerate(zip(X_tests, y_tests, t_tests), start=1):
    split_rows.append({
        "period": f"test_{i}",
        "n": int(xt.shape[0]),
        "start": str(min(tt)) if len(tt) else "",
        "end": str(max(tt)) if len(tt) else "",
        "malware_rate": float(yt.mean()) if len(yt) else 0,
    })

with open(TABLE_OUT / "time_split_check.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=split_rows[0].keys())
    writer.writeheader()
    writer.writerows(split_rows)

for r in split_rows:
    print(r)


# ══════════════════════════════════════════════════════════════════════════════
# V15 — kiểm chứng cửa sổ huấn luyện theo tháng lịch
# ══════════════════════════════════════════════════════════════════════════════
from collections import Counter as _Counter

if set(np.unique(y_train)) != {0, 1}:
    raise ValueError(f"Tập huấn luyện ban đầu không có đủ hai lớp: {np.unique(y_train)}")
if not X_tests or any(len(test_labels) == 0 for test_labels in y_tests):
    raise ValueError("Sau lọc vẫn còn chu kỳ kiểm thử rỗng.")
first_test_time = min(min(period_times) for period_times in t_tests)
if max(t_train) >= first_test_time:
    raise ValueError(
        f"Vi phạm thứ tự thời gian: max(train)={max(t_train)} >= "
        f"min(test)={first_test_time}")

def kiem_chung_cua_so_huan_luyen(t_train, t_tests) -> dict:
    """In từng tháng lịch trong cửa sổ huấn luyện và chỉ rõ tháng nào khuyết.

    Tesseract cắt theo BIÊN LỊCH nên cửa sổ `TRAINING_WINDOW` tháng có thể chứa ít
    tháng thực có dữ liệu hơn. Báo cáo phải phát biểu đúng con số này.
    """
    def mkey(d): return f"{d.year:04d}-{d.month:02d}"
    dem = _Counter(mkey(d) for d in t_train)
    d0 = min(t_train); m0 = d0.year * 12 + d0.month
    lich = []
    for k in range(TRAINING_WINDOW):
        m = m0 + k
        lich.append(f"{(m - 1) // 12:04d}-{((m - 1) % 12) + 1:02d}")
    co = [k for k in lich if dem.get(k, 0) > 0]
    khuyet = [k for k in lich if dem.get(k, 0) == 0]

    print(f"Cửa sổ huấn luyện trải {TRAINING_WINDOW} tháng lịch: {lich[0]} … {lich[-1]}")
    print(f"  Tháng CÓ dữ liệu : {len(co)}")
    print(f"  Tháng KHUYẾT     : {len(khuyet)}  {khuyet if khuyet else ''}")
    print(f"  Tổng số mẫu      : {len(t_train):,}".replace(",", "."))
    for k in lich:
        n = dem.get(k, 0)
        print(f"    {k}  {('khuyết' if n == 0 else format(n, ',').replace(',', '.')):>10}")
    if khuyet:
        print(f"\n  ⚠️  Phát biểu đúng trong báo cáo: 'cửa sổ huấn luyện trải "
              f"{TRAINING_WINDOW} tháng lịch, chứa dữ liệu của {len(co)} tháng'.")
        print("      KHÔNG viết 'huấn luyện trên 12 tháng dữ liệu'.")
        print("      Ràng buộc C1 vẫn thoả mãn: mọi mẫu huấn luyện đều trước mọi mẫu kiểm thử.")
    return {"training_window_calendar_months": TRAINING_WINDOW,
            "months_with_data": len(co), "months_missing": khuyet,
            "n_train_samples": int(len(t_train)), "n_test_periods": len(t_tests)}

TRAIN_WINDOW_AUDIT = kiem_chung_cua_so_huan_luyen(t_train, t_tests)
(TABLE_OUT / "training_window_audit.json").write_text(
    json.dumps(TRAIN_WINDOW_AUDIT, indent=2, ensure_ascii=False), encoding="utf-8")

EXPECTED_PERIODS = len(X_tests)
OBSERVED_TEST_SAMPLES = int(sum(int(labels.size) for labels in y_tests))
if EXPECTED_PERIODS != EXPECTED_TEST_PERIODS:
    raise ValueError(
        f"Số chu kỳ kiểm thử {EXPECTED_PERIODS}, kỳ vọng {EXPECTED_TEST_PERIODS}.")
if OBSERVED_TEST_SAMPLES != EXPECTED_TEST_SAMPLES:
    raise ValueError(
        f"Số mẫu kiểm thử {OBSERVED_TEST_SAMPLES}, kỳ vọng {EXPECTED_TEST_SAMPLES}.")
if int(y_train.size) != EXPECTED_INITIAL_TRAIN_SAMPLES:
    raise ValueError(
        f"Số mẫu huấn luyện ban đầu {y_train.size}, "
        f"kỳ vọng {EXPECTED_INITIAL_TRAIN_SAMPLES}.")

observed_test_months = []
for period, (features, labels, times) in enumerate(
        zip(X_tests, y_tests, t_tests), start=1):
    if not (features.shape[0] == labels.size == times.size):
        raise ValueError(f"Chu kỳ {period}: X/y/t lệch số hàng.")
    months = np.asarray(times).astype("datetime64[M]")
    unique_months = np.unique(months)
    if unique_months.size != 1:
        raise ValueError(f"Chu kỳ {period} không nằm trong đúng một tháng lịch.")
    observed_test_months.append(unique_months[0])
if len(np.unique(observed_test_months)) != EXPECTED_PERIODS:
    raise ValueError("Các chu kỳ kiểm thử có tháng lịch bị trùng.")
if any(left >= right for left, right in zip(
        observed_test_months[:-1], observed_test_months[1:])):
    raise ValueError("Thứ tự tháng kiểm thử không tăng nghiêm ngặt.")

_data_pipeline_components = (
    baseline_candidates, matching_lamda_files, source_file_sha256,
    parse_year_month, label_to_int, feature_columns_from_schema,
    load_all_rows_metadata, build_Xyt_from_rows, validate_dataset,
    custom_time_aware_train_test_split, kiem_chung_cua_so_huan_luyen,
)
DATA_PIPELINE_IMPLEMENTATION_FINGERPRINT = hashlib.sha256(
    json.dumps(
        [_stable_code_descriptor(function.__code__)
         for function in _data_pipeline_components],
        ensure_ascii=False, sort_keys=True, separators=(",", ":"),
    ).encode()
).hexdigest()

split_fingerprint_payload = {
    "protocol_version": PROTOCOL_VERSION,
    "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
    "training_window_mode": TRAINING_WINDOW_MODE,
    "training_window": TRAINING_WINDOW,
    "testing_window": TESTING_WINDOW,
    "granularity": GRANULARITY,
    "feature_dim": int(X.shape[1]),
    "expected_test_periods": EXPECTED_TEST_PERIODS,
    "expected_test_samples": EXPECTED_TEST_SAMPLES,
    "observed_test_samples": OBSERVED_TEST_SAMPLES,
    "data_pipeline_implementation_fingerprint": (
        DATA_PIPELINE_IMPLEMENTATION_FINGERPRINT),
    "train": split_rows[0],
    "tests": split_rows[1:],
}
SPLIT_FINGERPRINT = hashlib.sha256(
    json.dumps(split_fingerprint_payload, ensure_ascii=False, sort_keys=True).encode()
).hexdigest()
(TABLE_OUT / "split_fingerprint.json").write_text(
    json.dumps({**split_fingerprint_payload,
                "split_fingerprint_sha256": SPLIT_FINGERPRINT},
               indent=2, ensure_ascii=False),
    encoding="utf-8")
print(f"\nEXPECTED_PERIODS = {EXPECTED_PERIODS}  (dùng cho ô kiểm tra toàn vẹn)")
print("SPLIT_FINGERPRINT =", SPLIT_FINGERPRINT)

## Ô 8 — Static-MLP đồng bộ, chỉ chạy trong Pha 1


In [ ]:
# MLP tĩnh đồng bộ trên cùng 10 seed và cùng split V15.
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix
import csv
import hashlib
import json
import os
import pickle
import time
import uuid
import zipfile

STATIC_METHOD = "Static-MLP"
STATIC_PROTOCOL = "static_initial_window_no_update"
STATIC_CACHE_DIR = RAW_OUT / "static_baselines"
STATIC_CACHE_DIR.mkdir(parents=True, exist_ok=True)
STATIC_MODEL_SPEC = {
    "scaler": "MaxAbsScaler",
    "hidden_layer_sizes": [512, 256, 128], "activation": "relu",
    "solver": "adam", "alpha": 1e-4,
    "learning_rate_init": 1e-3, "max_iter": 20,
    "shuffle": True, "early_stopping": True,
    "validation_fraction": 0.1, "n_iter_no_change": 10,
}
STATIC_ENV_SPEC = {
    key: ENVIRONMENT_STAMP[key]
    for key in ("python", "numpy", "scipy", "scikit_learn")
}

def _static_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _static_path(seed):
    return STATIC_CACHE_DIR / f"Static-MLP-static-Seed{int(seed)}.p"

def _static_sidecar(path):
    return Path(str(path) + ".meta.json")

def _static_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    if not np.isin(y_pred, [0, 1]).all():
        raise ValueError("Static-MLP sinh dự đoán ngoài miền 0/1.")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "p": int(tp + fn), "n": int(tn + fp), "tot": int(y_true.size),
        "precision": (float(tp / (tp + fp)) if tp + fp else float("nan")),
        "recall": (float(tp / (tp + fn)) if tp + fn else float("nan")),
        "f1": (float(2 * tp / (2 * tp + fp + fn))
               if 2 * tp + fp + fn else float("nan")),
        "fnr": (float(fn / (fn + tp)) if fn + tp else float("nan")),
        "fpr": (float(fp / (fp + tn)) if fp + tn else float("nan")),
    }

def _make_locked_mlp(seed):
    """Tạo lại đúng kiến trúc MLP tĩnh đã khóa."""
    return make_pipeline(
        MaxAbsScaler(copy=True),
        MLPClassifier(
            hidden_layer_sizes=(512, 256, 128), activation="relu",
            solver="adam", alpha=1e-4, learning_rate_init=1e-3,
            max_iter=20, shuffle=True, early_stopping=True,
            validation_fraction=0.1, n_iter_no_change=10,
            random_state=int(seed)))

def _evaluate_static_mlp(seed):
    model = _make_locked_mlp(seed)
    model.fit(X_train, y_train)
    rows = []
    period_predictions = []
    for period, (X_test, y_test, t_test) in enumerate(
            zip(X_tests, y_tests, t_tests), start=1):
        prediction = np.asarray(model.predict(X_test), dtype=np.int8)
        period_predictions.append(prediction)
        rows.append({
            "method": STATIC_METHOD, "seed": int(seed), "period": int(period),
            "period_start": str(np.min(t_test)), "period_end": str(np.max(t_test)),
            "protocol": STATIC_PROTOCOL,
            "true_malware_rate": float(np.mean(y_test)),
            **_static_metrics(y_test, prediction),
        })
    return rows, period_predictions

def _validate_static_artifact(path, seed, require_sidecar=False):
    with open(path, "rb") as handle:
        artifact = pickle.load(handle)
    expected = {
        "method": STATIC_METHOD, "seed": int(seed),
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "model_spec": STATIC_MODEL_SPEC, "environment_spec": STATIC_ENV_SPEC,
    }
    for key, value in expected.items():
        if artifact.get(key) != value:
            raise ValueError(f"Static cache sai {key}.")
    if int(artifact.get("schema_version", -1)) != 6:
        raise ValueError("Static cache không thuộc schema 6 có dự đoán gốc.")
    if artifact.get("protocol") != STATIC_PROTOCOL:
        raise ValueError("Static cache sai protocol.")
    if artifact.get("protocol_version") != PROTOCOL_VERSION:
        raise ValueError("Static cache thuộc protocol_version khác.")
    rows = artifact.get("period_rows", [])
    period_predictions = artifact.get("period_predictions", [])
    if (len(rows) != EXPECTED_PERIODS
            or len(period_predictions) != EXPECTED_PERIODS):
        raise ValueError("Static cache sai số chu kỳ hoặc thiếu dự đoán gốc.")
    observed_test_samples = 0
    for index, row in enumerate(rows):
        canonical_y = np.asarray(y_tests[index], dtype=int)
        canonical_t = np.asarray(t_tests[index]).astype("datetime64[ns]")
        canonical_total = int(X_tests[index].shape[0])
        prediction = np.asarray(
            period_predictions[index], dtype=np.int64).reshape(-1)
        if (prediction.size != canonical_total
                or not np.isin(prediction, [0, 1]).all()):
            raise ValueError(
                f"Static cache sai dự đoán gốc ở chu kỳ {index + 1}.")
        observed_test_samples += canonical_total
        required = {"tp", "tn", "fp", "fn", "p", "n", "tot",
                    "f1", "fnr", "fpr"}
        if not required.issubset(row):
            raise ValueError(f"Static cache thiếu cột ở chu kỳ {index + 1}.")
        recomputed = _static_metrics(canonical_y, prediction)
        counts = {key: int(row[key]) for key in ("tp", "tn", "fp", "fn")}
        recomputed_counts = {
            key: int(recomputed[key]) for key in ("tp", "tn", "fp", "fn")
        }
        if min(counts.values()) < 0:
            raise ValueError(f"Static cache có số đếm âm ở chu kỳ {index + 1}.")
        if counts != recomputed_counts:
            raise ValueError(
                f"Static cache sai ma trận nhầm lẫn ở chu kỳ {index + 1}.")
        expected_p = int(np.sum(canonical_y == 1))
        expected_n = int(np.sum(canonical_y == 0))
        period_start = np.datetime64(row.get("period_start"), "ns")
        period_end = np.datetime64(row.get("period_end"), "ns")
        if (row.get("method") != STATIC_METHOD
                or int(row.get("seed", -1)) != int(seed)
                or int(row.get("period", -1)) != index + 1
                or int(row.get("tot", -1)) != canonical_total
                or int(row.get("p", -1)) != expected_p
                or int(row.get("n", -1)) != expected_n
                or counts["tp"] + counts["fn"] != expected_p
                or counts["tn"] + counts["fp"] != expected_n
                or period_start != canonical_t.min()
                or period_end != canonical_t.max()):
            raise ValueError(f"Static cache sai hàng chu kỳ {index + 1}.")
        expected_metrics = {
            "f1": (2 * counts["tp"] /
                   (2 * counts["tp"] + counts["fp"] + counts["fn"])
                   if 2 * counts["tp"] + counts["fp"] + counts["fn"]
                   else float("nan")),
            "fnr": (counts["fn"] / expected_p if expected_p else float("nan")),
            "fpr": (counts["fp"] / expected_n if expected_n else float("nan")),
        }
        for metric, expected_value in expected_metrics.items():
            observed_value = float(row[metric])
            if not ((np.isnan(observed_value) and np.isnan(expected_value))
                    or np.isclose(observed_value, expected_value,
                                  rtol=0.0, atol=1e-12)):
                raise ValueError(
                    f"Static cache sai {metric} ở chu kỳ {index + 1}.")
    if observed_test_samples != EXPECTED_TEST_SAMPLES:
        raise ValueError("Static cache sai tổng số mẫu kiểm thử.")
    if (artifact.get("n_test_samples") is not None
            and int(artifact["n_test_samples"]) != EXPECTED_TEST_SAMPLES):
        raise ValueError("Static cache khai báo sai n_test_samples.")
    metadata_path = _static_sidecar(path)
    if require_sidecar and not metadata_path.exists():
        raise FileNotFoundError("Static cache V15 thiếu sidecar.")
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if (metadata.get("status") != "complete"
                or int(metadata.get("schema_version", -1)) != 6
                or metadata.get("protocol_version") != PROTOCOL_VERSION
                or metadata.get("method") != STATIC_METHOD
                or int(metadata.get("seed", -1)) != int(seed)
                or metadata.get("dataset_source_fingerprint")
                != DATASET_SOURCE_FINGERPRINT
                or metadata.get("split_fingerprint") != SPLIT_FINGERPRINT
                or metadata.get("artifact_sha256") != _static_sha256(path)):
            raise ValueError("Static sidecar không khớp artifact.")
    return artifact

def _save_static_artifact(seed, rows, period_predictions, imported_from=None):
    final_path = _static_path(seed)
    temporary = final_path.with_name(final_path.name + f".{uuid.uuid4().hex}.tmp")
    artifact = {
        "schema_version": 6, "protocol_version": PROTOCOL_VERSION,
        "method": STATIC_METHOD, "seed": int(seed), "protocol": STATIC_PROTOCOL,
        "n_periods": len(rows),
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "model_spec": STATIC_MODEL_SPEC, "environment_spec": STATIC_ENV_SPEC,
        "n_test_samples": int(sum(int(row["tot"]) for row in rows)),
        "imported_from": imported_from, "period_rows": rows,
        "period_predictions": [
            np.asarray(values, dtype=np.int8) for values in period_predictions
        ],
    }
    with open(temporary, "wb") as handle:
        pickle.dump(artifact, handle)
        handle.flush(); os.fsync(handle.fileno())
    os.replace(temporary, final_path)
    metadata = {
        "status": "complete", "schema_version": 6,
        "protocol_version": PROTOCOL_VERSION,
        "method": STATIC_METHOD, "seed": int(seed),
        "artifact_sha256": _static_sha256(final_path),
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "imported_from": imported_from,
    }
    metadata_path = _static_sidecar(final_path)
    metadata_tmp = metadata_path.with_name(metadata_path.name + f".{uuid.uuid4().hex}.tmp")
    metadata_tmp.write_text(json.dumps(metadata, indent=2, ensure_ascii=False),
                            encoding="utf-8")
    os.replace(metadata_tmp, metadata_path)
    return final_path

def _static_candidates(seed):
    filename = _static_path(seed).name
    candidates = [_static_path(seed)]
    if INPUT_ROOT.exists():
        candidates.extend(sorted(INPUT_ROOT.rglob(filename)))
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique

def _write_static_csv(path, rows):
    if not rows:
        return
    keys = []
    for row in rows:
        for key in row:
            if key not in keys:
                keys.append(key)
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader(); writer.writerows(rows)

STATIC_RESULTS = {}
STATIC_CACHE_AUDIT = []
STATIC_RUNS_STARTED_THIS_SESSION = 0
STATIC_SESSION_STOP_REASON = "completed_static_plan"
for seed in SEEDS:
    artifact = None
    accepted_path = None
    for candidate in _static_candidates(seed):
        if not candidate.exists():
            continue
        try:
            artifact = _validate_static_artifact(
                candidate, seed, require_sidecar=(candidate == _static_path(seed)))
            accepted_path = candidate
            break
        except Exception as exc:
            STATIC_CACHE_AUDIT.append({
                "seed": int(seed), "candidate": str(candidate), "state": "rejected",
                "detail": f"{type(exc).__name__}: {exc}"})
    if artifact is not None:
        rows = artifact["period_rows"]
        period_predictions = artifact["period_predictions"]
        if (accepted_path != _static_path(seed)
                or artifact.get("protocol_version") != PROTOCOL_VERSION
                or int(artifact.get("schema_version", 0)) != 6):
            accepted_path = _save_static_artifact(
                seed, rows, period_predictions,
                imported_from=str(accepted_path))
            state = "validated_import"
        else:
            state = "validated_v15_cache"
        STATIC_RESULTS[(STATIC_METHOD, int(seed))] = rows
        STATIC_CACHE_AUDIT.append({
            "seed": int(seed), "candidate": str(accepted_path), "state": state,
            "detail": "exact data/split/model/environment match"})
        print({"static": STATIC_METHOD, "seed": int(seed), "state": state})
        continue
    if KAGGLE_RUN_PHASE != "phase_1_baselines":
        STATIC_CACHE_AUDIT.append({
            "seed": int(seed), "candidate": "", "state": "missing",
            "detail": "aggregate_only does not train"})
        continue
    if (STOP_AFTER_STATIC_RUNS is not None
            and STATIC_RUNS_STARTED_THIS_SESSION >= STOP_AFTER_STATIC_RUNS):
        STATIC_SESSION_STOP_REASON = "static_run_cap"
        break
    if ((time.monotonic() - SESSION_STARTED_AT) / 3600
            >= SESSION_LAUNCH_CUTOFF_HOURS):
        STATIC_SESSION_STOP_REASON = "static_launch_cutoff"
        break
    try:
        STATIC_RUNS_STARTED_THIS_SESSION += 1
        rows, period_predictions = _evaluate_static_mlp(seed)
        path = _save_static_artifact(seed, rows, period_predictions)
        _validate_static_artifact(path, seed, require_sidecar=True)
        STATIC_RESULTS[(STATIC_METHOD, int(seed))] = rows
        STATIC_CACHE_AUDIT.append({
            "seed": int(seed), "candidate": str(path), "state": "trained_v15",
            "detail": "fit once on initial temporal window"})
        print({"static": STATIC_METHOD, "seed": int(seed), "state": "trained_v15"})
    except Exception as exc:
        STATIC_CACHE_AUDIT.append({
            "seed": int(seed), "candidate": "", "state": "failed",
            "detail": f"{type(exc).__name__}: {exc}"})
        print("Static-MLP failed", seed, repr(exc))

_write_static_csv(
    TABLE_OUT / "v15_static_mlp_monthly_raw.csv",
    [row for rows in STATIC_RESULTS.values() for row in rows])
_write_static_csv(TABLE_OUT / "v15_static_cache_audit.csv", STATIC_CACHE_AUDIT)

def _export_static_checkpoint_atomic():
    checkpoint_path = OUT / f"{EXPERIMENT_NAME}_checkpoint_raw.zip"
    temporary = checkpoint_path.with_name(checkpoint_path.name + ".tmp")
    with zipfile.ZipFile(
            temporary, "w", compression=zipfile.ZIP_DEFLATED,
            compresslevel=1) as archive:
        for path in sorted(STATIC_CACHE_DIR.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(OUT))
    os.replace(temporary, checkpoint_path)
    return checkpoint_path

STATIC_CHECKPOINT = _export_static_checkpoint_atomic()
print({"static_valid": len(STATIC_RESULTS), "static_expected": len(SEEDS),
       "static_runs_started_this_session": STATIC_RUNS_STARTED_THIS_SESSION,
       "static_stop_reason": STATIC_SESSION_STOP_REASON,
       "static_checkpoint": str(STATIC_CHECKPOINT)})


## Ô 9 — Huấn luyện các nhánh DRMD của pha đã chọn


In [ ]:
from datetime import datetime, timezone
from copy import deepcopy
import csv
import hashlib
import json
import os
import pickle
import shutil
import traceback
import uuid

import DRMD.base as drmd_base
drmd_base.os = os
from DRMD.base import run
from DRMD.config import Settings, generate_experiment_configs

RUN_PROGRESS_LOG = OUT / "run_progress.jsonl"
FAILURE_LOG = OUT / "failed_runs.jsonl"
QUARANTINE_LOG = OUT / "quarantine_log.jsonl"
RUN_STATUS_PATH = OUT / "run_status.json"
DRMD_RUNS_STARTED_THIS_SESSION = 0
DRMD_RUNS_COMPLETED_THIS_SESSION = 0
DRMD_SESSION_STOPPED = False
SESSION_STOP_REASON = "completed_plan"
FAILED_RUNS = []

REQUIRED_RESULT_SERIES = {
    "tp", "tn", "fp", "fn", "p", "n", "tot",
    "selected", "rejected", "selected_indexes_all", "rejected_indexes_all",
    "rejected_malware_count", "rejected_benign_count",
    "y_tests", "y_preds", "t_tests",
    "binary_y_true_all", "binary_y_preds_all", "binary_t_all",
    "selection_diagnostics", "audit_indexes_all",
    "fn_penalty_used", "fn_penalty_next", "audit_count",
    "policy_action_all",
    "afnp_target_fnr", "afnp_estimated_fnr", "fn_controller_telemetry",
    "audit_positive_count", "audit_fn_count",
    "rejected_y_true", "selected_y_true",
    "selected_malware_count", "selected_benign_count",
    "bhr_telemetry", "drift_gate_telemetry",
    "adaptive_bhr_quota_telemetry",
}

def session_elapsed_hours():
    return (time.monotonic() - SESSION_STARTED_AT) / 3600

def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), default=str)

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def append_jsonl(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False, default=str) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

def atomic_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False,
                                    default=str), encoding="utf-8")
    os.replace(temporary, path)

def write_progress_event(event, cfg=None, extra=None):
    payload = {
        "time_utc": datetime.now(timezone.utc).isoformat(),
        "event": event,
        "elapsed_hours": round(session_elapsed_hours(), 4),
        "runtime_platform_requested": RUNTIME_PLATFORM,
        "runtime_platform_resolved": RESOLVED_RUNTIME_PLATFORM,
        "runtime_input_root": str(RUNTIME_INPUT_ROOT),
        "runtime_work_root": str(RUNTIME_WORK_ROOT),
    }
    if cfg is not None:
        payload.update({"seed": int(cfg.seed), "al": int(cfg.al_rate),
                        "rej": int(cfg.reject_rate), "reject_cost": float(cfg.reject_cost),
                        "is_reject_action": bool(cfg.is_reject_action),
                        "is_fine_tuning": bool(cfg.is_fine_tuning),
                        "al_select_mode": str(cfg.al_select_mode),
                        "select_reject_labeled": bool(cfg.select_reject_labeled),
                        "posthoc_augmented_AL": int(cfg.posthoc_augmented_AL)})
    if extra:
        payload.update(extra)
    append_jsonl(RUN_PROGRESS_LOG, payload)

def legacy_result_stem(cfg):
    """Tái tạo tên dài của DRMD gốc, chỉ dùng cho truy vết và cache cũ."""
    data = str(cfg.dataset).split(",")
    ds = data[0].split("/")[1]
    fs = data[1].split("-")[0]
    stem = f"DRMD-{cfg.model}-DS{ds}-FS{fs}"
    stem += f"-mp{cfg.minority_priority}"
    stem += f"{f'-Ts{cfg.temporal_scaling}' if cfg.temporal_rewards else ''}"
    stem += f"-bs{cfg.minibatch_size}"
    stem += f"-e{cfg.update_epochs}x{cfg.training_epochs}"
    stem += f"-HL{cfg.layer_size}x{cfg.hidden_layers}"
    stem += f"{f'-FT{cfg.finetuning_size}' if cfg.is_fine_tuning else ''}"
    stem += f"-Rej{cfg.reject_rate}-RA{cfg.is_reject_action}-RC{cfg.reject_cost}"
    stem += f"-RO{cfg.reward_rejected_outcome}-RS{cfg.reject_positive_scale}x{cfg.reject_negative_scale}"
    stem += f"-AL{cfg.al_rate}-SelectReject{cfg.select_reject_labeled}"
    stem += f"{f'-PHAAL{cfg.posthoc_augmented_AL}' if cfg.posthoc_augmented_AL > 0 else ''}"
    stem += f"-ALMode{cfg.al_select_mode}-Seed{cfg.seed}"
    return stem

def expected_result_path(cfg):
    return Path(cfg.results_save_location) / (compact_artifact_stem(
        legacy_result_stem(cfg)) + ".p")

def legacy_result_path(cfg):
    return Path(cfg.results_save_location) / (legacy_result_stem(cfg) + ".p")

def compatible_cached_paths(cfg):
    """Nhận cache tên ngắn và cache cũ nếu tên cũ hợp lệ trên hệ tệp."""
    paths = [expected_result_path(cfg)]
    legacy_path = legacy_result_path(cfg)
    # ``.meta.json`` được ghi cùng artifact; giữ biên nhỏ hơn 255 byte.
    if (legacy_path != paths[0]
            and len((legacy_path.name + ".meta.json").encode("utf-8")) <= 255):
        paths.append(legacy_path)
    return paths

def sidecar_path(artifact_path):
    return Path(str(artifact_path) + ".meta.json")

def source_code_fingerprints():
    paths = {
        "drmd_environment": DRMD_ROOT / "DRMD/environment.py",
        "drmd_base": DRMD_ROOT / "DRMD/base.py",
        "drmd_classifier_utils": DRMD_ROOT / "DRMD/utils/classifier_utils.py",
        "drmd_config": DRMD_ROOT / "DRMD/config.py",
        "drmd_agents": DRMD_ROOT / "DRMD/agents.py",
        "tesseract_temporal": TESSERACT_ROOT / "tesseract/temporal.py",
        "tesseract_metrics": TESSERACT_ROOT / "tesseract/metrics.py",
        "tesseract_selection": TESSERACT_ROOT / "tesseract/selection.py",
        "tesseract_evaluation": TESSERACT_ROOT / "tesseract/evaluation.py",
        "balanced_hard_replay": STRATEGY_MODULE_ROOT / "balanced_hard_replay.py",
        "coordinated_bhr": STRATEGY_MODULE_ROOT / "coordinated_balanced_hard_replay.py",
        "reliable_fn_controller": STRATEGY_MODULE_ROOT / "reliable_fn_penalty.py",
        "drift_aware_fn_controller": STRATEGY_MODULE_ROOT / "drift_aware_fn_penalty.py",
        "adaptive_bhr": STRATEGY_MODULE_ROOT / "adaptive_balanced_hard_replay.py",
    }
    missing_sources = {
        name: str(path) for name, path in paths.items() if not path.is_file()
    }
    if missing_sources:
        raise FileNotFoundError(
            "Thiếu tệp nguồn bắt buộc để lập chữ ký: "
            + json.dumps(missing_sources, ensure_ascii=False))
    fingerprints = {
        name: sha256_file(path) for name, path in paths.items()
    }
    fingerprints["notebook_runtime_patch"] = PATCH_IMPLEMENTATION_FINGERPRINT
    fingerprints["notebook_data_pipeline"] = (
        DATA_PIPELINE_IMPLEMENTATION_FINGERPRINT)
    return fingerprints

SOURCE_CODE_FINGERPRINTS = source_code_fingerprints()

def run_spec(cfg, label, track, variant, policy=None):
    return {
        "label": label,
        "variant_name": variant["name"],
        "track": track,
        "policy": policy["name"] if policy else "external",
        "policy_scope": policy.get("scope") if policy else "primary",
        "fn_penalty": float(variant["fn_penalty"]),
        "audit_sampling": bool(variant.get("audit_sampling", False)),
        "adaptive_fn_penalty": bool(variant.get("adaptive_fn_penalty", False)),
        "drift_aware_fn_penalty": bool(variant.get("drift_aware_fn_penalty", False)),
        "adaptive_bhr_quota": bool(variant.get("adaptive_bhr_quota", False)),
        "implementation": str(variant.get("implementation", "existing")),
        "fn_controller_confidence_level": FN_CONTROLLER_CONFIDENCE_LEVEL,
        "fn_controller_window_periods": FN_CONTROLLER_WINDOW_PERIODS,
        "balanced_hard_replay": bool(variant.get("balanced_hard_replay", False)),
        "bhr_memory_cap": STRATEGY_FINETUNING_SIZE,
        "bhr_fn_fraction": BHR_FN_FRACTION,
        "bhr_fp_fraction": BHR_FP_FRACTION,
        "bhr_background_fraction": BHR_BACKGROUND_FRACTION,
        "bhr_fn_max_repeat": BHR_FN_MAX_REPEAT,
        "bhr_fp_max_repeat": BHR_FP_MAX_REPEAT,
        "software_environment": {
            key: ENVIRONMENT_STAMP.get(key)
            for key in ("python", "torch", "numpy", "scipy",
                        "scikit_learn", "cuda",
                        "cublas_workspace_config",
                        "determinism_policy",
                        "installed_runtime_distributions")
        },
        "mp": float(cfg.minority_priority),
        "external_al_budget": int(cfg.al_rate),
        "external_reject_budget": int(cfg.reject_rate),
        "reject_cost": float(cfg.reject_cost),
        "seed": int(cfg.seed),
        "is_reject_action": bool(cfg.is_reject_action),
        "is_reject": bool(cfg.is_reject),
        "reward_rejected_outcome": bool(cfg.reward_rejected_outcome),
        "is_active": bool(cfg.is_active),
        "is_fine_tuning": bool(cfg.is_fine_tuning),
        "select_reject_labeled": bool(cfg.select_reject_labeled),
        "posthoc_augmented_AL": int(cfg.posthoc_augmented_AL),
        "integrated_reject_volume": "policy_endogenous",
        "feedback_set_rule": (
            "Q_t=all_samples" if variant.get("full_feedback", False)
            else ("Q_t=R_t" if variant.get("selector_mode") == "iral"
                  else f"Q_t={variant.get('selector_mode')}_budget_{variant.get('feedback_budget')}")),
        "al_select_mode": str(cfg.al_select_mode),
        "training_window_mode": TRAINING_WINDOW_MODE,
        "temporal_reference_policy": FIXED_TEMPORAL_REFERENCE_POLICY,
        "training_window": TRAINING_WINDOW,
        "testing_window": TESTING_WINDOW,
        "granularity": GRANULARITY,
        "training_epochs": int(cfg.training_epochs),
        "update_epochs": int(cfg.update_epochs),
        "minibatch_size": int(cfg.minibatch_size),
        "layer_size": int(cfg.layer_size),
        "hidden_layers_parameter": int(cfg.hidden_layers),
        # PPO_Network tạo một lớp Linear đầu vào rồi lặp hidden_layers lần.
        "actual_hidden_linear_layers": int(cfg.hidden_layers) + 1,
        "expected_periods": int(EXPECTED_PERIODS),
        "expected_test_samples": int(EXPECTED_TEST_SAMPLES),
        "bhr_error_partition_prediction": "binary_argmax_head",
        "policy_action_space": [0, 1, 2],
        "reject_action_index": 2,
        "fn_reward_scope": "direct_classification_only",
        "reject_counterfactual_reward": "unpenalized_drmd_classification",
        "fn_controller": ("drift_aware_beta_controller" if
            variant.get("drift_aware_fn_penalty", False) else
            ("beta_credible_interval_gated" if
             variant.get("adaptive_fn_penalty", False) else "disabled")),
        "bhr_fn_coordination": ("adaptive_audit_risk_quota" if
            variant.get("adaptive_bhr_quota", False) else
            ("base_fraction_div_sqrt_penalty" if
             variant.get("balanced_hard_replay", False) else "disabled")),
        "feedback_selector_mode": str(variant.get("selector_mode", "iral")),
        "feedback_budget": int(variant.get("feedback_budget", 0)),
        "candidate_multiplier": int(variant.get("candidate_multiplier", 1)),
        "full_feedback": bool(variant.get("full_feedback", False)),
        "training_history_cap": getattr(cfg, "_training_history_cap", None),
        "training_history_policy": (
            RECENCY_POLICY if getattr(cfg, "_training_history_cap", None) is not None
            else None),
        "feedback_timing": (
            "predict_t_then_fit_t_plus_1"
            if variant.get("full_feedback", False)
            else "query_after_predict_t_then_fit_t_plus_1"),
        "single_classifier_initialization": True,
    }

SIGNATURE_SCHEMA_VERSION = 9
def _signature_for_components(spec, dataset_fingerprint, split_fingerprint,
                              source_fingerprints, parameter_provenance):
    payload = {
        "protocol_version": PROTOCOL_VERSION,
        "dataset_source_fingerprint": dataset_fingerprint,
        "split_fingerprint": split_fingerprint,
        "parameter_provenance": parameter_provenance,
        "source_code_fingerprints": source_fingerprints,
        "run_spec": spec,
    }
    return hashlib.sha256(canonical_json(payload).encode()).hexdigest()

def signature_for_spec(spec):
    return _signature_for_components(
        spec, DATASET_SOURCE_FINGERPRINT, SPLIT_FINGERPRINT,
        SOURCE_CODE_FINGERPRINTS, PARAMETER_PROVENANCE)

def _signature_from_sidecar_snapshot(metadata):
    """Tái tạo chữ ký gốc mà không thay dấu vết đã lưu trong sidecar."""
    return _signature_for_components(
        metadata["run_spec"], metadata["dataset_source_fingerprint"],
        metadata["split_fingerprint"], metadata["source_code_fingerprints"],
        metadata.get("parameter_provenance", PARAMETER_PROVENANCE))

def _as_index_vector(values, period, name):
    raw = np.asarray(values)
    if raw.ndim != 1:
        raise ValueError(f"Chu kỳ {period}: {name} phải là vector một chiều.")
    if raw.size and not np.all(np.equal(raw, raw.astype(np.int64))):
        raise ValueError(f"Chu kỳ {period}: {name} chứa chỉ số không nguyên.")
    indexes = raw.astype(np.int64, copy=False)
    if np.unique(indexes).size != indexes.size:
        raise ValueError(f"Chu kỳ {period}: {name} chứa chỉ số trùng.")
    return indexes

def _as_time_vector(values, period, name):
    try:
        return np.asarray(values).reshape(-1).astype("datetime64[ns]")
    except Exception as exc:
        raise ValueError(
            f"Chu kỳ {period}: {name} không chuyển được sang thời gian chuẩn.") from exc

def _same_float(observed, expected, atol=1e-10):
    try:
        left, right = float(observed), float(expected)
    except (TypeError, ValueError):
        return False
    if np.isnan(left) and np.isnan(right):
        return True
    return bool(np.isclose(left, right, rtol=0.0, atol=atol))

def _expected_fit_periods(selected_counts, expected_periods):
    try:
        period_count = int(expected_periods)
    except (TypeError, ValueError) as exc:
        raise ValueError("expected_periods không hợp lệ để suy ra lịch fit.") from exc
    if period_count < 0 or not _same_float(expected_periods, period_count):
        raise ValueError("expected_periods không hợp lệ để suy ra lịch fit.")
    if (not hasattr(selected_counts, "__len__")
            or len(selected_counts) != period_count):
        raise ValueError(
            "Không suy ra được lịch fit: selected sai số chu kỳ.")

    counts = []
    for period, raw_count in enumerate(selected_counts):
        try:
            count = int(raw_count)
        except (TypeError, ValueError) as exc:
            raise ValueError(
                f"Chu kỳ {period}: selected không phải số nguyên không âm.") from exc
        if count < 0 or not _same_float(raw_count, count):
            raise ValueError(
                f"Chu kỳ {period}: selected không phải số nguyên không âm.")
        counts.append(count)

    if period_count == 0:
        return []
    return [0] + [
        period for period in range(1, period_count)
        if counts[period - 1] > 0
    ]

def _validate_temporal_reference_history(
        selected_counts, temporal_history, canonical_times,
        expected_periods, training_window):
    fit_periods = _expected_fit_periods(selected_counts, expected_periods)
    if (not hasattr(temporal_history, "__len__")
            or len(temporal_history) != len(fit_periods)):
        observed = (len(temporal_history)
                    if hasattr(temporal_history, "__len__") else "không có độ dài")
        raise ValueError(
            "temporal_reference_history sai số lần fit: "
            f"nhận {observed}, kỳ vọng {len(fit_periods)} "
            f"theo lịch phản hồi {fit_periods}.")
    if (not hasattr(canonical_times, "__len__")
            or len(canonical_times) != int(expected_periods)):
        raise ValueError("t_tests chuẩn sai số chu kỳ khi kiểm tra lịch fit.")

    for fit_index, period in enumerate(fit_periods):
        temporal = temporal_history[fit_index]
        if not isinstance(temporal, dict):
            raise TypeError(
                f"Lần fit {fit_index}: temporal reference không phải dict.")
        canonical_t = _as_time_vector(
            canonical_times[period], period, "t_tests chuẩn")
        if canonical_t.size == 0:
            raise ValueError(
                f"Chu kỳ {period}: không có thời gian để kiểm tra lần fit.")
        test_month = canonical_t[0].astype("datetime64[M]")
        year = int(test_month.astype("datetime64[Y]").astype(int) + 1970)
        month = int(test_month.astype(int) % 12 + 1)
        test_absolute_month = year * 12 + month
        if (int(temporal.get("fit_index", -1)) != fit_index
                or int(temporal.get("reference_period", -1))
                != int(training_window)
                or int(temporal.get("memory_period", 0)) <= 0
                or int(temporal.get("memory_end", test_absolute_month))
                >= test_absolute_month):
            raise ValueError(
                f"Lần fit {fit_index} tại chu kỳ {period}: "
                "dấu vết thời gian vi phạm nhân quả.")
        factors = [float(temporal.get(key, np.nan)) for key in (
            "factor_min", "factor_max", "factor_mean")]
        if not np.isfinite(factors).all() or factors[0] > factors[1]:
            raise ValueError(
                f"Lần fit {fit_index} tại chu kỳ {period}: "
                "hệ số thời gian không hợp lệ.")
    return fit_periods

def _confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.int64).reshape(-1)
    return {
        "tp": int(np.sum((y_true == 1) & (y_pred == 1))),
        "tn": int(np.sum((y_true == 0) & (y_pred == 0))),
        "fp": int(np.sum((y_true == 0) & (y_pred == 1))),
        "fn": int(np.sum((y_true == 1) & (y_pred == 0))),
    }

def validate_result_object(result, expected_periods, policy=None, variant=None):
    if not isinstance(result, dict):
        raise TypeError("Artifact DRMD phải là dict.")
    missing = sorted(REQUIRED_RESULT_SERIES - set(result))
    if missing:
        raise ValueError(f"Artifact thiếu khóa: {missing}")
    wrong_lengths = {}
    for key in REQUIRED_RESULT_SERIES:
        if not hasattr(result[key], "__len__"):
            wrong_lengths[key] = "không có độ dài"
        elif len(result[key]) != expected_periods:
            wrong_lengths[key] = len(result[key])
    if wrong_lengths:
        raise ValueError(
            f"Số chu kỳ không đúng: {wrong_lengths}; kỳ vọng {expected_periods}")
    for key in ("selection_history", "temporal_reference_history",
                "strategy_runtime_config",
                "drift_calibration_telemetry"):
        if key not in result:
            raise ValueError(f"Artifact thiếu khóa cấp lượt chạy: {key}")
    if len(result["selection_history"]) != expected_periods:
        raise ValueError("selection_history sai số chu kỳ.")
    _validate_temporal_reference_history(
        result["selected"], result["temporal_reference_history"], t_tests,
        expected_periods, TRAINING_WINDOW)

    runtime = result["strategy_runtime_config"]
    if not isinstance(runtime, dict):
        raise TypeError("strategy_runtime_config phải là dict.")
    expected_runtime = {
        "feedback_timing": "predict_t_then_fit_t_plus_1",
        "single_classifier_initialization": True,
        "temporal_reference_policy": FIXED_TEMPORAL_REFERENCE_POLICY,
        "temporal_reference_period": TRAINING_WINDOW,
        "training_history_cap": STRATEGY_FINETUNING_SIZE,
        "bhr_error_partition_prediction": "binary_argmax_head",
        "policy_action_space": [0, 1, 2],
        "reject_action_index": 2,
        "fn_reward_scope": "direct_classification_only",
        "reject_counterfactual_reward": "unpenalized_drmd_classification",
    }
    if variant is not None:
        adaptive = bool(variant.get("adaptive_fn_penalty", False))
        drift_aware = bool(variant.get("drift_aware_fn_penalty", False))
        adaptive_quota = bool(variant.get("adaptive_bhr_quota", False))
        if adaptive and drift_aware:
            expected_controller_config = DriftAwareFNPenaltyController(
                label_budget=int(variant["feedback_budget"]),
                label_window_periods=FN_CONTROLLER_WINDOW_PERIODS,
                label_confidence_level=FN_CONTROLLER_CONFIDENCE_LEVEL,
                drift_projection_dim=DRIFT_PROJECTION_DIM,
                drift_reference_window=DRIFT_REFERENCE_WINDOW,
                drift_threshold_quantile=DRIFT_THRESHOLD_QUANTILE,
                drift_minimum_score_history=DRIFT_MINIMUM_SCORE_HISTORY,
                drift_max_rows_per_period=DRIFT_MAX_ROWS_PER_PERIOD,
                drift_decay_periods=DRIFT_DECAY_PERIODS).configuration()
        elif adaptive:
            expected_controller_config = ReliableAdaptiveFNPenaltyController(
                label_budget=int(variant["feedback_budget"]),
                window_periods=FN_CONTROLLER_WINDOW_PERIODS,
                confidence_level=FN_CONTROLLER_CONFIDENCE_LEVEL).configuration()
        else:
            expected_controller_config = None
        expected_runtime.update({
            "feedback_selector_mode": str(variant["selector_mode"]),
            "feedback_budget": int(variant["feedback_budget"]),
            "candidate_multiplier": int(variant["candidate_multiplier"]),
            "adaptive_fn_penalty": adaptive,
            "audit_sampling": bool(variant.get("audit_sampling", False)),
            "balanced_hard_replay": bool(
                variant.get("balanced_hard_replay", False)),
            "drift_aware_fn_penalty": drift_aware,
            "adaptive_bhr_quota": adaptive_quota,
            "full_feedback": bool(variant.get("full_feedback", False)),
            "bhr_fn_fraction": float(BHR_FN_FRACTION),
            "bhr_fp_fraction": float(BHR_FP_FRACTION),
            "bhr_fn_max_repeat": int(BHR_FN_MAX_REPEAT),
            "bhr_fp_max_repeat": int(BHR_FP_MAX_REPEAT),
            "fn_controller": ("drift_aware_beta_controller" if drift_aware
                else ("beta_credible_interval_gated" if adaptive else "disabled")),
            "bhr_fn_coordination": ("adaptive_audit_risk_quota" if adaptive_quota
                else ("base_fraction_div_sqrt_penalty" if
                      variant.get("balanced_hard_replay", False) else "disabled")),
            "fn_controller_config": expected_controller_config,
        })
    for key, expected in expected_runtime.items():
        if runtime.get(key) != expected:
            raise ValueError(
                f"strategy_runtime_config sai {key}: "
                f"{runtime.get(key)!r} != {expected!r}")
    if int(runtime.get("strategy_seed", -1)) not in SEEDS:
        raise ValueError("strategy_runtime_config thiếu/sai strategy_seed.")

    adaptive = bool(runtime.get("adaptive_fn_penalty", False))
    audit_sampling = bool(runtime.get("audit_sampling", False))
    bhr_enabled = bool(runtime.get("balanced_hard_replay", False))
    feedback_budget = int(runtime.get("feedback_budget", 0))
    if adaptive != audit_sampling:
        raise ValueError("V15 yêu cầu bộ điều khiển FN và mẫu kiểm toán cùng bật/tắt.")

    total_test_samples = 0
    for period in range(expected_periods):
        canonical_y = np.asarray(y_tests[period], dtype=np.int64).reshape(-1)
        canonical_t = _as_time_vector(t_tests[period], period, "t_tests chuẩn")
        canonical_total = int(X_tests[period].shape[0])
        if not (canonical_total == canonical_y.size == canonical_t.size):
            raise ValueError(f"Chu kỳ {period}: split chuẩn X/y/t lệch số hàng.")
        total_test_samples += canonical_total

        values = {
            key: int(result[key][period])
            for key in ("tp", "tn", "fp", "fn", "p", "n", "tot",
                        "selected", "rejected")
        }
        if min(values.values()) < 0:
            raise ValueError(f"Chu kỳ {period}: số đếm âm {values}")
        if values["tp"] + values["fn"] != values["p"]:
            raise ValueError(f"Chu kỳ {period}: TP + FN != P")
        if values["tn"] + values["fp"] != values["n"]:
            raise ValueError(f"Chu kỳ {period}: TN + FP != N")
        if (values["tp"] + values["tn"] + values["fp"] + values["fn"]
                != values["tot"]):
            raise ValueError(f"Chu kỳ {period}: ma trận nhầm lẫn != total accepted")
        if values["tot"] + values["rejected"] != canonical_total:
            raise ValueError(
                f"Chu kỳ {period}: tổng trước từ chối "
                f"{values['tot'] + values['rejected']} != {canonical_total}")

        selected = _as_index_vector(
            result["selected_indexes_all"][period], period, "selected indexes")
        rejected = _as_index_vector(
            result["rejected_indexes_all"][period], period, "rejected indexes")
        audit = _as_index_vector(
            result["audit_indexes_all"][period], period, "audit indexes")
        if selected.size != values["selected"] or rejected.size != values["rejected"]:
            raise ValueError(f"Chu kỳ {period}: số đếm chỉ số selected/rejected sai.")
        for name, indexes in (("selected", selected), ("rejected", rejected),
                              ("audit", audit)):
            if np.any((indexes < 0) | (indexes >= canonical_total)):
                raise ValueError(
                    f"Chu kỳ {period}: {name} vượt miền [0, {canonical_total}).")
        if np.setdiff1d(audit, selected).size:
            raise ValueError(f"Chu kỳ {period}: audit không phải tập con của selected.")

        binary_y = np.asarray(
            result["binary_y_true_all"][period], dtype=np.int64).reshape(-1)
        binary_pred_raw = np.asarray(result["binary_y_preds_all"][period]).reshape(-1)
        binary_t = _as_time_vector(
            result["binary_t_all"][period], period, "binary_t_all")
        if not (binary_y.size == binary_pred_raw.size == binary_t.size
                == canonical_total):
            raise ValueError(f"Chu kỳ {period}: dãy nhị phân sai số hàng.")
        if not np.array_equal(binary_y, canonical_y):
            raise ValueError(f"Chu kỳ {period}: binary_y_true lệch nhãn split chuẩn.")
        if not np.array_equal(binary_t, canonical_t):
            raise ValueError(f"Chu kỳ {period}: binary_t lệch thời gian split chuẩn.")
        if not np.isin(binary_pred_raw, [0, 1]).all():
            raise ValueError(f"Chu kỳ {period}: dự đoán nhị phân ngoài miền 0/1.")
        binary_pred = binary_pred_raw.astype(np.int64, copy=False)

        policy_action_raw = np.asarray(
            result["policy_action_all"][period]).reshape(-1)
        if policy_action_raw.size != canonical_total:
            raise ValueError(f"Chu kỳ {period}: dãy hành động chính sách sai số hàng.")
        if not np.isin(policy_action_raw, [0, 1, 2]).all():
            raise ValueError(f"Chu kỳ {period}: hành động chính sách ngoài miền 0/1/2.")
        policy_action = policy_action_raw.astype(np.int64, copy=False)
        expected_rejected = np.flatnonzero(policy_action == 2).astype(np.int64)
        if not np.array_equal(expected_rejected, rejected):
            raise ValueError(
                f"Chu kỳ {period}: chỉ số từ chối không khớp hành động 2.")

        kept = np.setdiff1d(
            np.arange(canonical_total, dtype=np.int64), rejected,
            assume_unique=True)
        accepted_y = np.asarray(result["y_tests"][period], dtype=np.int64).reshape(-1)
        accepted_pred_raw = np.asarray(result["y_preds"][period]).reshape(-1)
        accepted_t = _as_time_vector(
            result["t_tests"][period], period, "accepted t_tests")
        if not (accepted_y.size == accepted_pred_raw.size == accepted_t.size
                == values["tot"] == kept.size):
            raise ValueError(f"Chu kỳ {period}: dãy accepted sai số hàng.")
        if not np.isin(accepted_pred_raw, [0, 1]).all():
            raise ValueError(f"Chu kỳ {period}: dự đoán accepted ngoài miền 0/1.")
        accepted_pred = accepted_pred_raw.astype(np.int64, copy=False)
        if not np.array_equal(accepted_y, canonical_y[kept]):
            raise ValueError(f"Chu kỳ {period}: nhãn accepted lệch split chuẩn.")
        if not np.array_equal(accepted_t, canonical_t[kept]):
            raise ValueError(f"Chu kỳ {period}: thời gian accepted lệch split chuẩn.")
        if not np.array_equal(accepted_pred, binary_pred[kept]):
            raise ValueError(
                f"Chu kỳ {period}: dự đoán nhị phân và nhánh accepted không khớp.")
        recomputed = _confusion_counts(accepted_y, accepted_pred)
        if any(recomputed[key] != values[key] for key in recomputed):
            raise ValueError(
                f"Chu kỳ {period}: ma trận nhầm lẫn lưu {values} "
                f"khác ma trận tính lại {recomputed}.")

        expected_rejected_y = canonical_y[rejected]
        stored_rejected_y = np.asarray(
            result["rejected_y_true"][period], dtype=np.int64).reshape(-1)
        if not np.array_equal(stored_rejected_y, expected_rejected_y):
            raise ValueError(f"Chu kỳ {period}: nhãn rejected sai.")
        rejected_malware = int(np.sum(expected_rejected_y == 1))
        rejected_benign = int(np.sum(expected_rejected_y == 0))
        if (int(result["rejected_malware_count"][period]) != rejected_malware
                or int(result["rejected_benign_count"][period]) != rejected_benign):
            raise ValueError(f"Chu kỳ {period}: số lớp rejected sai.")

        expected_selected_y = canonical_y[selected]
        stored_selected_y = np.asarray(
            result["selected_y_true"][period], dtype=np.int64).reshape(-1)
        if not np.array_equal(stored_selected_y, expected_selected_y):
            raise ValueError(f"Chu kỳ {period}: nhãn selected sai.")
        selected_malware = int(np.sum(expected_selected_y == 1))
        selected_benign = int(np.sum(expected_selected_y == 0))
        if (int(result["selected_malware_count"][period]) != selected_malware
                or int(result["selected_benign_count"][period]) != selected_benign):
            raise ValueError(f"Chu kỳ {period}: số lớp selected sai.")

        expected_audit_positive = int(np.sum(canonical_y[audit] == 1))
        expected_audit_fn = int(np.sum(
            (canonical_y[audit] == 1) & (binary_pred[audit] == 0)))
        if (int(result["audit_count"][period]) != audit.size
                or int(result["audit_positive_count"][period])
                != expected_audit_positive
                or int(result["audit_fn_count"][period]) != expected_audit_fn):
            raise ValueError(f"Chu kỳ {period}: thống kê mẫu kiểm toán sai.")
        total_budget = min(feedback_budget, canonical_total)
        expected_audit_count = (
            min(total_budget, max(1, int(np.ceil(np.sqrt(total_budget)))))
            if audit_sampling and total_budget > 0 else 0)
        if audit.size != expected_audit_count:
            raise ValueError(
                f"Chu kỳ {period}: audit={audit.size}, "
                f"kỳ vọng {expected_audit_count}.")

        if policy is not None:
            name = policy["name"]
            if name == "IR" and selected.size:
                raise ValueError(f"Chu kỳ {period}: IR không được chọn mẫu huấn luyện")
            if name == "IRAL" and not np.array_equal(selected, rejected):
                raise ValueError(f"Chu kỳ {period}: IRAL phải dùng đúng tập bị từ chối")
            if name == "IRAAL" and selected.size != min(
                    int(policy["budget"]), canonical_total):
                raise ValueError(
                    f"Chu kỳ {period}: IRAAL chọn {selected.size}, "
                    f"kỳ vọng {min(int(policy['budget']), canonical_total)}")
            if name == "FFCR":
                expected_all = np.arange(canonical_total, dtype=np.int64)
                if not np.array_equal(selected, expected_all):
                    raise ValueError(
                        f"Chu kỳ {period}: FFCR không phủ toàn bộ mẫu.")

        diagnostic = result["selection_diagnostics"][period]
        if not isinstance(diagnostic, dict):
            raise TypeError(f"Chu kỳ {period}: selection diagnostic không phải dict.")
        if (int(diagnostic.get("n_rows", -1)) != canonical_total
                or int(diagnostic.get("n_selected", -1)) != selected.size
                or int(diagnostic.get("n_rejected_total", -1)) != rejected.size
                or bool(diagnostic.get("selection_used_labels", True))):
            raise ValueError(f"Chu kỳ {period}: selection diagnostic sai.")
        history = result["selection_history"][period]
        if (not isinstance(history, dict)
                or int(history.get("period_index", -1)) != period
                or int(history.get("n_selected", -1)) != selected.size
                or int(history.get("n_rejected_total", -1)) != rejected.size):
            raise ValueError(f"Chu kỳ {period}: selection_history sai.")

        telemetry = result["bhr_telemetry"][period]
        if not isinstance(telemetry, dict):
            raise TypeError(f"Chu kỳ {period}: BHR telemetry không phải dict.")
        if (bool(telemetry.get("enabled", False)) != bhr_enabled
                or int(telemetry.get("fit_period", -1)) != period + 1
                or not bool(telemetry.get("causal_lag_ok", False))):
            raise ValueError(f"Chu kỳ {period}: cờ BHR/nhân quả sai.")
        memory_rows = int(telemetry.get("memory_rows", -1))
        if memory_rows != STRATEGY_FINETUNING_SIZE:
            raise ValueError(
                f"Chu kỳ {period}: bộ nhớ {memory_rows}, "
                f"kỳ vọng {STRATEGY_FINETUNING_SIZE} hàng.")
        if bhr_enabled:
            if int(telemetry.get("memory_cap", -1)) != STRATEGY_FINETUNING_SIZE:
                raise ValueError(f"Chu kỳ {period}: memory_cap BHR sai.")
            component_rows = sum(int(telemetry.get(key, -1)) for key in (
                "fn_rows", "fp_rows", "background_rows"))
            if component_rows != memory_rows:
                raise ValueError(f"Chu kỳ {period}: thành phần BHR không cộng đủ.")
            if int(telemetry.get("newest_source_period", -1)) != period:
                raise ValueError(f"Chu kỳ {period}: BHR dùng phản hồi cùng/tương lai.")
            if bool(variant.get("adaptive_bhr_quota", False)):
                if telemetry.get("quota_evidence_source") != "random_audit_only":
                    raise ValueError(f"Chu kỳ {period}: quota không dùng kiểm toán ngẫu nhiên.")
                requested = sum(float(telemetry.get(key, np.nan)) for key in (
                    "requested_fn_fraction", "requested_fp_fraction",
                    "requested_background_fraction"))
                effective = sum(float(telemetry.get(key, np.nan)) for key in (
                    "effective_fn_fraction", "effective_fp_fraction",
                    "effective_background_fraction"))
                if (not _same_float(requested, 1.0)
                        or not _same_float(effective, 1.0)
                        or float(telemetry.get("requested_hard_fraction", 2.0))
                           > ADAPTIVE_BHR_MAX_HARD_FRACTION + 1e-12):
                    raise ValueError(f"Chu kỳ {period}: quota BHR động không hợp lệ.")
            else:
                if not bool(telemetry.get("coordination_enabled", False)):
                    raise ValueError(f"Chu kỳ {period}: thiếu điều phối BHR-FN.")
                coordination_penalty = float(
                    telemetry.get("coordination_fn_penalty", np.nan))
                expected_penalty = float(result["fn_penalty_next"][period])
                if not _same_float(coordination_penalty, expected_penalty):
                    raise ValueError(
                        f"Chu kỳ {period}: BHR không dùng penalty_next của t+1.")
                base_fraction = float(telemetry.get("base_fn_fraction", np.nan))
                effective_fraction = float(
                    telemetry.get("effective_fn_fraction", np.nan))
                expected_fraction = base_fraction / np.sqrt(expected_penalty)
                if (not np.isfinite(base_fraction)
                        or not _same_float(base_fraction, BHR_FN_FRACTION)
                        or not _same_float(effective_fraction, expected_fraction)):
                    raise ValueError(f"Chu kỳ {period}: quota FN điều phối sai.")
            fn_replay = int(telemetry.get("fn_replay_rows", -1))
            fp_replay = int(telemetry.get("fp_replay_rows", -1))
            unique_rows = int(telemetry.get("unique_rows_used", -1))
            if (fn_replay < 0 or fp_replay < 0
                    or not 0 < unique_rows <= memory_rows):
                raise ValueError(f"Chu kỳ {period}: số hàng replay/duy nhất sai.")
            replay_fraction = float(telemetry.get("replay_fraction", np.nan))
            expected_replay_fraction = (fn_replay + fp_replay) / memory_rows
            if (not np.isfinite(replay_fraction)
                    or not _same_float(replay_fraction, expected_replay_fraction)):
                raise ValueError(f"Chu kỳ {period}: replay_fraction không hợp lệ.")
            mean_age = float(telemetry.get("mean_labeled_age_periods", np.nan))
            max_age = float(telemetry.get("max_labeled_age_periods", np.nan))
            if (not np.isfinite(mean_age) or not np.isfinite(max_age)
                    or mean_age < 1.0 or max_age < mean_age):
                raise ValueError(f"Chu kỳ {period}: tuổi phản hồi không hợp lệ.")


    if total_test_samples != EXPECTED_TEST_SAMPLES:
        raise ValueError(
            f"Mỗi lượt có {total_test_samples} mẫu, "
            f"kỳ vọng {EXPECTED_TEST_SAMPLES}.")

    # Tái dựng controller và cổng drift từ đúng dữ liệu quá khứ.
    adaptive = bool(variant.get("adaptive_fn_penalty", False))
    drift_aware = bool(variant.get("drift_aware_fn_penalty", False))
    adaptive_quota = bool(variant.get("adaptive_bhr_quota", False))
    if adaptive and drift_aware:
        replay_controller = DriftAwareFNPenaltyController(
            label_budget=feedback_budget,
            label_window_periods=FN_CONTROLLER_WINDOW_PERIODS,
            label_confidence_level=FN_CONTROLLER_CONFIDENCE_LEVEL,
            drift_projection_dim=DRIFT_PROJECTION_DIM,
            drift_reference_window=DRIFT_REFERENCE_WINDOW,
            drift_threshold_quantile=DRIFT_THRESHOLD_QUANTILE,
            drift_minimum_score_history=DRIFT_MINIMUM_SCORE_HISTORY,
            drift_max_rows_per_period=DRIFT_MAX_ROWS_PER_PERIOD,
            drift_decay_periods=DRIFT_DECAY_PERIODS)
        train_months = np.asarray(t_train).astype("datetime64[M]")
        expected_calibration = [snapshot.to_dict() for snapshot in
            replay_controller.calibrate_unlabeled(
                [X_train[np.flatnonzero(train_months == month)]
                 for month in np.sort(np.unique(train_months))],
                input_dim=int(X_train.shape[1]),
                seed=int(runtime["strategy_seed"]))]
        stored_calibration = result["drift_calibration_telemetry"]
        if len(stored_calibration) != len(expected_calibration):
            raise ValueError("Hiệu chuẩn drift không đủ số khối huấn luyện.")
        for calibration_index, expected_row in enumerate(expected_calibration):
            observed_row = stored_calibration[calibration_index]
            for key, expected_value in expected_row.items():
                observed_value = observed_row.get(key)
                if isinstance(expected_value, float):
                    if not _same_float(observed_value, expected_value):
                        raise ValueError(
                            f"Hiệu chuẩn drift {calibration_index}: {key} sai.")
                elif observed_value != expected_value:
                    raise ValueError(
                        f"Hiệu chuẩn drift {calibration_index}: {key} sai.")
    elif adaptive:
        replay_controller = ReliableAdaptiveFNPenaltyController(
            label_budget=feedback_budget,
            window_periods=FN_CONTROLLER_WINDOW_PERIODS,
            confidence_level=FN_CONTROLLER_CONFIDENCE_LEVEL)
    else:
        replay_controller = None
    if not drift_aware and result["drift_calibration_telemetry"] != []:
        raise ValueError("Nhánh không có cổng drift vẫn ghi dữ liệu hiệu chuẩn.")
    quota_replay = (AdaptiveReplayQuotaController(
        label_budget=feedback_budget,
        window_periods=ADAPTIVE_BHR_WINDOW_PERIODS,
        maximum_hard_fraction=ADAPTIVE_BHR_MAX_HARD_FRACTION)
        if adaptive_quota else None)
    previous_next = 1.0
    for period in range(expected_periods):
        used = float(result["fn_penalty_used"][period])
        next_value = float(result["fn_penalty_next"][period])
        telemetry = result["fn_controller_telemetry"][period]
        if not isinstance(telemetry, dict):
            raise TypeError(f"Chu kỳ {period}: controller telemetry không phải dict.")
        if (not _same_float(used, previous_next)
                or not _same_float(used, telemetry.get("penalty_used"))
                or not _same_float(next_value, telemetry.get("penalty_next"))):
            raise ValueError(f"Chu kỳ {period}: mức phạt không đúng độ trễ t+1.")
        ceiling = float(telemetry.get("penalty_ceiling", 1.0))
        if not (1.0 <= next_value <= ceiling + 1e-12):
            raise ValueError(f"Chu kỳ {period}: mức phạt vượt miền tự động.")

        audit = _as_index_vector(
            result["audit_indexes_all"][period], period, "audit indexes")
        audit_true = np.asarray(y_tests[period], dtype=int)[audit]
        audit_pred = np.asarray(
            result["binary_y_preds_all"][period], dtype=int)[audit]
        if replay_controller is not None:
            if drift_aware:
                expected_drift = replay_controller.observe_unlabeled(
                    X_tests[period])
                stored_drift = result["drift_gate_telemetry"][period]
                expected_drift_dict = expected_drift.to_dict()
                for key, expected_value in expected_drift_dict.items():
                    observed_value = stored_drift.get(key)
                    if isinstance(expected_value, float):
                        if not _same_float(observed_value, expected_value):
                            raise ValueError(
                                f"Chu kỳ {period}: drift {key} sai.")
                    elif observed_value != expected_value:
                        raise ValueError(f"Chu kỳ {period}: drift {key} sai.")
                expected_snapshot = replay_controller.observe_labeled(
                    audit_true, audit_pred, expected_drift).to_dict()
            else:
                expected_snapshot = replay_controller.observe(
                    audit_true, audit_pred).to_dict()
            for key, expected_value in expected_snapshot.items():
                observed_value = telemetry.get(key)
                if isinstance(expected_value, float):
                    if not _same_float(observed_value, expected_value):
                        raise ValueError(
                            f"Chu kỳ {period}: controller {key} sai.")
                elif observed_value != expected_value:
                    raise ValueError(f"Chu kỳ {period}: controller {key} sai.")
        elif (not _same_float(used, 1.0) or not _same_float(next_value, 1.0)
              or telemetry.get("update_direction") != "disabled"):
            raise ValueError(f"Chu kỳ {period}: nhánh không FN vẫn đổi mức phạt.")

        if quota_replay is not None:
            expected_quota = quota_replay.observe(
                audit_true, audit_pred, fn_penalty=next_value).to_dict()
            stored_quota = result["adaptive_bhr_quota_telemetry"][period]
            for key, expected_value in expected_quota.items():
                observed_value = stored_quota.get(key)
                if isinstance(expected_value, float):
                    if not _same_float(observed_value, expected_value):
                        raise ValueError(f"Chu kỳ {period}: quota {key} sai.")
                elif observed_value != expected_value:
                    raise ValueError(f"Chu kỳ {period}: quota {key} sai.")
        previous_next = next_value

    return {
        "periods": int(expected_periods),
        "test_samples": int(total_test_samples),
        "causal_lag_violations": 0,
        "memory_cap_violations": 0,
        "canonical_split_alignment": True,
        "confusion_counts_recomputed": True,
    }

def validate_result_file(path, expected_periods, policy=None, variant=None):
    with open(path, "rb") as handle:
        result = pickle.load(handle)
    return validate_result_object(
        result, expected_periods, policy=policy, variant=variant)

def _non_patch_fingerprint_differences(stored, current):
    ignored = {"notebook_runtime_patch"}
    keys = (set(stored) | set(current)) - ignored
    return sorted(key for key in keys if stored.get(key) != current.get(key))

def validate_sidecar(artifact_path, signature):
    metadata_path = sidecar_path(artifact_path)
    if not metadata_path.exists():
        raise FileNotFoundError("Thiếu sidecar của artifact giao thức V15")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("status") != "complete":
        raise ValueError("Sidecar không có trạng thái complete")
    if metadata.get("protocol_version") != PROTOCOL_VERSION:
        raise ValueError("Sai protocol_version")
    if int(metadata.get("signature_schema_version", -1)) != SIGNATURE_SCHEMA_VERSION:
        raise ValueError("Sai signature_schema_version")
    if metadata.get("dataset_source_fingerprint") != DATASET_SOURCE_FINGERPRINT:
        raise ValueError("Sai dataset_source_fingerprint")
    if metadata.get("split_fingerprint") != SPLIT_FINGERPRINT:
        raise ValueError("Sai split_fingerprint")
    if metadata.get("source_code_fingerprints") != SOURCE_CODE_FINGERPRINTS:
        raise ValueError("Sai source_code_fingerprints")
    if metadata.get("parameter_provenance") != PARAMETER_PROVENANCE:
        raise ValueError("Sai parameter_provenance")
    if metadata.get("run_signature") != signature:
        raise ValueError("Sai run_signature; V15 không nhận cache giao thức cũ")
    if metadata.get("artifact_sha256") != sha256_file(artifact_path):
        raise ValueError("Sai SHA-256 của artifact")
    if metadata.get("run_signature") != signature_for_spec(metadata["run_spec"]):
        raise ValueError("run_spec của sidecar không khớp cấu hình đang yêu cầu")
    if int(metadata["run_spec"].get("expected_periods", -1)) != EXPECTED_TEST_PERIODS:
        raise ValueError("run_spec sai số chu kỳ kiểm thử")
    if int(metadata["run_spec"].get("expected_test_samples", -1)) != EXPECTED_TEST_SAMPLES:
        raise ValueError("run_spec sai số mẫu kiểm thử")
    validation = metadata.get("validation", {})
    if (int(validation.get("periods", -1)) != EXPECTED_TEST_PERIODS
            or int(validation.get("test_samples", -1)) != EXPECTED_TEST_SAMPLES
            or not bool(validation.get("canonical_split_alignment", False))
            or int(validation.get("causal_lag_violations", -1)) != 0
            or int(validation.get("memory_cap_violations", -1)) != 0):
        raise ValueError("Sidecar thiếu bằng chứng toàn vẹn V15")
    return metadata

def quarantine_path(path, reason):
    if not path.exists():
        return None
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    try:
        relative = path.relative_to(OUT)
    except ValueError:
        relative = Path(path.name)
    target = QUARANTINE_OUT / timestamp / uuid.uuid4().hex[:8] / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(path), str(target))
    append_jsonl(QUARANTINE_LOG, {
        "time_utc": datetime.now(timezone.utc).isoformat(),
        "source": str(path), "target": str(target), "reason": str(reason),
    })
    return target

def quarantine_artifact(artifact_path, reason):
    moved = quarantine_path(artifact_path, reason)
    metadata_path = sidecar_path(artifact_path)
    if metadata_path.exists():
        quarantine_path(metadata_path, reason)
    return moved

def cache_is_valid(artifact_path, signature, policy, variant, require_sidecar):
    if not artifact_path.exists():
        return False
    try:
        validate_result_file(
            artifact_path, EXPECTED_PERIODS, policy=policy, variant=variant)
        if require_sidecar:
            validate_sidecar(artifact_path, signature)
        elif sidecar_path(artifact_path).exists():
            validate_sidecar(artifact_path, signature)
        return True
    except Exception as exc:
        print(f"⚠️ Cache không hợp lệ, chuyển vùng cách ly: {artifact_path.name}: {exc}")
        quarantine_artifact(artifact_path, f"{type(exc).__name__}: {exc}")
        return False

def audit_reward_weights(variant):
    """Chỉ kiểm tra nhãn huấn luyện ban đầu; không đọc nhãn kiểm thử tương lai."""
    benign, malware = int(np.sum(y_train == 0)), int(np.sum(y_train == 1))
    ratio = benign / malware if malware else float("inf")
    effective_fn_weight = variant["minority_priority"] * variant["fn_penalty"]
    print({"variant": variant["name"], "initial_train_benign": benign,
           "initial_train_malware": malware, "benign_to_malware_ratio": ratio,
           "mp": variant["minority_priority"], "fn_penalty": variant["fn_penalty"],
           "effective_fn_reward_weight": effective_fn_weight})

def write_variant_metadata(raw_dir, variant, operating_values, track, policy=None):
    raw_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "experiment": EXPERIMENT_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "runtime_platform_requested": RUNTIME_PLATFORM,
        "runtime_platform_resolved": RESOLVED_RUNTIME_PLATFORM,
        "runtime_input_root": str(RUNTIME_INPUT_ROOT),
        "runtime_work_root": str(RUNTIME_WORK_ROOT),
        "strategy_run_tag": STRATEGY_RUN_TAG if track == "strategy" else None,
                "dataset_years": DATASET_YEARS,
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "training_window_mode": TRAINING_WINDOW_MODE,
        "temporal_reference_policy": FIXED_TEMPORAL_REFERENCE_POLICY,
        "training_window": TRAINING_WINDOW,
        "testing_window": TESTING_WINDOW,
        "granularity": GRANULARITY,
        "feature_dim": int(X.shape[1]),
        "initial_train_benign": int(np.sum(y_train == 0)),
        "initial_train_malware": int(np.sum(y_train == 1)),
        "raw_artifact_directory": str(raw_dir),
        "track": track,
        "variant": variant,
        "policy": policy,
        "operating_values": operating_values,
        "seeds": SEEDS,
        "mp_interpretation": "hệ số phần thưởng của nhãn mã độc",
        "effective_fn_weight": variant["minority_priority"] * variant["fn_penalty"],
        "source_code_fingerprints": SOURCE_CODE_FINGERPRINTS,
    }
    atomic_json(raw_dir / "variant_metadata.json", payload)

def build_strategy_plan():
    """Cấu hình chiến lược và kiểm tra chi phí IRAL, ghép theo cùng seed."""
    records = []
    dataset_path = (
        f"{DATA_OUT},{DATASET_ARTIFACT_TAG}-X.p,{DATASET_ARTIFACT_TAG}-y.p,"
        f"{DATASET_ARTIFACT_TAG}-t.p,{X.shape[1]}")
    for variant in STRATEGY_VARIANTS:
        policy = variant["policy"]
        raw_dir = STRATEGY_RAW_OUT / variant["name"]
        audit_reward_weights(variant)
        write_variant_metadata(
            raw_dir, variant,
            [{"reject_cost": variant["reject_cost"],
              "selector_mode": variant["selector_mode"],
              "feedback_budget": variant["feedback_budget"],
              "feedback_timing": "after-prediction; update-from-next-month"}],
            "strategy", policy=policy)
        settings = Settings(
            dataset=[dataset_path], model=[variant["model"]], seed=SEEDS,
            is_active=[True], al_select_mode=[AL_SELECTION_MODE],
            is_reject=[True], is_reject_action=[True],
            al_rate=[0], reject_rate=[0], reject_cost=[variant["reject_cost"]],
            reward_rejected_outcome=[True], select_reject_labeled=[True],
            posthoc_augmented_AL=[0],
            minority_priority=[MP_FIXED], majority_priority=[1.0],
            temporal_rewards=[True], temporal_scaling=[6.0],
            is_fine_tuning=[True], training_epochs=[DRMD_TRAINING_EPOCHS],
            finetuning_size=[STRATEGY_FINETUNING_SIZE])
        for cfg in generate_experiment_configs(settings):
            cfg.results_save_location = str(raw_dir) + "/"
            cfg.training_window, cfg.testing_window = TRAINING_WINDOW, TESTING_WINDOW
            cfg.granularity, cfg.cuda, cfg.mps = GRANULARITY, True, False
            cfg._fn_penalty = 1.0
            cfg._adaptive_fn_penalty = bool(
                variant.get('adaptive_fn_penalty', False))
            cfg._afnp_audit_sampling = bool(
                variant.get('audit_sampling', False))
            cfg._afnp_budget = int(variant.get('feedback_budget', 0))
            cfg._afnp_seed = int(cfg.seed)
            cfg._afnp_window = int(FN_CONTROLLER_WINDOW_PERIODS)
            cfg._afnp_confidence = float(FN_CONTROLLER_CONFIDENCE_LEVEL)
            cfg._drift_aware_fn_penalty = bool(
                variant.get('drift_aware_fn_penalty', False))
            cfg._adaptive_bhr_quota = bool(
                variant.get('adaptive_bhr_quota', False))
            cfg._feedback_selector_mode = variant["selector_mode"]
            cfg._feedback_budget = variant["feedback_budget"]
            cfg._candidate_multiplier = variant["candidate_multiplier"]
            cfg._full_feedback = variant["full_feedback"]
            cfg._training_history_cap = STRATEGY_FINETUNING_SIZE
            cfg._balanced_hard_replay = bool(variant.get('balanced_hard_replay', False))
            cfg._bhr_fn_fraction = BHR_FN_FRACTION
            cfg._bhr_fp_fraction = BHR_FP_FRACTION
            cfg._bhr_fn_max_repeat = BHR_FN_MAX_REPEAT
            cfg._bhr_fp_max_repeat = BHR_FP_MAX_REPEAT
            artifact_path = expected_result_path(cfg)
            artifact_name_bytes = len((artifact_path.name + ".meta.json").encode("utf-8"))
            if artifact_name_bytes > ARTIFACT_FILENAME_MAX_BYTES:
                raise OSError(
                    f"Tên artifact {artifact_path.name!r} dài {artifact_name_bytes} byte.")
            label = f"strategy::{variant['name']}::{policy['name']}::Seed{cfg.seed}"
            records.append({
                "cfg": cfg, "label": label, "track": "strategy",
                "variant": variant, "policy": policy, "require_sidecar": True,
            })
    order = {variant["name"]: index for index, variant in enumerate(STRATEGY_VARIANTS)}
    return sorted(records, key=lambda row: (
        order[row["variant"]["name"]], int(row["cfg"].seed)))

def build_session_plan():
    if not RUN_STRATEGY_EXPERIMENT or KAGGLE_RUN_PHASE == "aggregate_only":
        return []
    return build_strategy_plan()

def stage_and_promote(record):
    global CURRENT_RUN_TAG, CURRENT_PERIOD_IDX
    cfg, label = record["cfg"], record["label"]
    policy = record["policy"]
    spec = run_spec(cfg, label, record["track"], record["variant"], policy=policy)
    signature = signature_for_spec(spec)
    final_path = expected_result_path(cfg)

    for cached_path in compatible_cached_paths(cfg):
        if cache_is_valid(
                cached_path, signature, policy, record["variant"],
                record["require_sidecar"]):
            write_progress_event("skip_validated_cache", cfg, {
                "label": label, "path": str(cached_path), "run_signature": signature})
            print("⏩ Cache hợp lệ:", cached_path.name)
            return "cached"

    stage_dir = STAGING_OUT / signature
    if stage_dir.exists():
        quarantine_path(stage_dir, "staging dở dang từ phiên trước")
    stage_dir.mkdir(parents=True, exist_ok=False)
    stage_cfg = deepcopy(cfg)
    stage_cfg.results_save_location = str(stage_dir) + "/"
    staged_path = expected_result_path(stage_cfg)

    co_dinh_hat_giong(int(cfg.seed))
    CURRENT_RUN_TAG, CURRENT_PERIOD_IDX = label, -1
    write_progress_event("start_run", cfg, {
        "label": label, "staging_path": str(staged_path),
        "final_path": str(final_path), "run_signature": signature})
    gpu_available_for_run = bool(torch.cuda.is_available())
    gpu_device_name = (torch.cuda.get_device_name(0)
                       if gpu_available_for_run else None)
    if gpu_available_for_run:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    run_started_monotonic = time.perf_counter()
    run_exception = None
    try:
        run(stage_cfg)
    except Exception as exc:
        # DRMD gốc có thể ném lỗi ở bước in AUT sau khi đã ghi pickle. Chỉ chấp
        # nhận trường hợp đó khi artifact đã vượt toàn bộ kiểm tra cấu trúc.
        run_exception = exc

    try:
        validation = validate_result_file(
            staged_path, EXPECTED_PERIODS, policy=policy,
            variant=record["variant"])
    except Exception:
        if run_exception is not None:
            raise run_exception
        raise

    if run_exception is not None:
        write_progress_event("post_save_exception_ignored", cfg, {
            "label": label, "error_type": type(run_exception).__name__,
            "message": str(run_exception)[:500]})

    if gpu_available_for_run:
        torch.cuda.synchronize()
    run_wall_seconds = time.perf_counter() - run_started_monotonic
    gpu_peak_allocated_bytes = (
        int(torch.cuda.max_memory_allocated()) if gpu_available_for_run else None)
    gpu_peak_reserved_bytes = (
        int(torch.cuda.max_memory_reserved()) if gpu_available_for_run else None)
    artifact_bytes = int(staged_path.stat().st_size)
    metadata = {
        "status": "complete",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "protocol_version": PROTOCOL_VERSION,
        "run_signature": signature,
        "artifact_sha256": sha256_file(staged_path),
        "artifact_naming": {
            "scheme": "sha256_20_of_legacy_drmd_stem",
            "stored_stem": final_path.stem,
            "legacy_stem": legacy_result_stem(cfg),
        },
        "dataset_source_fingerprint": DATASET_SOURCE_FINGERPRINT,
        "split_fingerprint": SPLIT_FINGERPRINT,
        "signature_schema_version": SIGNATURE_SCHEMA_VERSION,
        "parameter_provenance": PARAMETER_PROVENANCE,
        "source_code_fingerprints": SOURCE_CODE_FINGERPRINTS,
        "run_spec": spec,
        "raw_artifact": str(final_path.relative_to(OUT)),
        "environment": ENVIRONMENT_STAMP,
        "run_wall_seconds": float(run_wall_seconds),
        "gpu_available": gpu_available_for_run,
        "gpu_device_name": gpu_device_name,
        "gpu_peak_allocated_bytes": gpu_peak_allocated_bytes,
        "gpu_peak_reserved_bytes": gpu_peak_reserved_bytes,
        "artifact_bytes": artifact_bytes,
        "validation": validation,
    }
    atomic_json(sidecar_path(staged_path), metadata)

    final_path.parent.mkdir(parents=True, exist_ok=True)
    os.replace(staged_path, final_path)
    os.replace(sidecar_path(staged_path), sidecar_path(final_path))
    try:
        stage_dir.rmdir()
    except OSError:
        pass
    validate_result_file(
        final_path, EXPECTED_PERIODS, policy=policy,
        variant=record["variant"])
    validate_sidecar(final_path, signature)
    write_progress_event("finish_run", cfg, {
        "label": label, "path": str(final_path),
        "run_signature": signature, "artifact_sha256": metadata["artifact_sha256"],
        "run_wall_seconds": float(run_wall_seconds),
        "gpu_peak_allocated_bytes": gpu_peak_allocated_bytes,
        "gpu_peak_reserved_bytes": gpu_peak_reserved_bytes,
        "artifact_bytes": artifact_bytes})
    return "completed"

def record_failure(record, exc):
    cfg = record["cfg"]
    payload = {
        "time_utc": datetime.now(timezone.utc).isoformat(),
        "label": record["label"], "track": record["track"],
        "policy": record["policy"]["name"] if record["policy"] else "external",
        "seed": int(cfg.seed), "al": int(cfg.al_rate), "rej": int(cfg.reject_rate),
        "reject_cost": float(cfg.reject_cost),
        "fn_penalty": float(record["variant"]["fn_penalty"]),
        "error_type": type(exc).__name__, "message": str(exc)[:1000],
        "traceback": traceback.format_exc()[-6000:],
    }
    FAILED_RUNS.append(payload)
    append_jsonl(FAILURE_LOG, payload)
    write_progress_event("run_failed", cfg, payload)
    print(f"❌ {record['label']} — {type(exc).__name__}: {exc}")
    if payload["traceback"]:
        print(payload["traceback"])

SESSION_PLAN = build_session_plan()
print({"session_phase": KAGGLE_RUN_PHASE, "planned_drmd_runs": len(SESSION_PLAN),
       "launch_cutoff_hours": SESSION_LAUNCH_CUTOFF_HOURS,
       "strategy_root": str(STRATEGY_RAW_OUT),
       
       "phase_1_methods": PHASE_1_METHODS,
       "phase_2_methods": PHASE_2_METHODS,
       "training_methods_this_phase": [
           variant["name"] for variant in STRATEGY_VARIANTS]})

consecutive_failures = 0
for record in SESSION_PLAN:
    if (STOP_AFTER_DRMD_RUNS is not None
            and DRMD_RUNS_STARTED_THIS_SESSION >= STOP_AFTER_DRMD_RUNS):
        DRMD_SESSION_STOPPED = True
        SESSION_STOP_REASON = "session_run_cap"
        break
    if session_elapsed_hours() >= SESSION_LAUNCH_CUTOFF_HOURS:
        DRMD_SESSION_STOPPED = True
        SESSION_STOP_REASON = "launch_cutoff"
        break

    final_path = expected_result_path(record["cfg"])
    # Chỉ tăng bộ đếm khi artifact chưa được xác nhận là cache hợp lệ. Hàm
    # stage_and_promote vẫn tự kiểm tra lần nữa để tránh race.
    try:
        result_state = stage_and_promote(record)
        if result_state == "completed":
            DRMD_RUNS_STARTED_THIS_SESSION += 1
            DRMD_RUNS_COMPLETED_THIS_SESSION += 1
        consecutive_failures = 0
    except (KeyboardInterrupt, SystemExit):
        raise
    except Exception as exc:
        DRMD_RUNS_STARTED_THIS_SESSION += 1
        consecutive_failures += 1
        record_failure(record, exc)
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            DRMD_SESSION_STOPPED = True
            SESSION_STOP_REASON = "maximum_consecutive_failures"
            break

if FAILED_RUNS and not DRMD_SESSION_STOPPED:
    SESSION_STOP_REASON = "completed_plan_with_failed_runs"

status_payload = {
    "time_utc": datetime.now(timezone.utc).isoformat(),
    "protocol_version": PROTOCOL_VERSION,
    "phase": KAGGLE_RUN_PHASE,
    "runtime_platform_requested": RUNTIME_PLATFORM,
    "runtime_platform_resolved": RESOLVED_RUNTIME_PLATFORM,
    "runtime_input_root": str(RUNTIME_INPUT_ROOT),
    "runtime_work_root": str(RUNTIME_WORK_ROOT),
    "planned_drmd_runs_this_phase": len(SESSION_PLAN),
    "runs_started_this_session": DRMD_RUNS_STARTED_THIS_SESSION,
    "runs_completed_this_session": DRMD_RUNS_COMPLETED_THIS_SESSION,
    "failures_this_session": len(FAILED_RUNS),
    "session_stopped": DRMD_SESSION_STOPPED,
    "stop_reason": SESSION_STOP_REASON,
    "elapsed_hours": round(session_elapsed_hours(), 4),
    "runtime_prior": RUNTIME_PRIOR,
}
atomic_json(RUN_STATUS_PATH, status_payload)
write_progress_event("session_training_cell_done", extra=status_payload)

if FAILED_RUNS:
    keys = sorted({key for row in FAILED_RUNS for key in row})
    with open(TABLE_OUT / "failed_runs.csv", "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader()
        writer.writerows(FAILED_RUNS)

# Xuất checkpoint ngay tại cuối ô huấn luyện, trước mọi phép tổng hợp. Vì vậy một
# bảng/hình chưa đủ dữ liệu hoặc module báo cáo lỗi không thể làm mất artifact của phiên.
import zipfile

def export_session_checkpoint_atomic():
    checkpoint_path = OUT / f"{EXPERIMENT_NAME}_checkpoint_raw.zip"
    temporary = checkpoint_path.with_name(checkpoint_path.name + ".tmp")
    roots = [STRATEGY_RAW_OUT, RAW_OUT / "static_baselines",
             RUN_PROGRESS_LOG, RUN_STATUS_PATH, FAILURE_LOG]
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=1) as archive:
        for root in roots:
            if not root.exists():
                continue
            paths = [root] if root.is_file() else sorted(root.rglob("*"))
            for path in paths:
                if path.is_file():
                    archive.write(path, path.relative_to(OUT))
    os.replace(temporary, checkpoint_path)
    print(f"Checkpoint nguyên tử: {checkpoint_path.name} "
          f"({checkpoint_path.stat().st_size / 1024**2:.2f} MiB)")
    return checkpoint_path

export_session_checkpoint_atomic()
print(status_payload)

## Ô 10 — Kiểm toán và tổng hợp đủ sáu phương pháp


In [ ]:
# Tổng hợp V15: kiểm toán sáu phương pháp × mười seed.
import csv
import json
import pickle
from scipy import stats


def _stable_one_sample_ttest(values):
    """Tính kiểm định t một mẫu mà không phát cảnh báo suy biến.

    Hiệu số hoàn toàn bằng không biểu thị không có bằng chứng
    chống lại giả thuyết không, nên trả về p=1. Khi mọi hiệu số
    bằng cùng một hằng khác không, phương sai bằng không làm
    kiểm định t không xác định; trường hợp này được ghi
    NaN thay vì gán sai p=0.
    """

    import numpy as _np

    finite = _np.asarray(values, dtype=float).reshape(-1)
    finite = finite[_np.isfinite(finite)]
    if finite.size < 2:
        return float("nan"), "insufficient_or_nonfinite"

    scale = max(1.0, float(_np.max(_np.abs(finite))))
    tolerance = 100.0 * _np.finfo(float).eps * scale
    if float(_np.max(_np.abs(finite))) <= tolerance:
        return 1.0, "all_zero"
    if float(_np.ptp(finite)) <= tolerance:
        return float("nan"), "zero_variance_nonzero"

    from scipy import stats as _stats

    p_value = float(_stats.ttest_1samp(finite, 0.0).pvalue)
    if not _np.isfinite(p_value):
        return float("nan"), "ttest_nonfinite"
    return p_value, "ttest_1samp"



def _safe_rate(numerator, denominator):
    return float(numerator / denominator) if denominator else float("nan")


def _metric_counts(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tn, fp, fn, tp = skmetrics.confusion_matrix(
        y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "positive_support": int(tp + fn),
        "negative_support": int(tn + fp),
        "precision": _safe_rate(int(tp), int(tp + fp)),
        "recall": _safe_rate(int(tp), int(tp + fn)),
        "f1": _safe_rate(2 * int(tp), 2 * int(tp) + int(fp) + int(fn)),
        "fnr": _safe_rate(int(fn), int(tp + fn)),
        "fpr": _safe_rate(int(fp), int(tn + fp)),
    }


def _aut(values, months):
    values = np.asarray(values, dtype=float)
    month_number = np.asarray([
        int(month[:4]) * 12 + int(month[5:7]) for month in months], dtype=int)
    valid = (np.isfinite(values[:-1]) & np.isfinite(values[1:])
             & (np.diff(month_number) == 1))
    return (float(np.mean((values[:-1][valid] + values[1:][valid]) / 2))
            if np.any(valid) else float("nan"))


def _write_csv(path, rows):
    if not rows:
        return
    keys = []
    for row in rows:
        for key in row:
            if key not in keys:
                keys.append(key)
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=keys, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            # Đại lượng không xác định do chu kỳ không có hỗ trợ lớp được để
            # trống trong bảng thay vì in chuỗi "nan". Các cột support đi kèm
            # cho phép phân biệt "không xác định" với giá trị bằng 0.
            writer.writerow({
                key: ("" if isinstance(value, (float, np.floating))
                      and not np.isfinite(value) else value)
                for key, value in row.items()
            })


period_months = []
for split_row in split_rows[1:]:
    start_month, end_month = str(split_row["start"])[:7], str(split_row["end"])[:7]
    if start_month != end_month:
        raise ValueError(f"Chu kỳ {split_row['period']} đi qua hai tháng.")
    period_months.append(start_month)

variant_map = {variant["name"]: variant for variant in ALL_STRATEGY_VARIANTS}
# Hai tên dưới đây chỉ là nhãn lưu trữ của các artifact đã hoàn tất trước khi
# notebook nộp được rút gọn tên phương pháp. Việc ánh xạ không thay đổi cấu
# hình chạy: artifact chỉ được nhận nếu toàn bộ cờ cơ chế khớp nhánh cuối.
legacy_final_method_aliases = {
    "DRMD-FN-V15": "DRMD-FN",
    "DRMD-FN-BHR-V15": "DRMD-FN-BHR",
}


def _spec_matches_final_variant(spec, variant):
    expected = {
        "audit_sampling": bool(variant.get("audit_sampling", False)),
        "adaptive_fn_penalty": bool(
            variant.get("adaptive_fn_penalty", False)),
        "drift_aware_fn_penalty": bool(
            variant.get("drift_aware_fn_penalty", False)),
        "adaptive_bhr_quota": bool(
            variant.get("adaptive_bhr_quota", False)),
        "balanced_hard_replay": bool(
            variant.get("balanced_hard_replay", False)),
        "full_feedback": bool(variant.get("full_feedback", False)),
        "feedback_selector_mode": str(variant.get("selector_mode", "iral")),
        "feedback_budget": int(variant.get("feedback_budget", 0)),
        "policy": str(variant["policy"]["name"]),
    }
    return all(spec.get(key) == value for key, value in expected.items())


def _resolve_final_method(spec):
    stored_name = str(spec.get("variant_name"))
    logical_name = legacy_final_method_aliases.get(stored_name, stored_name)
    variant = variant_map.get(logical_name)
    if variant is None or not _spec_matches_final_variant(spec, variant):
        return None
    # Artifact mang tên chuẩn được ưu tiên nếu cả tên chuẩn và tên lưu trữ cũ
    # cùng tồn tại sau khi khôi phục nhiều checkpoint.
    priority = 1 if stored_name == logical_name else 0
    return logical_name, priority, stored_name


expected_strategy_keys = {
    (name, int(seed)) for name in ALL_STRATEGY_NAMES for seed in SEEDS}
strategy_artifacts, strategy_priorities = {}, {}
strategy_sources, compatibility_imports = {}, []
ignored_incompatible_artifacts, shadowed_compatibility_artifacts = [], []
integrity_errors, duplicates = [], []
for metadata_path in sorted(STRATEGY_RAW_OUT.rglob("*.p.meta.json")):
    artifact_path = Path(str(metadata_path).removesuffix(".meta.json"))
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        spec = metadata["run_spec"]
        if metadata.get("protocol_version") != PROTOCOL_VERSION:
            continue
        resolved = _resolve_final_method(spec)
        if resolved is None:
            ignored_incompatible_artifacts.append({
                "path": str(artifact_path),
                "stored_method": spec.get("variant_name"),
                "reason": "method_or_mechanism_flags_not_in_final_matrix",
            })
            continue
        method, priority, stored_method = resolved
        seed = int(spec.get("seed"))
        key = (method, seed)
        if seed not in SEEDS:
            continue
        validate_sidecar(artifact_path, signature_for_spec(spec))
        validate_result_file(
            artifact_path, EXPECTED_TEST_PERIODS,
            policy=variant_map[method]["policy"], variant=variant_map[method])
        if key in strategy_artifacts:
            previous_priority = strategy_priorities[key]
            if priority != previous_priority:
                if priority > previous_priority:
                    shadowed_path = strategy_artifacts[key]
                    shadowed_source = strategy_sources[key]
                    strategy_artifacts[key] = artifact_path
                    strategy_priorities[key] = priority
                    strategy_sources[key] = stored_method
                else:
                    shadowed_path = artifact_path
                    shadowed_source = stored_method
                shadowed_compatibility_artifacts.append({
                    "method": method, "seed": int(seed),
                    "stored_method": shadowed_source,
                    "path": str(shadowed_path),
                })
                continue
            duplicates.append({
                "method": method, "seed": int(seed),
                "stored_method": stored_method, "path": str(artifact_path),
            })
            continue
        strategy_artifacts[key] = artifact_path
        strategy_priorities[key] = priority
        strategy_sources[key] = stored_method
    except Exception as exc:
        integrity_errors.append({
            "path": str(artifact_path), "error": f"{type(exc).__name__}: {exc}"})

compatibility_imports = [
    {
        "method": method, "seed": int(seed),
        "stored_method": strategy_sources[(method, seed)],
        "path": str(strategy_artifacts[(method, seed)]),
    }
    for method, seed in sorted(strategy_artifacts)
    if strategy_sources[(method, seed)] != method
]

static_artifacts, static_errors = {}, []
for seed in SEEDS:
    path = _static_path(seed)
    try:
        static_artifacts[("Static-MLP", int(seed))] = _validate_static_artifact(
            path, seed, require_sidecar=True)
    except Exception as exc:
        static_errors.append({
            "seed": int(seed), "path": str(path),
            "error": f"{type(exc).__name__}: {exc}"})

expected_static_keys = {("Static-MLP", int(seed)) for seed in SEEDS}
missing_strategy = sorted(expected_strategy_keys - set(strategy_artifacts))
missing_static = sorted(expected_static_keys - set(static_artifacts))
REPORT_READY = not (
    missing_strategy or missing_static or duplicates
    or integrity_errors or static_errors)
integrity_summary = {
    "protocol_version": PROTOCOL_VERSION,
    "expected_runs": 60,
    "valid_runs": len(strategy_artifacts) + len(static_artifacts),
    "expected_strategy_runs": 50,
    "valid_strategy_runs": len(strategy_artifacts),
    "expected_static_runs": 10,
    "valid_static_runs": len(static_artifacts),
    "expected_methods": EXPECTED_METHOD_NAMES,
    "seeds": SEEDS,
    "expected_periods_per_run": EXPECTED_TEST_PERIODS,
    "expected_test_samples_per_run": EXPECTED_TEST_SAMPLES,
    "missing_strategy_keys": missing_strategy,
    "missing_static_keys": missing_static,
    "duplicate_strategy_keys": duplicates,
    "legacy_final_method_aliases": legacy_final_method_aliases,
    "compatibility_imports": compatibility_imports,
    "ignored_incompatible_artifacts": ignored_incompatible_artifacts,
    "shadowed_compatibility_artifacts": shadowed_compatibility_artifacts,
    "errors": integrity_errors,
    "static_errors": static_errors,
    "report_ready": REPORT_READY,
}
(TABLE_OUT / "v15_integrity_summary.json").write_text(
    json.dumps(integrity_summary, indent=2, ensure_ascii=False),
    encoding="utf-8")
if STRICT_INVARIANTS and not REPORT_READY:
    raise RuntimeError("V15 chưa đủ 60/60 artifact hợp lệ; không tổng hợp kết luận.")

monthly = []
controller_rows, drift_rows, bhr_rows, quota_rows = [], [], [], []
for (method, seed), artifact_path in sorted(strategy_artifacts.items()):
    with open(artifact_path, "rb") as handle:
        result = pickle.load(handle)
    for period, month in enumerate(period_months):
        y_true = np.asarray(result["binary_y_true_all"][period], dtype=int)
        auto_pred = np.asarray(result["binary_y_preds_all"][period], dtype=int)
        rejected = np.asarray(result["rejected_indexes_all"][period], dtype=int)
        system_pred = auto_pred.copy()
        system_pred[rejected] = y_true[rejected]
        auto = _metric_counts(y_true, auto_pred)
        system = _metric_counts(y_true, system_pred)
        monthly.append({
            "method": method, "seed": int(seed), "period": int(period),
            "month": month, "n": int(y_true.size),
            **{f"auto_{key}": value for key, value in auto.items()},
            **{f"system_{key}": value for key, value in system.items()},
            "rejected": int(rejected.size),
            "reject_rate": _safe_rate(int(rejected.size), int(y_true.size)),
            "expert_labels": int(result["selected"][period]),
            "fn_penalty_used": float(result["fn_penalty_used"][period]),
            "fn_penalty_next": float(result["fn_penalty_next"][period]),
        })
        controller_rows.append({
            "method": method, "seed": int(seed), "period": int(period),
            "month": month, **result["fn_controller_telemetry"][period]})
        drift_rows.append({
            "method": method, "seed": int(seed), "period": int(period),
            "month": month, **result["drift_gate_telemetry"][period]})
        bhr_rows.append({
            "method": method, "seed": int(seed), "period": int(period),
            "month": month, **result["bhr_telemetry"][period]})
        quota_rows.append({
            "method": method, "seed": int(seed), "period": int(period),
            "month": month, **result["adaptive_bhr_quota_telemetry"][period]})

for (method, seed), artifact in sorted(static_artifacts.items()):
    for period, row in enumerate(artifact["period_rows"]):
        counts = {key: int(row[key]) for key in ("tp", "tn", "fp", "fn")}
        derived = {
            "positive_support": int(row["p"]),
            "negative_support": int(row["n"]),
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1": float(row["f1"]),
            "fnr": float(row["fnr"]), "fpr": float(row["fpr"]),
        }
        metrics = {**counts, **derived}
        monthly.append({
            "method": method, "seed": int(seed), "period": int(period),
            "month": period_months[period], "n": int(row["tot"]),
            **{f"auto_{key}": value for key, value in metrics.items()},
            **{f"system_{key}": value for key, value in metrics.items()},
            "rejected": 0, "reject_rate": 0.0, "expert_labels": 0,
            "fn_penalty_used": 1.0, "fn_penalty_next": 1.0,
        })

_write_csv(TABLE_OUT / "v15_all_methods_monthly.csv", monthly)
_write_csv(TABLE_OUT / "v15_fn_controller_monthly.csv", controller_rows)
_write_csv(TABLE_OUT / "v15_drift_gate_monthly.csv", drift_rows)
_write_csv(TABLE_OUT / "v15_bhr_telemetry_monthly.csv", bhr_rows)
_write_csv(TABLE_OUT / "v15_adaptive_bhr_quota_monthly.csv", quota_rows)

summary = []
for method in EXPECTED_METHOD_NAMES:
    for seed in SEEDS:
        rows = sorted(
            [row for row in monthly
             if row["method"] == method and int(row["seed"]) == int(seed)],
            key=lambda row: row["period"])
        if len(rows) != EXPECTED_TEST_PERIODS:
            continue
        months = [row["month"] for row in rows]
        record = {
            "method": method, "seed": int(seed),
            "n_periods": len(rows), "n_test_samples": sum(row["n"] for row in rows),
            "total_expert_labels": sum(row["expert_labels"] for row in rows),
            "total_rejected": sum(row["rejected"] for row in rows),
        }
        for scope in ("auto", "system"):
            for metric in ("f1", "fnr", "fpr"):
                values = [row[f"{scope}_{metric}"] for row in rows]
                record[f"aut_{scope}_{metric}"] = _aut(values, months)
                record[f"mean_{scope}_{metric}"] = float(np.nanmean(values))
            for count in ("tp", "tn", "fp", "fn"):
                record[f"total_{scope}_{count}"] = int(sum(
                    row[f"{scope}_{count}"] for row in rows))
            # Pooled được tính trực tiếp từ số đếm để tránh phụ thuộc thứ tự mẫu.
            tp, fp = record[f"total_{scope}_tp"], record[f"total_{scope}_fp"]
            fn, tn = record[f"total_{scope}_fn"], record[f"total_{scope}_tn"]
            record[f"pooled_{scope}_f1"] = _safe_rate(2 * tp, 2 * tp + fp + fn)
            record[f"pooled_{scope}_fnr"] = _safe_rate(fn, tp + fn)
            record[f"pooled_{scope}_fpr"] = _safe_rate(fp, tn + fp)
        summary.append(record)
_write_csv(TABLE_OUT / "v15_all_methods_summary_by_seed.csv", summary)

comparisons = [
    ("IRAAL_minus_IRAL", "DRMD-IRAAL", "DRMD-IRAL"),
    ("FN_minus_IRAAL", "DRMD-FN", "DRMD-IRAAL"),
    ("FN_BHR_minus_FN", "DRMD-FN-BHR", "DRMD-FN"),
    ("FN_BHR_minus_IRAAL", "DRMD-FN-BHR", "DRMD-IRAAL"),
    ("FFCR_minus_FN", "DRMD-FFCR", "DRMD-FN"),
    ("FFCR_minus_FN_BHR", "DRMD-FFCR", "DRMD-FN-BHR"),
]
for method in [name for name in EXPECTED_METHOD_NAMES if name != "Static-MLP"]:
    comparisons.append((f"{method}_minus_Static", method, "Static-MLP"))
summary_map = {(row["method"], int(row["seed"])): row for row in summary}
paired = []
for comparison, left, right in comparisons:
    for seed in SEEDS:
        if (left, int(seed)) not in summary_map or (right, int(seed)) not in summary_map:
            continue
        a, b = summary_map[(left, int(seed))], summary_map[(right, int(seed))]
        row = {"comparison": comparison, "left": left, "right": right,
               "seed": int(seed)}
        for metric in ("aut_auto_f1", "aut_auto_fnr", "aut_auto_fpr",
                       "pooled_auto_f1", "pooled_auto_fnr", "pooled_auto_fpr"):
            row[f"delta_{metric}"] = float(a[metric] - b[metric])
        for metric in ("total_auto_tp", "total_auto_fn", "total_auto_fp",
                       "total_expert_labels"):
            row[f"delta_{metric}"] = int(a[metric] - b[metric])
        paired.append(row)
_write_csv(TABLE_OUT / "v15_paired_effects_by_seed.csv", paired)

paired_summary = []
for comparison, left, right in comparisons:
    rows = [row for row in paired if row["comparison"] == comparison]
    if not rows:
        continue
    record = {"comparison": comparison, "left": left, "right": right,
              "n_pairs": len(rows)}
    for metric in ("delta_aut_auto_f1", "delta_aut_auto_fnr",
                   "delta_aut_auto_fpr", "delta_total_auto_tp",
                   "delta_total_auto_fn", "delta_total_auto_fp",
                   "delta_total_expert_labels"):
        values = np.asarray([row[metric] for row in rows], dtype=float)
        record[f"mean_{metric}"] = float(np.mean(values))
        record[f"sd_{metric}"] = (float(np.std(values, ddof=1))
                                    if values.size > 1 else float("nan"))
        paired_t_p, paired_t_status = _stable_one_sample_ttest(values)
        record[f"paired_t_p_{metric}"] = paired_t_p
        record[f"paired_t_status_{metric}"] = paired_t_status
    paired_summary.append(record)
_write_csv(TABLE_OUT / "v15_paired_effects_summary.csv", paired_summary)

# Các kết luận chính dùng cùng 10 hạt giống và được hiệu chỉnh Holm theo từng
# chỉ số. Bảng dài này thuận tiện hơn bảng rộng ở trên khi đưa vào báo cáo:
# mỗi dòng là một giả thuyết, kèm KTC 95%, kiểm định tham số, kiểm định hạng
# có dấu và số hạt giống cho hiệu cùng/khác chiều.
primary_comparisons = [
    "IRAAL_minus_IRAL",
    "FN_minus_IRAAL",
    "FN_BHR_minus_FN",
    "FN_BHR_minus_IRAAL",
]
primary_metrics = [
    "delta_aut_auto_f1",
    "delta_aut_auto_fnr",
    "delta_aut_auto_fpr",
    "delta_total_auto_tp",
    "delta_total_auto_fn",
    "delta_total_auto_fp",
    "delta_total_expert_labels",
]
count_metrics = {
    "delta_total_auto_tp", "delta_total_auto_fn", "delta_total_auto_fp",
    "delta_total_expert_labels",
}
favorable_direction = {
    "delta_aut_auto_f1": "positive",
    "delta_aut_auto_fnr": "negative",
    "delta_aut_auto_fpr": "negative",
    "delta_total_auto_tp": "positive",
    "delta_total_auto_fn": "negative",
    "delta_total_auto_fp": "negative",
    "delta_total_expert_labels": "negative",
}
primary_tests = []
for comparison in primary_comparisons:
    rows = [row for row in paired if row["comparison"] == comparison]
    if not rows:
        continue
    for metric in primary_metrics:
        values = np.asarray([row[metric] for row in rows], dtype=float)
        values = values[np.isfinite(values)]
        n_pairs = int(values.size)
        mean_delta = float(np.mean(values)) if n_pairs else float("nan")
        sd_delta = (float(np.std(values, ddof=1))
                    if n_pairs > 1 else float("nan"))
        paired_t_p, paired_t_status = _stable_one_sample_ttest(values)
        if n_pairs > 1 and np.isfinite(sd_delta):
            margin = float(stats.t.ppf(0.975, n_pairs - 1) *
                           sd_delta / np.sqrt(n_pairs))
            ci_low, ci_high = mean_delta - margin, mean_delta + margin
        else:
            ci_low = ci_high = float("nan")
        if n_pairs:
            try:
                wilcoxon_p = (1.0 if np.allclose(values, 0.0) else
                              float(stats.wilcoxon(
                                  values, zero_method="wilcox",
                                  alternative="two-sided").pvalue))
            except ValueError:
                wilcoxon_p = float("nan")
        else:
            wilcoxon_p = float("nan")
        if np.isfinite(paired_t_p):
            inference_test = "paired_t"
            inference_p_raw = paired_t_p
        elif np.isfinite(wilcoxon_p):
            inference_test = "wilcoxon"
            inference_p_raw = wilcoxon_p
        else:
            inference_test = "not_available"
            inference_p_raw = float("nan")
        primary_tests.append({
            "comparison": comparison,
            "metric": metric,
            "unit": "count" if metric in count_metrics else "proportion",
            "favorable_direction": favorable_direction[metric],
            "n_pairs": n_pairs,
            "mean_delta": mean_delta,
            "sd_delta": sd_delta,
            "ci95_low": float(ci_low),
            "ci95_high": float(ci_high),
            "paired_t_p_raw": paired_t_p,
            "paired_t_status": paired_t_status,
            "wilcoxon_p_raw": wilcoxon_p,
            "inference_test": inference_test,
            "inference_p_raw": inference_p_raw,
            "positive_seeds": int(np.sum(values > 0)),
            "negative_seeds": int(np.sum(values < 0)),
            "zero_seeds": int(np.sum(values == 0)),
            "holm_family": "primary_comparisons_within_metric",
            "holm_adjusted_p": float("nan"),
        })

# Holm step-down: hiệu chỉnh năm đối chứng chính độc lập trong từng chỉ số.
for metric in primary_metrics:
    indices = [
        index for index, row in enumerate(primary_tests)
        if row["metric"] == metric and np.isfinite(row["inference_p_raw"])
    ]
    ordered = sorted(indices,
                     key=lambda index: primary_tests[index]["inference_p_raw"])
    running_max = 0.0
    family_size = len(ordered)
    for rank, index in enumerate(ordered):
        raw_p = float(primary_tests[index]["inference_p_raw"])
        adjusted = min(1.0, (family_size - rank) * raw_p)
        running_max = max(running_max, adjusted)
        primary_tests[index]["holm_adjusted_p"] = float(running_max)

_write_csv(TABLE_OUT / "v15_primary_paired_tests_holm.csv", primary_tests)

print({"v15_report_ready": REPORT_READY,
       "valid_runs": len(strategy_artifacts) + len(static_artifacts),
       "expected_runs": 60, "tables": str(TABLE_OUT)})


## Ô 11 — Hiện vật báo cáo

Chỉ sử dụng kết quả khi `REPORT_READY=True`, tương ứng đủ 60/60 khóa phương pháp–hạt giống và toàn bộ cổng nhân quả/sidecar đạt.


## Ô 12 — Đóng gói checkpoint và bảng V15


In [ ]:
# Đóng gói nguyên tử: bảng, hình và checkpoint V15.
import os
import zipfile

ARCHIVE_RAW_RESULTS = True

def zip_tree_atomic(zip_path, roots, compresslevel=6):
    temporary = zip_path.with_name(zip_path.name + ".tmp")
    with zipfile.ZipFile(
            temporary, "w", compression=zipfile.ZIP_DEFLATED,
            compresslevel=compresslevel) as archive:
        for root in roots:
            if not root.exists():
                continue
            paths = [root] if root.is_file() else sorted(root.rglob("*"))
            for path in paths:
                if path.is_file() and path != temporary:
                    archive.write(path, path.relative_to(OUT))
    os.replace(temporary, zip_path)
    print(f"Đã tạo {zip_path.name}: {zip_path.stat().st_size / 1024**2:.2f} MiB")

bundle_kind = ("full_test_report_bundle" if REPORT_READY
               else "provisional_audit_bundle")
report_bundle = OUT / f"{EXPERIMENT_NAME}_{bundle_kind}.zip"
zip_tree_atomic(report_bundle, [TABLE_OUT, FIG_OUT, RUN_STATUS_PATH,
                                RUN_PROGRESS_LOG, FAILURE_LOG, QUARANTINE_LOG])

if ARCHIVE_RAW_RESULTS:
    checkpoint_bundle = OUT / f"{EXPERIMENT_NAME}_checkpoint_raw.zip"
    zip_tree_atomic(checkpoint_bundle,
                    [STRATEGY_RAW_OUT, RAW_OUT / "static_baselines",
                     RUN_PROGRESS_LOG, RUN_STATUS_PATH, FAILURE_LOG],
                    compresslevel=1)


# Trên Colab, sao lưu không ghi đè nếu Google Drive đã được mount trước đó.
def _backup_colab_archives_to_mounted_drive(archives):
    if not IS_GOOGLE_COLAB or not COLAB_BACKUP_TO_MOUNTED_DRIVE:
        return []
    mounted_my_drive = Path("/content/drive/MyDrive")
    if not mounted_my_drive.is_dir():
        print(
            "ℹ️ Google Drive chưa được mount; checkpoint vẫn nằm trong "
            f"{OUT}. Mount Drive trước khi chạy nếu muốn tự động sao lưu."
        )
        return []

    import shutil

    backup_root = Path(COLAB_DRIVE_BACKUP_DIR)
    backup_root.mkdir(parents=True, exist_ok=True)
    timestamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    saved = []
    for source_path in archives:
        source_path = Path(source_path)
        if not source_path.is_file():
            continue
        destination = backup_root / (
            f"{source_path.stem}_{KAGGLE_RUN_PHASE}_{timestamp}"
            f"{source_path.suffix}"
        )
        duplicate_index = 1
        while destination.exists():
            destination = backup_root / (
                f"{source_path.stem}_{KAGGLE_RUN_PHASE}_{timestamp}_"
                f"{duplicate_index}{source_path.suffix}"
            )
            duplicate_index += 1
        temporary = destination.with_name(destination.name + ".tmp")
        try:
            shutil.copy2(source_path, temporary)
            os.replace(temporary, destination)
        except Exception as exc:
            if temporary.exists():
                temporary.unlink()
            print({"colab_drive_backup": "failed",
                   "source": str(source_path),
                   "error_type": type(exc).__name__})
            continue
        saved.append(str(destination))
    print({"colab_drive_backup": "complete", "saved": saved})
    return saved


_colab_archives = [report_bundle]
if ARCHIVE_RAW_RESULTS:
    _colab_archives.append(checkpoint_bundle)
COLAB_SAVED_ARCHIVES = _backup_colab_archives_to_mounted_drive(_colab_archives)
